# Step30 — Tumor Microenvironment Cell–Cell Communication

## Objective

Identify tumor-microenvironment signals associated with the frozen malignant-state regulatory architecture.

Primary mechanistic framework:

**TME sender cell**  
→ **ligand**  
→ **malignant-cell receptor / signaling**  
→ **Step29 regulatory architecture**  
→ **MS5 transitional plasticity / MS3 proliferative-CIN biology**

## Frozen upstream framework

- Malignant-cell universe: 12,057 cells
- Frozen malignant states: MS1–MS8
- Primary biological axis: MS6–MS5–MS3
- Step29 MS6 hepatic-identity regulators:
  HNF4A, NR1H4, NR1I3, CEBPA, FOXA3
- Step29 MS5 transitional/plasticity regulators:
  KLF4, FOS, FOSB, ELF3, SOX4
- Step29 MS3 proliferative/CIN regulators:
  FOXM1, MYBL2, E2F8, TFDP1, E2F1, BRCA1

## Step30 principle

Cell–cell communication analysis will not redefine malignant states or Step29 regulons.

The primary goal is to connect extracellular TME signaling with the frozen malignant-state regulatory architecture.

Communication inference represents ligand–receptor association and signaling compatibility, not direct proof of causal intercellular signaling.

In [ ]:
# =============================================================================
# Step30A Cell 2
# Environment audit
# =============================================================================

import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import scanpy as sc
import anndata as ad

import liana as li
import decoupler as dc
import omnipath

print("Python:", sys.executable)
print("Python version:", platform.python_version())

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Scanpy:", sc.__version__)
print("AnnData:", ad.__version__)
print("LIANA:", li.__version__)

print("\nSTEP30 ENVIRONMENT: PASS")

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step30A Cell 3
# Project directories
# =============================================================================

ROOT = Path(_hcc_path(''))

STEP30_DIR = (
    ROOT /
    "LIHC_scRNA_Step30_Communication"
)

DATA_DIR = STEP30_DIR / "Data"
TABLE_DIR = STEP30_DIR / "Tables"
FIGURE_DIR = STEP30_DIR / "Figures"
AUDIT_DIR = STEP30_DIR / "Audit"
RESOURCE_DIR = STEP30_DIR / "Resources"

for p in [
    DATA_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    AUDIT_DIR,
    RESOURCE_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

print("Step30 directory:")
print(STEP30_DIR)

print("\nSTEP30 DIRECTORIES: PASS")

In [ ]:
# =============================================================================
# Step30A Cell 4
# Frozen Step29 upstream lock
# =============================================================================

STEP29_DIR = (
    ROOT /
    "LIHC_scRNA_Step29_GRN"
)

STEP29_LOCK = (
    STEP29_DIR /
    "Audit" /
    "Step29_FINAL_RegulatoryArchitecture.json"
)

STEP27_H5AD = (
    ROOT /
    "LIHC_scRNA_Step27_Trajectory" /
    "Data" /
    "GSE149614_STEP27_FINAL_TRAJECTORY.h5ad"
)

print(
    "Step29 Final Lock:",
    STEP29_LOCK.exists()
)

print(
    "Step27 malignant object:",
    STEP27_H5AD.exists()
)

assert STEP29_LOCK.exists()
assert STEP27_H5AD.exists()

print(
    "\nFROZEN UPSTREAM INPUTS: PASS"
)

In [ ]:
# =============================================================================
# Step30A Cell 5
# Read frozen Step29 regulatory architecture
# =============================================================================

import json

with open(
    STEP29_LOCK,
    "r",
    encoding="utf-8"
) as f:

    step29_lock = json.load(f)

for key in [
    "MS6_Hepatic_Identity",
    "MS5_Transitional_Plasticity",
    "MS3_Proliferative_CIN"
]:

    print(
        f"\n{key}:"
    )

    print(
        ", ".join(
            step29_lock[key]
        )
    )

print(
    "\nSTEP29 REGULATORY ARCHITECTURE LOADED: PASS"
)

In [ ]:
# =============================================================================
# Step30A Cell 6
# Load frozen malignant reference
# =============================================================================

adata_malignant = sc.read_h5ad(
    STEP27_H5AD
)

print(
    "Frozen malignant object:",
    adata_malignant.shape
)

print(
    "Patients:"
)

print(
    adata_malignant.obs[
        "patient"
    ].value_counts()
)

print(
    "\nStates:"
)

print(
    adata_malignant.obs[
        "final_malignant_state_id"
    ].value_counts()
)

assert adata_malignant.n_obs == 12057

print(
    "\nFROZEN MALIGNANT REFERENCE: PASS"
)

In [ ]:
# =============================================================================
# Step30A Cell 7
# Targeted scRNA project directory audit
# =============================================================================

project_dirs = []

for p in ROOT.iterdir():

    if not p.is_dir():
        continue

    name_low = p.name.lower()

    if any(
        token in name_low
        for token in [
            "scrna",
            "gse149614",
            "step22",
            "step23",
            "step24",
            "step25",
            "step26",
            "step27",
            "step28",
            "step29"
        ]
    ):

        project_dirs.append(
            p
        )


print(
    "Candidate scRNA directories:"
)

for p in project_dirs:

    print(
        p
    )

In [ ]:
# =============================================================================
# Step30A Cell 8
# Targeted h5ad inventory
# =============================================================================

h5ad_rows = []

for project_dir in project_dirs:

    search_dirs = [
        project_dir,
        project_dir / "Data"
    ]

    for search_dir in search_dirs:

        if not search_dir.exists():
            continue

        for path in search_dir.glob(
            "*.h5ad"
        ):

            h5ad_rows.append({
                "Project":
                    project_dir.name,

                "File":
                    path.name,

                "Path":
                    str(path),

                "Size_MB":
                    round(
                        path.stat().st_size
                        / 1024**2,
                        2
                    )
            })


h5ad_inventory = (
    pd.DataFrame(
        h5ad_rows
    )
    .drop_duplicates(
        subset="Path"
    )
)

if not h5ad_inventory.empty:

    h5ad_inventory = (
        h5ad_inventory
        .sort_values(
            "Size_MB",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


print(
    "Candidate h5ad files:",
    len(h5ad_inventory)
)

display(
    h5ad_inventory
)

In [ ]:
# =============================================================================
# Step30A Cell 9
# Rank possible full-TME objects
# =============================================================================

if h5ad_inventory.empty:

    full_candidates = (
        h5ad_inventory.copy()
    )

else:

    candidate_pattern = (
        "full|all|integrated|annot|"
        "qc|processed|final|harmony"
    )

    exclude_pattern = (
        "malignant|trajectory|"
        "step27|step28|step29|"
        "lineage_clean"
    )

    full_candidates = (
        h5ad_inventory.loc[
            h5ad_inventory[
                "File"
            ].str.lower().str.contains(
                candidate_pattern,
                regex=True
            )
            &
            ~h5ad_inventory[
                "File"
            ].str.lower().str.contains(
                exclude_pattern,
                regex=True
            )
        ]
        .copy()
    )


print(
    "Possible full-TME objects:"
)

display(
    full_candidates
)

In [ ]:
# =============================================================================
# Step30A Cell 10
# REPORT — full-TME object discovery
# =============================================================================

print("=" * 100)

print(
    "STEP30A FULL-TME OBJECT DISCOVERY REPORT"
)

print("=" * 100)


print(
    "\nFrozen malignant reference:"
)

print(
    STEP27_H5AD
)

print(
    "Shape:",
    adata_malignant.shape
)


print(
    "\nCandidate scRNA directories:",
    len(project_dirs)
)

print(
    "Candidate h5ad files:",
    len(h5ad_inventory)
)

print(
    "Possible full-TME objects:",
    len(full_candidates)
)


print(
    "\nTOP CANDIDATE H5AD FILES:"
)

display(
    h5ad_inventory.head(30)
)


print(
    "\nPOSSIBLE FULL-TME OBJECTS:"
)

display(
    full_candidates
)


print(
    "\nSTEP30A DISCOVERY COMPLETE"
)

print(
    "No communication analysis has been performed."
)

In [ ]:
# =============================================================================
# Step30A Cell 11
# Lock four full-TME candidate objects
# =============================================================================

CANDIDATES = {
    "Step22_FinalMalignancy": (
        ROOT /
        "LIHC_scRNA_Step22_Malignant_Identification" /
        "Data" /
        "GSE149614_STEP22_FINAL_MALIGNANCY_ANNOTATED.h5ad"
    ),

    "Step21_MajorAnnotation": (
        ROOT /
        "LIHC_scRNA_Step21_Major_Cell_Annotation" /
        "Data" /
        "GSE149614_STEP21_R0_6_MARKERS.h5ad"
    ),

    "Step20_LogNormAllGenes": (
        ROOT /
        "LIHC_scRNA_Step20_Normalization_Integration" /
        "Data" /
        "GSE149614_LOGNORM_ALL_GENES.h5ad"
    ),

    "Step19_AllCells": (
        ROOT /
        "LIHC_scRNA_Step19_QC" /
        "Data" /
        "GSE149614_SCRUBLET_ANNOTATED_ALL_CELLS.h5ad"
    )
}

for name, path in CANDIDATES.items():
    print(
        name,
        "->",
        path.exists(),
        "|",
        path
    )

assert all(
    p.exists()
    for p in CANDIDATES.values()
)

print(
    "\nFULL-TME CANDIDATE PATHS: PASS"
)

In [ ]:
# =============================================================================
# Step30A Cell 12
# Backed audit of candidate full-TME objects
# =============================================================================

malignant_cells = set(
    adata_malignant.obs_names.astype(str)
)

audit_rows = []

annotation_tokens = [
    "cell",
    "type",
    "major",
    "lineage",
    "annot",
    "malig",
    "tumor",
    "patient",
    "sample"
]

for name, path in CANDIDATES.items():

    a = ad.read_h5ad(
        path,
        backed="r"
    )

    obs_cols = list(
        map(
            str,
            a.obs.columns
        )
    )

    likely_cols = [
        c
        for c in obs_cols
        if any(
            token in c.lower()
            for token in annotation_tokens
        )
    ]

    candidate_cells = set(
        a.obs_names.astype(str)
    )

    overlap = len(
        malignant_cells
        &
        candidate_cells
    )

    audit_rows.append({
        "Candidate":
            name,

        "Cells":
            a.n_obs,

        "Genes":
            a.n_vars,

        "Frozen_malignant_overlap":
            overlap,

        "Frozen_malignant_overlap_pct":
            overlap / 12057,

        "N_obs_columns":
            len(obs_cols),

        "Raw_present":
            a.raw is not None,

        "Layers":
            ",".join(
                list(
                    a.layers.keys()
                )
            ),

        "Likely_annotation_columns":
            " | ".join(
                likely_cols
            )
    })

    a.file.close()


candidate_audit = pd.DataFrame(
    audit_rows
)

display(
    candidate_audit
)

In [ ]:
# =============================================================================
# Step30A Cell 13
# Inspect Step22 full-TME candidate annotations
# =============================================================================

STEP22_FULL = (
    CANDIDATES[
        "Step22_FinalMalignancy"
    ]
)

a22 = ad.read_h5ad(
    STEP22_FULL,
    backed="r"
)

print(
    "Shape:",
    a22.shape
)

print(
    "\nOBS columns:"
)

for col in a22.obs.columns:
    print(col)

In [ ]:
interesting_cols = [
    col
    for col in a22.obs.columns
    if any(
        token in col.lower()
        for token in [
            "patient",
            "sample",
            "cell",
            "major",
            "type",
            "annot",
            "malig",
            "tumor",
            "lineage"
        ]
    )
]

print(
    "\nPotential annotation columns:"
)

for col in interesting_cols:

    n_unique = (
        a22.obs[col]
        .nunique(
            dropna=False
        )
    )

    print(
        f"\n--- {col} ---"
    )

    print(
        "N unique:",
        n_unique
    )

    if n_unique <= 50:

        print(
            a22.obs[col]
            .value_counts(
                dropna=False
            )
            .head(50)
        )

a22.file.close()

In [ ]:
# =============================================================================
# Step30A Cell 14
# Frozen malignant-cell mapping feasibility
# =============================================================================

a22 = ad.read_h5ad(
    STEP22_FULL,
    backed="r"
)

step22_cells = set(
    a22.obs_names.astype(str)
)

frozen_cells = set(
    adata_malignant.obs_names.astype(str)
)

matched = (
    frozen_cells
    &
    step22_cells
)

missing = (
    frozen_cells
    -
    step22_cells
)

print(
    "Frozen malignant cells:",
    len(frozen_cells)
)

print(
    "Matched to Step22:",
    len(matched)
)

print(
    "Missing from Step22:",
    len(missing)
)

print(
    "Mapping fraction:",
    len(matched)
    /
    len(frozen_cells)
)

if missing:

    print(
        "\nFirst missing cells:"
    )

    print(
        list(
            sorted(missing)
        )[:20]
    )

a22.file.close()

In [ ]:
# =============================================================================
# Step30A Cell 15
# Audit Step21 full-TME master candidate
# =============================================================================

STEP21_FULL = (
    CANDIDATES[
        "Step21_MajorAnnotation"
    ]
)

a21 = ad.read_h5ad(
    STEP21_FULL,
    backed="r"
)

print(
    "Step21 shape:",
    a21.shape
)

print(
    "\nOBS columns:"
)

for col in a21.obs.columns:
    print(col)

print(
    "\nPatients:"
)

print(
    a21.obs[
        "patient"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nSamples:"
)

print(
    a21.obs[
        "sample"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nCell types:"
)

print(
    a21.obs[
        "celltype"
    ].value_counts(
        dropna=False
    )
)

a21.file.close()

In [ ]:
# =============================================================================
# Step30A Cell 16
# Cell-universe relationship audit
# =============================================================================

a21 = ad.read_h5ad(
    STEP21_FULL,
    backed="r"
)

a22 = ad.read_h5ad(
    STEP22_FULL,
    backed="r"
)

cells21 = set(
    a21.obs_names.astype(str)
)

cells22 = set(
    a22.obs_names.astype(str)
)

cells27 = set(
    adata_malignant.obs_names.astype(str)
)

print(
    "Step21 cells:",
    len(cells21)
)

print(
    "Step22 cells:",
    len(cells22)
)

print(
    "Step27 frozen malignant:",
    len(cells27)
)

print(
    "\nStep22 contained in Step21:",
    cells22.issubset(cells21)
)

print(
    "Step27 contained in Step21:",
    cells27.issubset(cells21)
)

print(
    "Step27 contained in Step22:",
    cells27.issubset(cells22)
)

print(
    "\nStep21 ∩ Step22:",
    len(
        cells21 & cells22
    )
)

print(
    "Step21 ∩ Step27:",
    len(
        cells21 & cells27
    )
)

print(
    "Step22 ∩ Step27:",
    len(
        cells22 & cells27
    )
)

print(
    "\nStep21-only cells:",
    len(
        cells21 - cells22
    )
)

a21.file.close()
a22.file.close()

In [ ]:
# =============================================================================
# Step30A Cell 17
# Step21 expression-matrix audit — FIXED
# =============================================================================

import scipy.sparse as sp

X = a21.X

print(
    "Shape:",
    a21.shape
)

print(
    "Sparse:",
    sp.issparse(X)
)

print(
    "Raw present:",
    a21.raw is not None
)

print(
    "Layers:",
    list(
        a21.layers.keys()
    )
)

if sp.issparse(X):
    data = X.data
else:
    data = np.asarray(X).ravel()

print(
    "\nMinimum:",
    float(data.min())
)

print(
    "Maximum:",
    float(data.max())
)

print(
    "Mean nonzero:",
    float(data.mean())
)

integer_like_fraction = float(
    np.mean(
        np.isclose(
            data,
            np.round(data)
        )
    )
)

print(
    "Integer-like fraction:",
    integer_like_fraction
)

print(
    "\nGenes unique:",
    a21.var_names.is_unique
)

print(
    "First genes:",
    list(
        a21.var_names[:20]
    )
)

print(
    "\nSTEP30A EXPRESSION AUDIT: PASS"
)

In [ ]:
# =============================================================================
# Step30A Cell 18
# Gene-universe compatibility audit
# =============================================================================

genes21 = set(
    a21.var_names.astype(str)
)

genes27 = set(
    adata_malignant.var_names.astype(str)
)

print(
    "Step21 genes:",
    len(genes21)
)

print(
    "Step27 genes:",
    len(genes27)
)

print(
    "Gene overlap:",
    len(
        genes21 & genes27
    )
)

print(
    "Same gene set:",
    genes21 == genes27
)

print(
    "Same gene order:",
    np.array_equal(
        a21.var_names.astype(str),
        adata_malignant.var_names.astype(str)
    )
)

In [ ]:
# =============================================================================
# Step30A Cell 19
# Prepare Step22 malignancy metadata for back-mapping
# =============================================================================

a22 = sc.read_h5ad(
    STEP22_FULL
)

malignancy_cols = [
    "malignant_status",
    "malignancy_confidence",
    "malignancy_basis"
]

for col in malignancy_cols:

    assert col in a22.obs.columns

step22_malignancy = (
    a22.obs[
        malignancy_cols
    ]
    .copy()
)

print(
    "Step22 malignancy metadata:",
    step22_malignancy.shape
)

display(
    step22_malignancy.head()
)

In [ ]:
# =============================================================================
# Step30A Cell 20
# Prepare frozen malignant-state metadata
# =============================================================================

frozen_cols = [
    "patient",
    "final_malignant_state_id",
    "MS6_anchored_pseudotime",
    "MS3_conserved_Core50_score",
    "Program__Proliferation_G2M",
    "DeepProgram__Mitotic_Entry",
    "DeepProgram__Spindle_Chromosome_Segregation",
    "DeepProgram__Hepatic_Differentiation",
    "DeepProgram__Bile_Acid_Metabolism"
]

missing = [
    c
    for c in frozen_cols
    if c not in adata_malignant.obs.columns
]

print(
    "Missing frozen fields:",
    missing
)

assert len(missing) == 0

frozen_meta = (
    adata_malignant.obs[
        frozen_cols
    ]
    .copy()
)

print(
    "Frozen malignant metadata:",
    frozen_meta.shape
)

display(
    frozen_meta.head()
)

In [ ]:
# =============================================================================
# Step30A Cell 21
# Build full-TME master annotation
# =============================================================================

adata30 = a21.copy()

# -------------------------------------------------------------------------
# Add Step22 malignancy metadata
# -------------------------------------------------------------------------

for col in malignancy_cols:

    adata30.obs[col] = (
        step22_malignancy[
            col
        ]
        .reindex(
            adata30.obs_names
        )
    )


# -------------------------------------------------------------------------
# Add frozen Step27 malignant-state metadata
# -------------------------------------------------------------------------

for col in frozen_cols:

    if col == "patient":
        continue

    new_col = (
        "frozen_" + col
        if not col.startswith("final_")
        else col
    )

    adata30.obs[
        new_col
    ] = (
        frozen_meta[
            col
        ]
        .reindex(
            adata30.obs_names
        )
    )


print(
    "Step30 master shape:",
    adata30.shape
)

print(
    "\nFrozen states mapped:"
)

print(
    adata30.obs[
        "final_malignant_state_id"
    ].value_counts(
        dropna=False
    )
)

assert (
    adata30.obs[
        "final_malignant_state_id"
    ].notna().sum()
    ==
    12057
)

print(
    "\nFROZEN STATE BACK-MAPPING: PASS"
)

In [ ]:
# =============================================================================
# Step30A Cell 22
# Construct communication cell labels
# =============================================================================

adata30.obs[
    "communication_group"
] = (
    adata30.obs[
        "celltype"
    ]
    .astype(str)
)


frozen_mask = (
    adata30.obs[
        "final_malignant_state_id"
    ]
    .notna()
)


adata30.obs.loc[
    frozen_mask,
    "communication_group"
] = (
    "Malignant_"
    +
    adata30.obs.loc[
        frozen_mask,
        "final_malignant_state_id"
    ].astype(str)
)


print(
    "Communication groups:"
)

print(
    adata30.obs[
        "communication_group"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30A Cell 23
# Patient coverage of communication groups
# =============================================================================

group_patient = pd.crosstab(
    adata30.obs[
        "communication_group"
    ],
    adata30.obs[
        "patient"
    ]
)

display(
    group_patient
)

group_patient.to_csv(
    TABLE_DIR /
    "Step30A_CommunicationGroup_by_Patient.csv"
)

In [ ]:
# =============================================================================
# Step30A Cell 24
# Verify existing Step30 full-TME master object
# DO NOT overwrite frozen output
# =============================================================================

STEP30_MASTER = (
    DATA_DIR /
    "GSE149614_STEP30_FULL_TME_MASTER.h5ad"
)

if not STEP30_MASTER.exists():
    raise FileNotFoundError(
        f"Step30 master does not exist:\n{STEP30_MASTER}"
    )

saved30 = ad.read_h5ad(
    STEP30_MASTER,
    backed="r"
)

print(
    "Existing Step30 master:"
)

print(
    STEP30_MASTER
)

print(
    "\nShape:",
    saved30.shape
)

required_cols = [
    "patient",
    "sample",
    "celltype",
    "communication_group",
    "final_malignant_state_id"
]

missing_cols = [
    c
    for c in required_cols
    if c not in saved30.obs.columns
]

print(
    "Missing required columns:",
    missing_cols
)

n_frozen_saved = int(
    saved30.obs[
        "final_malignant_state_id"
    ].notna().sum()
)

n_groups_saved = int(
    saved30.obs[
        "communication_group"
    ].nunique()
)

print(
    "Frozen malignant cells:",
    n_frozen_saved
)

print(
    "Communication groups:",
    n_groups_saved
)

same_cells = np.array_equal(
    saved30.obs_names.astype(str),
    adata30.obs_names.astype(str)
)

same_genes = np.array_equal(
    saved30.var_names.astype(str),
    adata30.var_names.astype(str)
)

print(
    "Same cell order as current object:",
    same_cells
)

print(
    "Same gene order as current object:",
    same_genes
)

MASTER_VERIFY_PASS = (
    saved30.shape == (71782, 25712)
    and
    n_frozen_saved == 12057
    and
    n_groups_saved == 14
    and
    len(missing_cols) == 0
    and
    same_cells
    and
    same_genes
)

print(
    "\nMASTER_VERIFY_PASS:",
    MASTER_VERIFY_PASS
)

saved30.file.close()

assert MASTER_VERIFY_PASS

print(
    "\nSTEP30 MASTER: VERIFIED / NOT OVERWRITTEN"
)

In [ ]:
# =============================================================================
# Step30A Cell 25
# REPORT — Full-TME master input lock
# =============================================================================

n_frozen = int(
    adata30.obs[
        "final_malignant_state_id"
    ].notna().sum()
)

n_groups = int(
    adata30.obs[
        "communication_group"
    ].nunique()
)

STEP30A_PASS = (
    adata30.n_obs == 71782
    and
    n_frozen == 12057
    and
    adata30.var_names.is_unique
)


print("=" * 100)

print(
    "STEP30A FULL-TME MASTER INPUT LOCK REPORT"
)

print("=" * 100)


print(
    "\nMaster source:"
)

print(
    STEP21_FULL
)


print(
    "\nCells:",
    adata30.n_obs
)

print(
    "Genes:",
    adata30.n_vars
)

print(
    "Patients:",
    adata30.obs[
        "patient"
    ].nunique()
)

print(
    "Samples:",
    adata30.obs[
        "sample"
    ].nunique()
)


print(
    "\nFrozen malignant cells mapped:",
    n_frozen
)

print(
    "Communication groups:",
    n_groups
)


print(
    "\nCommunication group counts:"
)

print(
    adata30.obs[
        "communication_group"
    ].value_counts()
)


print(
    "\nPatient coverage:"
)

display(
    group_patient
)


print(
    "\nSTEP30A_PASS:",
    STEP30A_PASS
)


print(
    "\nDesign boundary:"
)

print(
    "Step21 full-TME expression and major-cell annotations were retained."
)

print(
    "Step22 malignancy annotations were mapped by exact Cell ID."
)

print(
    "Step27 frozen MS1-MS8 states and biological scores were mapped "
    "only to the 12,057 frozen malignant cells."
)

print(
    "No re-clustering, re-annotation, or communication inference "
    "was performed."
)

print(
    "\nSTEP30A MASTER INPUT LOCK COMPLETE"
)

## Step30B — Sample-aware communication design

### Objective

Determine which biological samples can support local TME-to-malignant communication analysis on the frozen MS6–MS5–MS3 axis.

### Principle

Cell–cell communication will be inferred within biologically matched samples.

Cells from different anatomical samples of the same patient will not be pooled as if they occupied the same local microenvironment.

The frozen MS1–MS8 malignant states will not be redefined.

In [ ]:
# =============================================================================
# Step30B Cell 27
# Sample-site map
# =============================================================================

sample_site = (
    adata30.obs[
        ["patient", "sample", "site"]
    ]
    .drop_duplicates()
    .sort_values(
        ["patient", "sample"]
    )
    .reset_index(drop=True)
)

display(sample_site)

print("\nSite distribution:")

print(
    adata30.obs[
        "site"
    ].value_counts(
        dropna=False
    )
)

In [ ]:
# =============================================================================
# Step30B Cell 28
# Frozen malignant states by biological sample
# =============================================================================

malignant_mask = (
    adata30.obs[
        "final_malignant_state_id"
    ].notna()
)

state_by_sample = pd.crosstab(
    adata30.obs.loc[
        malignant_mask,
        "sample"
    ],
    adata30.obs.loc[
        malignant_mask,
        "final_malignant_state_id"
    ]
)

display(state_by_sample)

state_by_sample.to_csv(
    TABLE_DIR /
    "Step30B_FrozenState_by_Sample.csv"
)

In [ ]:
# =============================================================================
# Step30B Cell 29
# MS3-MS5-MS6 axis by sample
# =============================================================================

AXIS_STATES = [
    "MS3",
    "MS5",
    "MS6"
]

axis_mask = (
    adata30.obs[
        "final_malignant_state_id"
    ].isin(
        AXIS_STATES
    )
)

axis_state_sample = (
    adata30.obs.loc[
        axis_mask,
        [
            "patient",
            "sample",
            "site",
            "final_malignant_state_id"
        ]
    ]
    .groupby(
        [
            "patient",
            "sample",
            "site",
            "final_malignant_state_id"
        ],
        observed=True
    )
    .size()
    .reset_index(
        name="N_cells"
    )
)

axis_state_sample = (
    axis_state_sample.loc[
        axis_state_sample[
            "N_cells"
        ] > 0
    ]
    .reset_index(drop=True)
)

display(axis_state_sample)

In [ ]:
# =============================================================================
# Step30B Cell 30
# Identify axis-bearing samples
# =============================================================================

axis_samples = (
    axis_state_sample[
        "sample"
    ]
    .drop_duplicates()
    .tolist()
)

print(
    "Axis-bearing samples:"
)

for sample in axis_samples:
    print(sample)

print(
    "\nN axis-bearing samples:",
    len(axis_samples)
)

In [ ]:
# =============================================================================
# Step30B Cell 31
# Local TME composition in axis-bearing samples
# =============================================================================

axis_sample_mask = (
    adata30.obs[
        "sample"
    ].isin(
        axis_samples
    )
)

axis_sample_tme = pd.crosstab(
    adata30.obs.loc[
        axis_sample_mask,
        "communication_group"
    ],
    adata30.obs.loc[
        axis_sample_mask,
        "sample"
    ]
)

display(axis_sample_tme)

axis_sample_tme.to_csv(
    TABLE_DIR /
    "Step30B_AxisSamples_CommunicationGroups.csv"
)

In [ ]:
# =============================================================================
# Step30B Cell 32
# Sample-level communication eligibility audit
# =============================================================================

SENDER_GROUPS = [
    "T/NK",
    "Myeloid",
    "B",
    "Fibroblast",
    "Endothelial"
]

TARGET_GROUPS = [
    "Malignant_MS3",
    "Malignant_MS5",
    "Malignant_MS6"
]

rows = []

for sample in axis_samples:

    sub = adata30.obs.loc[
        adata30.obs[
            "sample"
        ] == sample
    ]

    row = {
        "sample": sample,
        "patient": sub["patient"].iloc[0],
        "site": sub["site"].iloc[0],
        "total_cells": len(sub)
    }

    counts = (
        sub[
            "communication_group"
        ]
        .value_counts()
    )

    for group in (
        SENDER_GROUPS +
        TARGET_GROUPS
    ):

        row[group] = int(
            counts.get(
                group,
                0
            )
        )

    rows.append(row)


sample_eligibility = pd.DataFrame(
    rows
)

display(
    sample_eligibility
)

sample_eligibility.to_csv(
    TABLE_DIR /
    "Step30B_Sample_Eligibility.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step30B Cell 33
# Samples containing the complete MS3-MS5-MS6 axis
# =============================================================================

complete_axis = (
    sample_eligibility.loc[
        (
            sample_eligibility[
                "Malignant_MS3"
            ] > 0
        )
        &
        (
            sample_eligibility[
                "Malignant_MS5"
            ] > 0
        )
        &
        (
            sample_eligibility[
                "Malignant_MS6"
            ] > 0
        )
    ]
    .copy()
)

print(
    "Samples containing MS3 + MS5 + MS6:"
)

display(
    complete_axis
)

In [ ]:
# =============================================================================
# Step30B Cell 34
# REPORT — sample-aware communication design
# =============================================================================

print("=" * 100)

print(
    "STEP30B SAMPLE-AWARE COMMUNICATION DESIGN REPORT"
)

print("=" * 100)


print(
    "\nAxis-state distribution:"
)

display(
    axis_state_sample
)


print(
    "\nAxis-bearing sample composition:"
)

display(
    sample_eligibility
)


print(
    "\nSamples containing all MS3/MS5/MS6 states:"
)

display(
    complete_axis
)


print(
    "\nDesign rules:"
)

print(
    "1. Communication inference will be performed within biological samples."
)

print(
    "2. Different samples from the same patient will not be pooled as one local TME."
)

print(
    "3. MS3/MS5/MS6 labels remain frozen from Step27."
)

print(
    "4. Cross-patient consistency will be prioritized over pooled-cell significance."
)

print(
    "5. No communication inference has yet been performed."
)


print(
    "\nSTEP30B DESIGN AUDIT COMPLETE"
)

## Step30B — Final Design Lock

### Primary discovery cohort

Two independent primary-tumor samples containing the complete frozen
MS6–MS5–MS3 malignant-state axis:

- HCC09T
- HCC10T

### Primary TME senders

- T/NK
- Myeloid
- B
- Fibroblast

### Primary malignant receivers

- Malignant_MS6
- Malignant_MS5
- Malignant_MS3

### Primary inference principle

Communication inference will be performed separately within HCC09T and HCC10T.

Interactions will subsequently be evaluated for concordance across the two
independent primary tumors.

### Secondary analyses

HCC10L will be retained as a metastatic-site sensitivity analysis and will
not be treated as an independent patient replicate.

Endothelial signaling will be considered secondary because HCC10T contains
only 9 endothelial cells.

HCC08T/HCC08P may later provide supplementary evidence for MS3–MS5-associated
signals but cannot test the complete MS6–MS5–MS3 axis.

### Claim boundary

Cross-sample ligand–receptor inference indicates communication compatibility
and expression-supported signaling hypotheses. It does not demonstrate
physical ligand delivery, receptor activation, causal TF regulation, or
state conversion.

In [ ]:
# =============================================================================
# Step30B Cell 36
# Save final communication design lock
# =============================================================================

import json

step30_design = {
    "primary_samples": [
        "HCC09T",
        "HCC10T"
    ],

    "primary_sites": [
        "Tumor"
    ],

    "primary_senders": [
        "T/NK",
        "Myeloid",
        "B",
        "Fibroblast"
    ],

    "primary_targets": [
        "Malignant_MS6",
        "Malignant_MS5",
        "Malignant_MS3"
    ],

    "secondary_samples": [
        "HCC10L"
    ],

    "secondary_senders": [
        "Endothelial",
        "Hepatocyte"
    ],

    "formal_min_cells": 20,
    "expr_prop": 0.10,

    "primary_method":
        "LIANA Rank_Aggregate consensus",

    "primary_resource":
        "consensus",

    "formal_permutations": 1000,

    "claim_boundary":
        "Communication compatibility, not causal signaling or state conversion."
}

STEP30_DESIGN_LOCK = (
    AUDIT_DIR /
    "Step30B_FINAL_CommunicationDesign.json"
)

with open(
    STEP30_DESIGN_LOCK,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step30_design,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    STEP30_DESIGN_LOCK
)

print(
    "\nSTEP30B FINAL DESIGN LOCK: COMPLETE"
)

In [ ]:
# =============================================================================
# Step30C Cell 37
# Reload frozen Step30 master
# =============================================================================

import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import liana as li

adata_master = sc.read_h5ad(
    STEP30_MASTER
)

print(
    "Master:",
    adata_master.shape
)

assert (
    adata_master.n_obs == 71782
)

assert (
    adata_master.obs[
        "final_malignant_state_id"
    ].notna().sum()
    == 12057
)

print(
    "\nSTEP30 MASTER RELOAD: PASS"
)

In [ ]:
# =============================================================================
# Step30C Cell 38
# Construct primary communication object
# =============================================================================

PRIMARY_SAMPLES = [
    "HCC09T",
    "HCC10T"
]

PRIMARY_SENDERS = [
    "T/NK",
    "Myeloid",
    "B",
    "Fibroblast"
]

PRIMARY_TARGETS = [
    "Malignant_MS6",
    "Malignant_MS5",
    "Malignant_MS3"
]

PRIMARY_GROUPS = (
    PRIMARY_SENDERS +
    PRIMARY_TARGETS
)

primary_mask = (
    adata_master.obs[
        "sample"
    ].isin(
        PRIMARY_SAMPLES
    )
    &
    adata_master.obs[
        "communication_group"
    ].isin(
        PRIMARY_GROUPS
    )
)

adata_primary = (
    adata_master[
        primary_mask
    ]
    .copy()
)

print(
    "Primary object:",
    adata_primary.shape
)

print(
    "\nSamples:"
)

print(
    adata_primary.obs[
        "sample"
    ].value_counts()
)

print(
    "\nGroups:"
)

print(
    pd.crosstab(
        adata_primary.obs[
            "communication_group"
        ],
        adata_primary.obs[
            "sample"
        ]
    )
)

In [ ]:
# =============================================================================
# Step30C Cell 39
# Expression-matrix audit before LIANA
# =============================================================================

X = adata_primary.X

print(
    "Shape:",
    adata_primary.shape
)

print(
    "Sparse:",
    sp.issparse(X)
)

print(
    "Raw present:",
    adata_primary.raw is not None
)

print(
    "Layers:",
    list(
        adata_primary.layers.keys()
    )
)

if sp.issparse(X):
    values = X.data
else:
    values = np.asarray(X).ravel()

print(
    "\nMinimum:",
    float(values.min())
)

print(
    "Maximum:",
    float(values.max())
)

print(
    "Mean nonzero:",
    float(values.mean())
)

integer_like_fraction = float(
    np.mean(
        np.isclose(
            values,
            np.round(values)
        )
    )
)

print(
    "Integer-like fraction:",
    integer_like_fraction
)

print(
    "\nUnique genes:",
    adata_primary.var_names.is_unique
)

print(
    "Example genes:",
    list(
        adata_primary.var_names[:20]
    )
)

print(
    "\nSTEP30C EXPRESSION AUDIT COMPLETE"
)

In [ ]:
# =============================================================================
# Step30C Cell 40
# Pre-register TME -> malignant source-target pairs
# =============================================================================

groupby_pairs = pd.DataFrame(
    [
        {
            "source": sender,
            "target": target
        }

        for sender in PRIMARY_SENDERS

        for target in PRIMARY_TARGETS
    ]
)

print(
    "Primary source-target pairs:"
)

display(
    groupby_pairs
)

print(
    "\nN pairs:",
    len(groupby_pairs)
)

assert len(groupby_pairs) == 12

In [ ]:
# =============================================================================
# Step30C Cell 41
# Formal minimum-cell eligibility
# =============================================================================

FORMAL_MIN_CELLS = 20

primary_counts = pd.crosstab(
    adata_primary.obs[
        "communication_group"
    ],
    adata_primary.obs[
        "sample"
    ]
)

display(
    primary_counts
)

eligibility = (
    primary_counts
    >= FORMAL_MIN_CELLS
)

print(
    "\nEligibility at min_cells=20:"
)

display(
    eligibility
)

PRIMARY_CELLCOUNT_PASS = bool(
    eligibility.loc[
        PRIMARY_GROUPS,
        PRIMARY_SAMPLES
    ]
    .all()
    .all()
)

print(
    "\nPRIMARY_CELLCOUNT_PASS:",
    PRIMARY_CELLCOUNT_PASS
)

assert PRIMARY_CELLCOUNT_PASS

In [ ]:
# =============================================================================
# Step30C Cell 42
# LIANA ligand-receptor resource audit
# =============================================================================

resources = li.rs.show_resources()

print(
    resources
)

resource_strings = [
    str(x).lower()
    for x in resources
]

CONSENSUS_RESOURCE_PASS = any(
    "consensus" in x
    for x in resource_strings
)

print(
    "\nConsensus resource available:",
    CONSENSUS_RESOURCE_PASS
)

assert CONSENSUS_RESOURCE_PASS

print(
    "\nLIANA RESOURCE AUDIT: PASS"
)

In [ ]:
# =============================================================================
# Step30C Cell 43
# LIANA smoke test — NOT FOR BIOLOGICAL INTERPRETATION
# =============================================================================

adata_smoke = (
    adata_primary[
        adata_primary.obs[
            "sample"
        ] == "HCC09T"
    ]
    .copy()
)

li.mt.rank_aggregate(
    adata_smoke,

    groupby="communication_group",

    resource_name="consensus",

    expr_prop=0.10,

    min_cells=20,

    groupby_pairs=groupby_pairs,

    use_raw=False,

    n_perms=10,

    seed=42,

    n_jobs=4,

    return_all_lrs=False,

    verbose=True
)

smoke_res = (
    adata_smoke.uns[
        "liana_res"
    ]
)

print(
    "\nSmoke-result shape:",
    smoke_res.shape
)

print(
    "\nColumns:"
)

print(
    smoke_res.columns.tolist()
)

display(
    smoke_res.head(10)
)

assert len(smoke_res) > 0

print(
    "\nLIANA SMOKE TEST: PASS"
)

In [ ]:
# =============================================================================
# Step30C Cell 44
# FORMAL LIANA consensus inference by biological sample
# =============================================================================

li.mt.rank_aggregate.by_sample(
    adata_primary,

    sample_key="sample",

    groupby="communication_group",

    resource_name="consensus",

    expr_prop=0.10,

    min_cells=20,

    groupby_pairs=groupby_pairs,

    use_raw=False,

    n_perms=1000,

    seed=42,

    n_jobs=4,

    return_all_lrs=False,

    verbose=True
)

liana_primary = (
    adata_primary.uns[
        "liana_res"
    ]
    .copy()
)

print(
    "\nFormal LIANA result:",
    liana_primary.shape
)

print(
    "\nColumns:"
)

print(
    liana_primary.columns.tolist()
)

print(
    "\nSamples:"
)

print(
    liana_primary[
        "sample"
    ].value_counts()
)

print(
    "\nFORMAL LIANA PRIMARY INFERENCE: COMPLETE"
)

In [ ]:
# =============================================================================
# Step30C Cell 45
# Save formal LIANA primary results
# =============================================================================

LIANA_PRIMARY_CSV = (
    TABLE_DIR /
    "Step30C_LIANA_Primary_HCC09T_HCC10T.csv"
)

LIANA_PRIMARY_PARQUET = (
    TABLE_DIR /
    "Step30C_LIANA_Primary_HCC09T_HCC10T.parquet"
)

liana_primary.to_csv(
    LIANA_PRIMARY_CSV,
    index=False
)

liana_primary.to_parquet(
    LIANA_PRIMARY_PARQUET,
    index=False
)

print(
    "CSV:",
    LIANA_PRIMARY_CSV
)

print(
    "Parquet:",
    LIANA_PRIMARY_PARQUET
)

print(
    "\nFORMAL LIANA RESULTS SAVED"
)

In [ ]:
# =============================================================================
# Step30C Cell 46
# Initial formal-result audit
# =============================================================================

print(
    "Result shape:",
    liana_primary.shape
)

print(
    "\nColumns:"
)

for col in liana_primary.columns:
    print(col)

print(
    "\nResults per sample:"
)

print(
    liana_primary[
        "sample"
    ].value_counts()
)

print(
    "\nResults per receiver:"
)

print(
    liana_primary[
        "target"
    ].value_counts()
)

print(
    "\nResults per sender:"
)

print(
    liana_primary[
        "source"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30C Cell 47
# REPORT — primary LIANA inference
# =============================================================================

print("=" * 100)

print(
    "STEP30C PRIMARY LIANA COMMUNICATION REPORT"
)

print("=" * 100)

print(
    "\nPrimary samples:"
)

print(
    PRIMARY_SAMPLES
)

print(
    "\nPrimary senders:"
)

print(
    PRIMARY_SENDERS
)

print(
    "\nPrimary receivers:"
)

print(
    PRIMARY_TARGETS
)

print(
    "\nExpression threshold:"
)

print(
    "expr_prop = 0.10"
)

print(
    "min_cells = 20"
)

print(
    "permutations = 1000"
)

print(
    "resource = LIANA consensus"
)

print(
    "\nTotal inferred interactions:",
    len(liana_primary)
)

print(
    "\nInteractions by sample:"
)

print(
    liana_primary[
        "sample"
    ].value_counts()
)

print(
    "\nResult columns:"
)

print(
    liana_primary.columns.tolist()
)

print(
    "\nTop rows by magnitude_rank:"
)

display(
    liana_primary
    .sort_values(
        "magnitude_rank"
    )
    .head(30)
)

print(
    "\nSTEP30C PRIMARY LIANA INFERENCE COMPLETE"
)

print(
    "No biological interaction has yet been declared as conserved."
)

## Step30D — Cross-patient conserved and state-skewed communication

### Objective

Identify ligand–receptor interactions that are reproducibly detected in both
HCC09T and HCC10T and determine whether their communication magnitude shows
a concordant bias toward MS6, MS5, or MS3.

### Primary principles

1. Exact ligand–receptor conservation across two independent primary tumors
   is prioritized over pooled-cell significance.

2. Communication magnitude and permutation-based support are evaluated
   separately.

3. State preference is assessed within the same sender and ligand–receptor
   pair across the frozen MS6, MS5, and MS3 receivers.

4. A concordant preference does not establish biological specificity or
   causal state induction.

5. Candidate mechanisms will subsequently require integration with the
   frozen Step29 regulatory architecture.

In [ ]:
# =============================================================================
# Step30D Cell 49
# Result uniqueness audit and permutation FDR
# =============================================================================

from statsmodels.stats.multitest import multipletests

LR_KEY = [
    "source",
    "target",
    "ligand_complex",
    "receptor_complex"
]

duplicate_n = int(
    liana_primary.duplicated(
        subset=["sample"] + LR_KEY
    ).sum()
)

print(
    "Duplicate sample-specific LR rows:",
    duplicate_n
)

assert duplicate_n == 0


liana30d = (
    liana_primary
    .copy()
)

liana30d[
    "cellphone_fdr"
] = np.nan


for sample, idx in (
    liana30d
    .groupby("sample")
    .groups
    .items()
):

    idx = list(idx)

    p = pd.to_numeric(
        liana30d.loc[
            idx,
            "cellphone_pvals"
        ],
        errors="coerce"
    )

    valid = p.notna()

    q = np.full(
        len(idx),
        np.nan
    )

    q[
        valid.to_numpy()
    ] = multipletests(
        p.loc[valid].clip(0, 1),
        method="fdr_bh"
    )[1]

    liana30d.loc[
        idx,
        "cellphone_fdr"
    ] = q


print(
    "\nFDR < 0.05 by sample:"
)

print(
    liana30d
    .assign(
        FDR05=
        liana30d[
            "cellphone_fdr"
        ] < 0.05
    )
    .groupby("sample")[
        "FDR05"
    ]
    .sum()
)

print(
    "\nSTEP30D FDR AUDIT: PASS"
)

In [ ]:
# =============================================================================
# Step30D Cell 50
# Exact cross-patient LR intersection
# =============================================================================

METRIC_COLS = [
    "lr_means",
    "cellphone_pvals",
    "cellphone_fdr",
    "lr_logfc",
    "specificity_rank",
    "magnitude_rank"
]


h09 = (
    liana30d.loc[
        liana30d[
            "sample"
        ] == "HCC09T",
        LR_KEY + METRIC_COLS
    ]
    .copy()
)


h10 = (
    liana30d.loc[
        liana30d[
            "sample"
        ] == "HCC10T",
        LR_KEY + METRIC_COLS
    ]
    .copy()
)


conserved_exact = (
    h09.merge(
        h10,
        on=LR_KEY,
        how="inner",
        suffixes=(
            "_HCC09T",
            "_HCC10T"
        )
    )
)


conserved_exact[
    "both_fdr05"
] = (
    (
        conserved_exact[
            "cellphone_fdr_HCC09T"
        ] < 0.05
    )
    &
    (
        conserved_exact[
            "cellphone_fdr_HCC10T"
        ] < 0.05
    )
)


conserved_exact[
    "worst_magnitude_rank"
] = conserved_exact[
    [
        "magnitude_rank_HCC09T",
        "magnitude_rank_HCC10T"
    ]
].max(
    axis=1
)


conserved_exact[
    "mean_magnitude_rank"
] = conserved_exact[
    [
        "magnitude_rank_HCC09T",
        "magnitude_rank_HCC10T"
    ]
].mean(
    axis=1
)


conserved_exact[
    "mean_lr_means"
] = conserved_exact[
    [
        "lr_means_HCC09T",
        "lr_means_HCC10T"
    ]
].mean(
    axis=1
)


print(
    "HCC09T interactions:",
    len(h09)
)

print(
    "HCC10T interactions:",
    len(h10)
)

print(
    "Exact conserved interactions:",
    len(conserved_exact)
)

print(
    "Conserved + FDR<0.05 in both:",
    int(
        conserved_exact[
            "both_fdr05"
        ].sum()
    )
)


display(
    conserved_exact
    .sort_values(
        [
            "both_fdr05",
            "worst_magnitude_rank"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(30)
)

In [ ]:
# =============================================================================
# Step30D Cell 51
# Save exact conserved LR table
# =============================================================================

CONSERVED_LR_PATH = (
    TABLE_DIR /
    "Step30D_CrossPatient_ExactConserved_LR.csv"
)

conserved_exact.to_csv(
    CONSERVED_LR_PATH,
    index=False
)

print(
    "Saved:",
    CONSERVED_LR_PATH
)

In [ ]:
# =============================================================================
# Step30D Cell 52
# Within-sample malignant-state magnitude matrix
# =============================================================================

STATE_TARGETS = [
    "Malignant_MS6",
    "Malignant_MS5",
    "Malignant_MS3"
]

STATE_LR_KEY = [
    "sample",
    "source",
    "ligand_complex",
    "receptor_complex"
]


state_mag = (
    liana30d
    .pivot_table(
        index=STATE_LR_KEY,
        columns="target",
        values="lr_means",
        aggfunc="first"
    )
    .reset_index()
)


complete_triplet = (
    state_mag[
        STATE_TARGETS
    ]
    .notna()
    .all(axis=1)
)


state_mag = (
    state_mag.loc[
        complete_triplet
    ]
    .copy()
)


state_mag[
    "MS3_minus_MS6"
] = (
    state_mag[
        "Malignant_MS3"
    ]
    -
    state_mag[
        "Malignant_MS6"
    ]
)


state_mag[
    "MS5_minus_MS6"
] = (
    state_mag[
        "Malignant_MS5"
    ]
    -
    state_mag[
        "Malignant_MS6"
    ]
)


state_mag[
    "MS3_minus_MS5"
] = (
    state_mag[
        "Malignant_MS3"
    ]
    -
    state_mag[
        "Malignant_MS5"
    ]
)


state_mag[
    "preferred_target"
] = (
    state_mag[
        STATE_TARGETS
    ]
    .idxmax(
        axis=1
    )
)


values = (
    state_mag[
        STATE_TARGETS
    ]
    .to_numpy(
        dtype=float
    )
)

sorted_values = np.sort(
    values,
    axis=1
)


state_mag[
    "preference_margin"
] = (
    sorted_values[:, -1]
    -
    sorted_values[:, -2]
)


state_mag[
    "preference_margin_relative"
] = (
    state_mag[
        "preference_margin"
    ]
    /
    np.maximum(
        np.abs(
            sorted_values[:, -1]
        ),
        1e-12
    )
)


print(
    "Complete MS6/MS5/MS3 LR triplets:",
    len(state_mag)
)


print(
    "\nPreferred targets by sample:"
)

print(
    pd.crosstab(
        state_mag[
            "preferred_target"
        ],
        state_mag[
            "sample"
        ]
    )
)

In [ ]:
# =============================================================================
# Step30D Cell 53
# Cross-patient state-preference concordance
# =============================================================================

PREF_KEY = [
    "source",
    "ligand_complex",
    "receptor_complex"
]

PREF_METRICS = [
    "Malignant_MS6",
    "Malignant_MS5",
    "Malignant_MS3",
    "MS3_minus_MS6",
    "MS5_minus_MS6",
    "MS3_minus_MS5",
    "preferred_target",
    "preference_margin",
    "preference_margin_relative"
]


pref09 = (
    state_mag.loc[
        state_mag[
            "sample"
        ] == "HCC09T",
        PREF_KEY + PREF_METRICS
    ]
    .copy()
)


pref10 = (
    state_mag.loc[
        state_mag[
            "sample"
        ] == "HCC10T",
        PREF_KEY + PREF_METRICS
    ]
    .copy()
)


state_concordance = (
    pref09.merge(
        pref10,
        on=PREF_KEY,
        how="inner",
        suffixes=(
            "_HCC09T",
            "_HCC10T"
        )
    )
)


state_concordance[
    "concordant_preferred_target"
] = (
    state_concordance[
        "preferred_target_HCC09T"
    ]
    ==
    state_concordance[
        "preferred_target_HCC10T"
    ]
)


state_concordance[
    "min_preference_margin_relative"
] = state_concordance[
    [
        "preference_margin_relative_HCC09T",
        "preference_margin_relative_HCC10T"
    ]
].min(
    axis=1
)


for contrast in [
    "MS3_minus_MS6",
    "MS5_minus_MS6",
    "MS3_minus_MS5"
]:

    state_concordance[
        f"{contrast}_same_direction"
    ] = (
        np.sign(
            state_concordance[
                f"{contrast}_HCC09T"
            ]
        )
        ==
        np.sign(
            state_concordance[
                f"{contrast}_HCC10T"
            ]
        )
    )


print(
    "LR triplets available in both patients:",
    len(state_concordance)
)

print(
    "Same preferred malignant state:",
    int(
        state_concordance[
            "concordant_preferred_target"
        ].sum()
    )
)


print(
    "\nConcordant preferred-state counts:"
)

print(
    state_concordance.loc[
        state_concordance[
            "concordant_preferred_target"
        ],
        "preferred_target_HCC09T"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30D Cell 54
# Integrate exact LR conservation with concordant state preference
# =============================================================================

state_conserved = (
    state_concordance.loc[
        state_concordance[
            "concordant_preferred_target"
        ]
    ]
    .copy()
)


state_conserved[
    "target"
] = (
    state_conserved[
        "preferred_target_HCC09T"
    ]
)


state_conserved = (
    state_conserved.merge(
        conserved_exact,
        on=[
            "source",
            "target",
            "ligand_complex",
            "receptor_complex"
        ],
        how="inner"
    )
)


state_conserved = (
    state_conserved
    .sort_values(
        [
            "both_fdr05",
            "worst_magnitude_rank",
            "min_preference_margin_relative"
        ],
        ascending=[
            False,
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Conserved + concordant state-skewed candidates:",
    len(state_conserved)
)

print(
    "\nBy preferred state:"
)

print(
    state_conserved[
        "target"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30D Cell 55
# Top conserved state-skewed candidates
# =============================================================================

SHOW_COLS = [
    "source",
    "ligand_complex",
    "receptor_complex",
    "target",

    "lr_means_HCC09T",
    "lr_means_HCC10T",

    "cellphone_fdr_HCC09T",
    "cellphone_fdr_HCC10T",

    "magnitude_rank_HCC09T",
    "magnitude_rank_HCC10T",

    "worst_magnitude_rank",

    "preference_margin_relative_HCC09T",
    "preference_margin_relative_HCC10T",

    "min_preference_margin_relative"
]


for target in STATE_TARGETS:

    print(
        "\n" + "=" * 100
    )

    print(
        target
    )

    print(
        "=" * 100
    )

    temp = (
        state_conserved.loc[
            state_conserved[
                "target"
            ] == target
        ]
        .head(25)
    )

    display(
        temp[
            SHOW_COLS
        ]
    )

    temp.to_csv(
        TABLE_DIR /
        f"Step30D_Top25_{target}_Conserved_StateSkewed.csv",
        index=False
    )

In [ ]:
# =============================================================================
# Step30D Cell 56
# Audit prominent broad communication signals
# =============================================================================

AUDIT_LR = [
    ("TIMP1", "CD63"),
    ("LGALS1", "ITGB1"),
    ("HLA-A", "APLP2")
]

audit_rows = []


for ligand, receptor in AUDIT_LR:

    temp = (
        state_concordance.loc[
            (
                state_concordance[
                    "ligand_complex"
                ] == ligand
            )
            &
            (
                state_concordance[
                    "receptor_complex"
                ] == receptor
            )
        ]
        .copy()
    )

    if len(temp) > 0:

        audit_rows.append(
            temp
        )


if audit_rows:

    broad_signal_audit = pd.concat(
        audit_rows,
        ignore_index=True
    )

    display(
        broad_signal_audit
    )

else:

    broad_signal_audit = pd.DataFrame()

    print(
        "No requested audit LR pairs found."
    )

In [ ]:
# =============================================================================
# Step30D Cell 57
# Save cross-patient state-skewed result tables
# =============================================================================

STATE_CONCORDANCE_PATH = (
    TABLE_DIR /
    "Step30D_CrossPatient_StatePreference_All.csv"
)

STATE_CONSERVED_PATH = (
    TABLE_DIR /
    "Step30D_CrossPatient_Conserved_StateSkewed.csv"
)


state_concordance.to_csv(
    STATE_CONCORDANCE_PATH,
    index=False
)


state_conserved.to_csv(
    STATE_CONSERVED_PATH,
    index=False
)


print(
    "Saved:"
)

print(
    STATE_CONCORDANCE_PATH
)

print(
    STATE_CONSERVED_PATH
)

In [ ]:
# =============================================================================
# Step30D Cell 58
# REPORT — cross-patient conserved state-skewed communication
# =============================================================================

print("=" * 100)

print(
    "STEP30D CROSS-PATIENT CONSERVED / STATE-SKEWED COMMUNICATION REPORT"
)

print("=" * 100)


print(
    "\nExact LR interactions conserved across HCC09T and HCC10T:"
)

print(
    len(conserved_exact)
)


print(
    "\nExact conserved interactions with CellPhoneDB FDR < 0.05 in both:"
)

print(
    int(
        conserved_exact[
            "both_fdr05"
        ].sum()
    )
)


print(
    "\nLR triplets with MS6/MS5/MS3 estimates in both patients:"
)

print(
    len(state_concordance)
)


print(
    "\nConcordant preferred malignant state:"
)

print(
    int(
        state_concordance[
            "concordant_preferred_target"
        ].sum()
    )
)


print(
    "\nConcordant preference distribution:"
)

print(
    state_concordance.loc[
        state_concordance[
            "concordant_preferred_target"
        ],
        "preferred_target_HCC09T"
    ].value_counts()
)


print(
    "\nFinal conserved state-skewed candidate counts:"
)

print(
    state_conserved[
        "target"
    ].value_counts()
)


print(
    "\nTop MS6-skewed candidates:"
)

display(
    state_conserved.loc[
        state_conserved[
            "target"
        ] == "Malignant_MS6",
        SHOW_COLS
    ].head(15)
)


print(
    "\nTop MS5-skewed candidates:"
)

display(
    state_conserved.loc[
        state_conserved[
            "target"
        ] == "Malignant_MS5",
        SHOW_COLS
    ].head(15)
)


print(
    "\nTop MS3-skewed candidates:"
)

display(
    state_conserved.loc[
        state_conserved[
            "target"
        ] == "Malignant_MS3",
        SHOW_COLS
    ].head(15)
)


print(
    "\nInterpretation boundary:"
)

print(
    "Concordant target preference denotes a reproducible communication-magnitude skew, "
    "not proof of state-specific signaling or causal state induction."
)

print(
    "\nSTEP30D COMPLETE"
)

## Step30D.1 — Evidence-tier correction

The previous Step30D table identified ligand–receptor interactions with
concordant preferred malignant states across HCC09T and HCC10T.

However, concordant preference alone does not guarantee:

1. permutation-supported communication in both patients, or
2. a biologically meaningful magnitude difference between malignant states.

Therefore, Step30D.1 applies two additional evidence dimensions:

- CellPhoneDB FDR < 0.05 in both HCC09T and HCC10T
- cross-patient minimum relative state-preference margin

Effect-size tiers are used for interpretation only:

- Robust skew: >= 5% relative margin in both patients
- Moderate skew: >= 3% and < 5%
- Broad / weak skew: < 3%

These thresholds are descriptive effect-size tiers, not statistical
significance cutoffs.

In [ ]:
# =============================================================================
# Step30D.1 Cell 60
# Require permutation support in both patients
# =============================================================================

state_skew_fdr = (
    state_conserved.loc[
        state_conserved[
            "both_fdr05"
        ] == True
    ]
    .copy()
)

print(
    "Concordant preferred-state LR before FDR filter:",
    len(state_conserved)
)

print(
    "After FDR<0.05 in both patients:",
    len(state_skew_fdr)
)

print(
    "\nBy malignant target:"
)

print(
    state_skew_fdr[
        "target"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30D.1 Cell 61
# State-preference effect-size tiers
# =============================================================================

margin = (
    state_skew_fdr[
        "min_preference_margin_relative"
    ]
)

state_skew_fdr[
    "state_skew_tier"
] = np.select(
    [
        margin >= 0.05,
        margin >= 0.03
    ],
    [
        "Robust_ge_5pct",
        "Moderate_3to5pct"
    ],
    default="Broad_or_weak_lt_3pct"
)

tier_counts = pd.crosstab(
    state_skew_fdr[
        "target"
    ],
    state_skew_fdr[
        "state_skew_tier"
    ]
)

display(
    tier_counts
)

In [ ]:
# =============================================================================
# Step30D.1 Cell 62
# Effect-size threshold sensitivity
# =============================================================================

cutoffs = [
    0.00,
    0.01,
    0.03,
    0.05,
    0.10
]

rows = []

for cutoff in cutoffs:

    temp = (
        state_skew_fdr.loc[
            state_skew_fdr[
                "min_preference_margin_relative"
            ] >= cutoff
        ]
    )

    counts = (
        temp[
            "target"
        ]
        .value_counts()
    )

    rows.append({
        "Min_relative_margin":
            cutoff,

        "Total":
            len(temp),

        "MS6":
            int(
                counts.get(
                    "Malignant_MS6",
                    0
                )
            ),

        "MS5":
            int(
                counts.get(
                    "Malignant_MS5",
                    0
                )
            ),

        "MS3":
            int(
                counts.get(
                    "Malignant_MS3",
                    0
                )
            )
    })


margin_sensitivity = pd.DataFrame(
    rows
)

display(
    margin_sensitivity
)

margin_sensitivity.to_csv(
    TABLE_DIR /
    "Step30D1_StateSkew_MarginSensitivity.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step30D.1 Cell 63
# Robust cross-patient state-skewed candidates
# =============================================================================

robust_skew = (
    state_skew_fdr.loc[
        state_skew_fdr[
            "min_preference_margin_relative"
        ] >= 0.05
    ]
    .copy()
)

robust_skew = (
    robust_skew
    .sort_values(
        [
            "target",
            "min_preference_margin_relative",
            "worst_magnitude_rank"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Robust state-skewed LR:",
    len(robust_skew)
)

print(
    "\nBy target:"
)

print(
    robust_skew[
        "target"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30D.1 Cell 64
# Display robust candidates by malignant state
# =============================================================================

ROBUST_SHOW = [
    "source",
    "ligand_complex",
    "receptor_complex",
    "target",

    "lr_means_HCC09T",
    "lr_means_HCC10T",

    "cellphone_fdr_HCC09T",
    "cellphone_fdr_HCC10T",

    "magnitude_rank_HCC09T",
    "magnitude_rank_HCC10T",

    "preference_margin_relative_HCC09T",
    "preference_margin_relative_HCC10T",
    "min_preference_margin_relative"
]


for target in [
    "Malignant_MS6",
    "Malignant_MS5",
    "Malignant_MS3"
]:

    print(
        "\n" + "=" * 100
    )

    print(target)

    print("=" * 100)

    display(
        robust_skew.loc[
            robust_skew[
                "target"
            ] == target,
            ROBUST_SHOW
        ].head(30)
    )

In [ ]:
# =============================================================================
# Step30D.1 Cell 65
# Save corrected evidence tables
# =============================================================================

state_skew_fdr.to_csv(
    TABLE_DIR /
    "Step30D1_FDRSupported_StateSkew_All.csv",
    index=False
)

robust_skew.to_csv(
    TABLE_DIR /
    "Step30D1_Robust_StateSkew_ge5pct.csv",
    index=False
)

print(
    "STEP30D.1 corrected tables saved."
)

In [ ]:
# =============================================================================
# Step30D.1 Cell 66
# REPORT
# =============================================================================

print("=" * 100)

print(
    "STEP30D.1 CORRECTED STATE-SKEW EVIDENCE REPORT"
)

print("=" * 100)


print(
    "\nOriginal concordant preferred-state candidates:",
    len(state_conserved)
)


print(
    "FDR-supported in both patients:",
    len(state_skew_fdr)
)


print(
    "Robust >=5% skew in both patients:",
    len(robust_skew)
)


print(
    "\nFDR-supported candidates by state:"
)

print(
    state_skew_fdr[
        "target"
    ].value_counts()
)


print(
    "\nRobust candidates by state:"
)

print(
    robust_skew[
        "target"
    ].value_counts()
)


print(
    "\nThreshold sensitivity:"
)

display(
    margin_sensitivity
)


for target in [
    "Malignant_MS6",
    "Malignant_MS5",
    "Malignant_MS3"
]:

    print(
        f"\nTop robust {target}:"
    )

    display(
        robust_skew.loc[
            robust_skew[
                "target"
            ] == target,
            ROBUST_SHOW
        ].head(15)
    )


print(
    "\nSTEP30D.1 COMPLETE"
)

## Step30E — Receiver-side validation and intercellular prior audit

### Objectives

1. Validate whether cross-patient malignant-state communication skew is
   supported by concordant receptor-expression differences.

2. Separate robust (>=5%) from moderate (3–5%) state-skew evidence.

3. Audit whether prioritized ligand–receptor candidates have additional
   OmniPath intercellular ligand/receptor support.

4. Avoid assigning causal signaling mechanisms before receptor-side and
   prior-knowledge validation.

### Evidence hierarchy

- Primary: FDR < 0.05 in both patients + >=5% state skew in both patients
- Secondary: FDR < 0.05 in both patients + 3–5% state skew
- Broad/weak: <3% state skew

Absence of a robust MS5-skewed interaction is retained as a valid negative result.

In [ ]:
# =============================================================================
# Step30E Cell 68
# Freeze mechanistic candidate evidence tiers
# =============================================================================

candidate30e = (
    state_skew_fdr.loc[
        state_skew_fdr[
            "min_preference_margin_relative"
        ] >= 0.03
    ]
    .copy()
)

candidate30e[
    "evidence_tier"
] = np.where(
    candidate30e[
        "min_preference_margin_relative"
    ] >= 0.05,
    "Primary_Robust_ge5pct",
    "Secondary_Moderate_3to5pct"
)

print(
    "Step30E candidates:",
    len(candidate30e)
)

print(
    "\nEvidence tiers:"
)

print(
    pd.crosstab(
        candidate30e["target"],
        candidate30e["evidence_tier"]
    )
)

assert len(candidate30e) == 39

print(
    "\nSTEP30E CANDIDATE LOCK: PASS"
)

In [ ]:
# =============================================================================
# Step30E Cell 69
# Frozen malignant receiver object
# =============================================================================

RECEIVER_STATES = [
    "Malignant_MS6",
    "Malignant_MS5",
    "Malignant_MS3"
]

receiver_mask = (
    adata_primary.obs[
        "communication_group"
    ].isin(
        RECEIVER_STATES
    )
)

adata_receiver = (
    adata_primary[
        receiver_mask
    ]
    .copy()
)

print(
    "Receiver object:",
    adata_receiver.shape
)

print(
    pd.crosstab(
        adata_receiver.obs["communication_group"],
        adata_receiver.obs["sample"]
    )
)

print(
    "\nRECEIVER OBJECT: PASS"
)

In [ ]:
# =============================================================================
# Step30E Cell 70
# Receptor-complex expression audit
# =============================================================================

import scipy.sparse as sp

receptor_complexes = sorted(
    candidate30e[
        "receptor_complex"
    ].unique()
)

rows = []

for sample in PRIMARY_SAMPLES:

    for target in RECEIVER_STATES:

        mask = (
            (adata_receiver.obs["sample"] == sample)
            &
            (
                adata_receiver.obs[
                    "communication_group"
                ] == target
            )
        )

        sub = adata_receiver[
            mask
        ]

        for receptor in receptor_complexes:

            subunits = [
                g
                for g in receptor.split("_")
                if g in sub.var_names
            ]

            if len(subunits) == 0:
                continue

            mean_exprs = []
            pct_exprs = []

            for gene in subunits:

                x = sub[:, gene].X

                if sp.issparse(x):
                    arr = x.toarray().ravel()
                else:
                    arr = np.asarray(x).ravel()

                mean_exprs.append(
                    float(arr.mean())
                )

                pct_exprs.append(
                    float(
                        np.mean(arr > 0)
                    )
                )

            rows.append({
                "sample": sample,
                "target": target,
                "receptor_complex": receptor,
                "n_cells": sub.n_obs,
                "n_subunits": len(subunits),
                "all_subunits_present":
                    len(subunits)
                    ==
                    len(receptor.split("_")),

                "mean_subunit_expression":
                    float(
                        np.mean(mean_exprs)
                    ),

                "min_subunit_mean_expression":
                    float(
                        np.min(mean_exprs)
                    ),

                "min_subunit_pct_expressing":
                    float(
                        np.min(pct_exprs)
                    )
            })


receptor_stats = pd.DataFrame(
    rows
)

print(
    "Receptor-state rows:",
    len(receptor_stats)
)

display(
    receptor_stats.head()
)

In [ ]:
# =============================================================================
# Step30E Cell 71
# Receptor-state preference
# =============================================================================

receptor_pref = (
    receptor_stats
    .sort_values(
        "min_subunit_mean_expression",
        ascending=False
    )
    .groupby(
        [
            "sample",
            "receptor_complex"
        ],
        as_index=False
    )
    .first()
    [
        [
            "sample",
            "receptor_complex",
            "target",
            "min_subunit_mean_expression",
            "min_subunit_pct_expressing"
        ]
    ]
    .rename(
        columns={
            "target":
                "receptor_preferred_target"
        }
    )
)

display(
    receptor_pref.head(20)
)

In [ ]:
# =============================================================================
# Step30E Cell 72
# Does LIANA state skew agree with receptor expression?
# =============================================================================

pref09 = (
    receptor_pref.loc[
        receptor_pref["sample"] == "HCC09T",
        [
            "receptor_complex",
            "receptor_preferred_target"
        ]
    ]
    .rename(
        columns={
            "receptor_preferred_target":
                "receptor_pref_HCC09T"
        }
    )
)

pref10 = (
    receptor_pref.loc[
        receptor_pref["sample"] == "HCC10T",
        [
            "receptor_complex",
            "receptor_preferred_target"
        ]
    ]
    .rename(
        columns={
            "receptor_preferred_target":
                "receptor_pref_HCC10T"
        }
    )
)

candidate30e = (
    candidate30e
    .merge(
        pref09,
        on="receptor_complex",
        how="left"
    )
    .merge(
        pref10,
        on="receptor_complex",
        how="left"
    )
)

candidate30e[
    "receptor_match_HCC09T"
] = (
    candidate30e["target"]
    ==
    candidate30e["receptor_pref_HCC09T"]
)

candidate30e[
    "receptor_match_HCC10T"
] = (
    candidate30e["target"]
    ==
    candidate30e["receptor_pref_HCC10T"]
)

candidate30e[
    "receptor_match_both"
] = (
    candidate30e["receptor_match_HCC09T"]
    &
    candidate30e["receptor_match_HCC10T"]
)

print(
    "Receiver preference match in both patients:"
)

print(
    candidate30e[
        "receptor_match_both"
    ].value_counts()
)

print(
    "\nBy target:"
)

print(
    pd.crosstab(
        candidate30e["target"],
        candidate30e["receptor_match_both"]
    )
)

In [ ]:
# =============================================================================
# Step30E Cell 73
# Focused audit of the only robust MS3-skewed candidate
# =============================================================================

mmp9_ifnar1 = (
    candidate30e.loc[
        (
            candidate30e[
                "ligand_complex"
            ] == "MMP9"
        )
        &
        (
            candidate30e[
                "receptor_complex"
            ] == "IFNAR1"
        )
    ]
    .copy()
)

display(
    mmp9_ifnar1
)

display(
    receptor_stats.loc[
        receptor_stats[
            "receptor_complex"
        ] == "IFNAR1"
    ]
    .sort_values(
        [
            "sample",
            "min_subunit_mean_expression"
        ],
        ascending=[
            True,
            False
        ]
    )
)

In [ ]:
# =============================================================================
# Step30E Cell 74
# OmniPath ligand-receptor intercellular prior
# =============================================================================

import omnipath as op

intercell_lr = (
    op.interactions.import_intercell_network(
        include=[
            "omnipath",
            "pathwayextra",
            "ligrecextra"
        ],

        transmitter_params={
            "categories": [
                "ligand"
            ]
        },

        receiver_params={
            "categories": [
                "receptor"
            ]
        }
    )
)

print(
    "OmniPath LR network:",
    intercell_lr.shape
)

print(
    "\nColumns:"
)

for col in intercell_lr.columns:
    print(col)

In [ ]:
# =============================================================================
# Step30E Cell 75
# Identify OmniPath gene-symbol columns
# =============================================================================

symbol_cols = [
    c
    for c in intercell_lr.columns
    if "genesymbol" in c.lower()
]

print(
    "Gene-symbol columns:"
)

for c in symbol_cols:
    print(c)

In [ ]:
# =============================================================================
# Step30E Cell 76
# OmniPath support for simple LR candidates
# =============================================================================

source_col = next(
    c for c in intercell_lr.columns
    if c.lower()
    == "genesymbol_intercell_source"
)

target_col = next(
    c for c in intercell_lr.columns
    if c.lower()
    == "genesymbol_intercell_target"
)

candidate30e[
    "simple_lr"
] = (
    ~candidate30e[
        "ligand_complex"
    ].str.contains("_", regex=False)
    &
    ~candidate30e[
        "receptor_complex"
    ].str.contains("_", regex=False)
)

simple_candidates = (
    candidate30e.loc[
        candidate30e["simple_lr"]
    ]
    .copy()
)

op_pairs = set(
    zip(
        intercell_lr[source_col].astype(str),
        intercell_lr[target_col].astype(str)
    )
)

simple_candidates[
    "omnipath_generic_LR_support"
] = [
    (lig, rec) in op_pairs
    for lig, rec in zip(
        simple_candidates[
            "ligand_complex"
        ],
        simple_candidates[
            "receptor_complex"
        ]
    )
]

print(
    "Simple candidate LR:",
    len(simple_candidates)
)

print(
    "\nOmniPath generic ligand-receptor support:"
)

print(
    simple_candidates[
        "omnipath_generic_LR_support"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30E Cell 77
# Audit unconventional top pairs
# =============================================================================

AUDIT_PAIRS = [
    ("PTDSS1", "SCARB1"),
    ("PTPN6", "EGFR"),
    ("MMP9", "IFNAR1"),
    ("ADM", "RAMP1"),
    ("APOE", "SCARB1"),
    ("PLTP", "ABCA1")
]

audit_prior_rows = []

for ligand, receptor in AUDIT_PAIRS:

    match = intercell_lr.loc[
        (
            intercell_lr[source_col].astype(str)
            ==
            ligand
        )
        &
        (
            intercell_lr[target_col].astype(str)
            ==
            receptor
        )
    ].copy()

    audit_prior_rows.append({
        "ligand": ligand,
        "receptor": receptor,
        "omnipath_rows": len(match),
        "supported": len(match) > 0
    })


audit_prior = pd.DataFrame(
    audit_prior_rows
)

display(
    audit_prior
)

In [ ]:
# =============================================================================
# Step30E Cell 78
# Save receiver-side validation tables
# =============================================================================

candidate30e.to_csv(
    TABLE_DIR /
    "Step30E_MechanisticCandidates_ge3pct.csv",
    index=False
)

receptor_stats.to_csv(
    TABLE_DIR /
    "Step30E_ReceptorExpression_by_State.csv",
    index=False
)

simple_candidates.to_csv(
    TABLE_DIR /
    "Step30E_OmniPath_PriorAudit_SimpleLR.csv",
    index=False
)

audit_prior.to_csv(
    TABLE_DIR /
    "Step30E_UnconventionalPair_Audit.csv",
    index=False
)

print(
    "STEP30E TABLES SAVED"
)

In [ ]:
# =============================================================================
# Step30E Cell 79
# REPORT — receiver validation and prior audit
# =============================================================================

print("=" * 100)

print(
    "STEP30E RECEIVER-SIDE VALIDATION / INTERCELL PRIOR REPORT"
)

print("=" * 100)


print(
    "\nMechanistic candidates >=3%:",
    len(candidate30e)
)

print(
    "\nEvidence tier x target:"
)

display(
    pd.crosstab(
        candidate30e["target"],
        candidate30e["evidence_tier"]
    )
)


print(
    "\nReceiver expression preference agrees in BOTH patients:"
)

print(
    candidate30e[
        "receptor_match_both"
    ].value_counts()
)


print(
    "\nAgreement by malignant target:"
)

display(
    pd.crosstab(
        candidate30e["target"],
        candidate30e["receptor_match_both"]
    )
)


print(
    "\nOnly robust MS3 candidate:"
)

display(
    candidate30e.loc[
        (
            candidate30e["target"]
            ==
            "Malignant_MS3"
        )
        &
        (
            candidate30e["evidence_tier"]
            ==
            "Primary_Robust_ge5pct"
        )
    ]
)


print(
    "\nIFNAR1 receptor-state expression:"
)

display(
    receptor_stats.loc[
        receptor_stats[
            "receptor_complex"
        ] == "IFNAR1"
    ]
)


print(
    "\nOmniPath generic LR support among simple candidates:"
)

print(
    simple_candidates[
        "omnipath_generic_LR_support"
    ].value_counts()
)


print(
    "\nUnconventional-pair audit:"
)

display(
    audit_prior
)


print(
    "\nInterpretation:"
)

print(
    "Primary mechanistic candidates require cross-patient FDR support, "
    "state-skew effect size, receiver-side receptor support, and "
    "subsequent intracellular-network compatibility."
)

print(
    "\nSTEP30E COMPLETE"
)

In [ ]:
# =============================================================================
# Step30E.1 Cell 80
# Multi-resource LR provenance audit
# =============================================================================

available_resources = li.rs.show_resources()

print(
    "Available LIANA resources:",
    len(available_resources)
)

print(
    available_resources
)

In [ ]:
# =============================================================================
# Step30E.1 Cell 81
# Count supporting LIANA resources for candidate LR pairs
# =============================================================================

candidate_pairs = (
    candidate30e[
        [
            "ligand_complex",
            "receptor_complex"
        ]
    ]
    .drop_duplicates()
    .copy()
)

support_rows = []

for resource_name in available_resources:

    try:
        res = li.rs.select_resource(
            resource_name
        )

        if not {
            "ligand",
            "receptor"
        }.issubset(res.columns):
            continue

        pair_set = set(
            zip(
                res["ligand"].astype(str),
                res["receptor"].astype(str)
            )
        )

        for ligand, receptor in zip(
            candidate_pairs["ligand_complex"],
            candidate_pairs["receptor_complex"]
        ):

            if (
                ligand,
                receptor
            ) in pair_set:

                support_rows.append({
                    "ligand_complex":
                        ligand,

                    "receptor_complex":
                        receptor,

                    "resource":
                        resource_name
                })

    except Exception as e:

        print(
            "Skipped:",
            resource_name,
            "|",
            type(e).__name__
        )


resource_support = pd.DataFrame(
    support_rows
)

print(
    "Support rows:",
    len(resource_support)
)

In [ ]:
# =============================================================================
# Step30E.1 Cell 82
# Summarize LR resource provenance
# =============================================================================

if len(resource_support) > 0:

    support_summary = (
        resource_support
        .groupby(
            [
                "ligand_complex",
                "receptor_complex"
            ]
        )
        .agg(
            n_supporting_resources=(
                "resource",
                "nunique"
            ),

            supporting_resources=(
                "resource",
                lambda x:
                    " | ".join(
                        sorted(
                            set(x)
                        )
                    )
            )
        )
        .reset_index()
    )

else:

    support_summary = pd.DataFrame(
        columns=[
            "ligand_complex",
            "receptor_complex",
            "n_supporting_resources",
            "supporting_resources"
        ]
    )


candidate30e_prov = (
    candidate30e.merge(
        support_summary,
        on=[
            "ligand_complex",
            "receptor_complex"
        ],
        how="left"
    )
)

candidate30e_prov[
    "n_supporting_resources"
] = (
    candidate30e_prov[
        "n_supporting_resources"
    ]
    .fillna(0)
    .astype(int)
)

display(
    candidate30e_prov[
        [
            "source",
            "ligand_complex",
            "receptor_complex",
            "target",
            "evidence_tier",
            "n_supporting_resources",
            "supporting_resources"
        ]
    ]
    .sort_values(
        [
            "evidence_tier",
            "n_supporting_resources"
        ],
        ascending=[
            True,
            False
        ]
    )
)

In [ ]:
# =============================================================================
# Step30E.1 Cell 83
# Key candidate provenance
# =============================================================================

KEY_PAIRS = [
    ("MMP9", "IFNAR1"),
    ("PTDSS1", "SCARB1"),
    ("PTPN6", "EGFR"),
    ("ADM", "RAMP1"),
    ("APOE", "SCARB1"),
    ("PLTP", "ABCA1")
]

key_provenance = (
    candidate30e_prov.loc[
        candidate30e_prov.apply(
            lambda r:
                (
                    r["ligand_complex"],
                    r["receptor_complex"]
                )
                in KEY_PAIRS,
            axis=1
        ),
        [
            "source",
            "ligand_complex",
            "receptor_complex",
            "target",
            "evidence_tier",
            "n_supporting_resources",
            "supporting_resources"
        ]
    ]
)

display(
    key_provenance
)

In [ ]:
# =============================================================================
# Step30E.1 Cell 84
# REPORT — LR provenance
# =============================================================================

print("=" * 100)

print(
    "STEP30E.1 MULTI-RESOURCE LR PROVENANCE REPORT"
)

print("=" * 100)

print(
    "\nCandidate LR rows:",
    len(candidate30e_prov)
)

print(
    "\nNumber of supporting LIANA resources:"
)

print(
    candidate30e_prov[
        "n_supporting_resources"
    ].value_counts()
    .sort_index()
)

print(
    "\nKey LR provenance:"
)

display(
    key_provenance
)

print(
    "\nSTEP30E.1 COMPLETE"
)

## Step30F — Receptor-to-regulator causal-network preparation

### Objective

Connect state-skewed extracellular communication with the frozen Step29
malignant regulatory architecture.

Conceptual framework:

TME ligand
→ malignant receptor availability
→ signed intracellular signaling compatibility
→ Step29 TF/regulon activity
→ malignant-state biology

### Important distinction

Receptor expression and ligand–receptor inference do not directly measure
receptor activation.

Therefore, receptor scores in Step30F represent extracellular-input /
receptor-availability evidence, not biochemical receptor activity.

The causal network is interpreted as a prior-knowledge-compatible mechanistic
hypothesis, not direct proof of signaling causality.

In [ ]:
# =============================================================================
# Step30F Cell 86
# Frozen Step29 regulator sets
# =============================================================================

REGULATOR_SETS = {
    "Malignant_MS6": [
        "HNF4A",
        "NR1H4",
        "NR1I3",
        "CEBPA",
        "FOXA3"
    ],

    "Malignant_MS5": [
        "KLF4",
        "FOS",
        "FOSB",
        "ELF3",
        "SOX4"
    ],

    "Malignant_MS3": [
        "FOXM1",
        "MYBL2",
        "E2F8",
        "TFDP1",
        "E2F1",
        "BRCA1"
    ]
}

for state, regs in REGULATOR_SETS.items():
    print(
        state,
        "->",
        ", ".join(regs)
    )

In [ ]:
# =============================================================================
# Step30F Cell 87
# Locate frozen Step29 regulon activity matrix
# =============================================================================

STEP29_AUCELL = (
    ROOT /
    "LIHC_scRNA_Step29_GRN" /
    "Tables" /
    "Step29G_AUCell_RegulonActivity.csv"
)

print(
    "Step29 AUCell:"
)

print(
    STEP29_AUCELL
)

print(
    "Exists:",
    STEP29_AUCELL.exists()
)

assert STEP29_AUCELL.exists()

print(
    "\nSTEP29 AUCELL INPUT FOUND: PASS"
)

In [ ]:
# =============================================================================
# Step30F Cell 88
# AUCell file structure audit
# =============================================================================

aucell_raw = pd.read_csv(
    STEP29_AUCELL
)

print(
    "Shape:",
    aucell_raw.shape
)

print(
    "\nColumns:"
)

print(
    aucell_raw.columns.tolist()
)

print(
    "\nFirst rows:"
)

display(
    aucell_raw.head()
)

In [ ]:
# =============================================================================
# Step30F Cell 89
# Identify AUCell cell-ID column
# =============================================================================

frozen_cell_ids = set(
    adata_malignant.obs_names.astype(str)
)

id_audit = []

for col in aucell_raw.columns:

    values = set(
        aucell_raw[
            col
        ]
        .astype(str)
    )

    overlap = len(
        values &
        frozen_cell_ids
    )

    id_audit.append({
        "column":
            col,

        "overlap_with_12057":
            overlap
    })


id_audit = (
    pd.DataFrame(id_audit)
    .sort_values(
        "overlap_with_12057",
        ascending=False
    )
)

display(
    id_audit.head(10)
)

CELL_ID_COL = (
    id_audit.iloc[0][
        "column"
    ]
)

CELL_ID_OVERLAP = int(
    id_audit.iloc[0][
        "overlap_with_12057"
    ]
)

print(
    "\nDetected cell-ID column:",
    CELL_ID_COL
)

print(
    "Overlap:",
    CELL_ID_OVERLAP
)

assert CELL_ID_OVERLAP == 12057

In [ ]:
# =============================================================================
# Step30F Cell 90
# Frozen regulator availability
# =============================================================================

FROZEN_REGULATORS = sorted(
    {
        tf
        for regs in REGULATOR_SETS.values()
        for tf in regs
    }
)

regulator_column_map = {}

for tf in FROZEN_REGULATORS:

    matches = [
        c
        for c in aucell_raw.columns
        if (
            str(c) == tf
            or
            str(c).startswith(tf + "(")
            or
            str(c).startswith(tf + "_")
        )
    ]

    regulator_column_map[
        tf
    ] = matches


for tf, matches in regulator_column_map.items():

    print(
        tf,
        "->",
        matches
    )

In [ ]:
# =============================================================================
# Step30F Cell 91
# REPORT — causal-network input preparation
# =============================================================================

print("=" * 100)

print(
    "STEP30F CAUSAL-NETWORK INPUT PREPARATION REPORT"
)

print("=" * 100)


print(
    "\nAUCell shape:",
    aucell_raw.shape
)

print(
    "Cell-ID column:",
    CELL_ID_COL
)

print(
    "Cell-ID overlap:",
    CELL_ID_OVERLAP
)


print(
    "\nFrozen Step29 regulator availability:"
)

for state, regs in REGULATOR_SETS.items():

    print(
        f"\n{state}:"
    )

    for tf in regs:

        print(
            tf,
            "->",
            regulator_column_map[
                tf
            ]
        )


print(
    "\nNo causal network has been fitted yet."
)

print(
    "\nSTEP30F INPUT AUDIT COMPLETE"
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 92
# Freeze Step30E provenance and build AUCell master table
# =============================================================================

PROVENANCE_PATH = (
    TABLE_DIR /
    "Step30E1_MultiResource_Provenance.csv"
)

candidate30e_prov.to_csv(
    PROVENANCE_PATH,
    index=False
)

auc = (
    aucell_raw
    .set_index(CELL_ID_COL)
    .copy()
)

assert auc.index.is_unique

mal_meta = (
    adata_malignant.obs[
        [
            "patient",
            "final_malignant_state_id"
        ]
    ]
    .copy()
)

auc = auc.join(
    mal_meta,
    how="left"
)

assert auc["patient"].notna().all()

print(
    "AUCell master:",
    auc.shape
)

print(
    "Matched cells:",
    auc["patient"].notna().sum()
)

print(
    "\nSTEP30F AUCELL MASTER: PASS"
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 93
# Lock frozen Step29 regulon columns
# =============================================================================

TF_COL = {}

for tf in FROZEN_REGULATORS:

    matches = regulator_column_map[tf]

    assert len(matches) == 1, (
        tf,
        matches
    )

    TF_COL[tf] = matches[0]


print(
    "Frozen regulators:",
    len(TF_COL)
)

for tf, col in TF_COL.items():

    print(
        tf,
        "->",
        col
    )

assert len(TF_COL) == 16

print(
    "\nFROZEN REGULON COLUMN LOCK: PASS"
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 94
# Cross-patient malignant-axis AUCell object
# =============================================================================

AXIS_PATIENTS = [
    "HCC09",
    "HCC10"
]

AXIS_STATES = [
    "MS6",
    "MS5",
    "MS3"
]

auc_axis = (
    auc.loc[
        auc["patient"].isin(
            AXIS_PATIENTS
        )
        &
        auc[
            "final_malignant_state_id"
        ].isin(
            AXIS_STATES
        )
    ]
    .copy()
)

print(
    "Axis AUCell cells:",
    len(auc_axis)
)

print(
    "\nPatient × state:"
)

display(
    pd.crosstab(
        auc_axis[
            "final_malignant_state_id"
        ],
        auc_axis[
            "patient"
        ]
    )
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 95
# Patient-specific regulon state contrasts
# =============================================================================

tf_rows = []

for patient in AXIS_PATIENTS:

    sub = (
        auc_axis.loc[
            auc_axis[
                "patient"
            ] == patient
        ]
        .copy()
    )

    for tf, col in TF_COL.items():

        x = (
            pd.to_numeric(
                sub[col],
                errors="coerce"
            )
            .astype(float)
        )

        mu = float(
            x.mean()
        )

        sd = float(
            x.std(
                ddof=0
            )
        )

        if (
            not np.isfinite(sd)
            or
            sd == 0
        ):
            continue

        z = (
            x - mu
        ) / sd

        tmp = pd.DataFrame({
            "state":
                sub[
                    "final_malignant_state_id"
                ].values,

            "z":
                z.values
        })

        state_means = (
            tmp
            .groupby("state")["z"]
            .mean()
        )

        if not all(
            s in state_means.index
            for s in AXIS_STATES
        ):
            continue

        for target_state in AXIS_STATES:

            others = [
                s
                for s in AXIS_STATES
                if s != target_state
            ]

            contrast = (
                float(
                    state_means[
                        target_state
                    ]
                )
                -
                float(
                    np.mean([
                        state_means[s]
                        for s in others
                    ])
                )
            )

            tf_rows.append({
                "patient":
                    patient,

                "target_state":
                    target_state,

                "TF":
                    tf,

                "target_mean_z":
                    float(
                        state_means[
                            target_state
                        ]
                    ),

                "other_states_mean_z":
                    float(
                        np.mean([
                            state_means[s]
                            for s in others
                        ])
                    ),

                "contrast":
                    contrast
            })


tf_contrast_long = pd.DataFrame(
    tf_rows
)

print(
    "TF contrast rows:",
    len(tf_contrast_long)
)

display(
    tf_contrast_long.head()
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 96
# Cross-patient frozen-regulator output scores
# =============================================================================

tf_state_map = {
    tf:
    target.replace(
        "Malignant_",
        ""
    )

    for target, regs
    in REGULATOR_SETS.items()

    for tf in regs
}


tf_expected = (
    tf_contrast_long.loc[
        tf_contrast_long.apply(
            lambda r:
                tf_state_map[
                    r["TF"]
                ]
                ==
                r[
                    "target_state"
                ],
            axis=1
        )
    ]
    .copy()
)


tf_wide = (
    tf_expected
    .pivot(
        index=[
            "target_state",
            "TF"
        ],
        columns="patient",
        values="contrast"
    )
    .reset_index()
)


tf_wide[
    "positive_HCC09"
] = (
    tf_wide["HCC09"] > 0
)

tf_wide[
    "positive_HCC10"
] = (
    tf_wide["HCC10"] > 0
)

tf_wide[
    "positive_both"
] = (
    tf_wide[
        "positive_HCC09"
    ]
    &
    tf_wide[
        "positive_HCC10"
    ]
)

tf_wide[
    "mean_contrast"
] = (
    tf_wide[
        ["HCC09", "HCC10"]
    ]
    .mean(axis=1)
)

tf_wide[
    "min_contrast"
] = (
    tf_wide[
        ["HCC09", "HCC10"]
    ]
    .min(axis=1)
)


print(
    "Frozen regulators:"
)

display(
    tf_wide
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 97
# Normalize cross-patient TF output scores
# =============================================================================

tf_output_scores = (
    tf_wide.loc[
        tf_wide[
            "positive_both"
        ]
    ]
    .copy()
)


tf_output_scores[
    "output_score"
] = np.nan


for state in AXIS_STATES:

    mask = (
        tf_output_scores[
            "target_state"
        ] == state
    )

    if mask.sum() == 0:
        continue

    denom = float(
        tf_output_scores.loc[
            mask,
            "mean_contrast"
        ]
        .abs()
        .max()
    )

    if denom > 0:

        tf_output_scores.loc[
            mask,
            "output_score"
        ] = (
            tf_output_scores.loc[
                mask,
                "mean_contrast"
            ]
            /
            denom
        )


print(
    "Cross-patient-positive TF outputs:",
    len(tf_output_scores)
)

display(
    tf_output_scores
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 98
# Expand LR receptor complexes into receptor gene nodes
# =============================================================================

rec_rows = []

for _, r in (
    candidate30e_prov
    .iterrows()
):

    target_state = (
        str(
            r["target"]
        )
        .replace(
            "Malignant_",
            ""
        )
    )

    receptor_genes = (
        str(
            r[
                "receptor_complex"
            ]
        )
        .split("_")
    )

    for receptor_gene in receptor_genes:

        rec_rows.append({
            "target_state":
                target_state,

            "source":
                r["source"],

            "ligand_complex":
                r[
                    "ligand_complex"
                ],

            "receptor_complex":
                r[
                    "receptor_complex"
                ],

            "receptor_gene":
                receptor_gene,

            "evidence_tier":
                r[
                    "evidence_tier"
                ],

            "n_supporting_resources":
                r[
                    "n_supporting_resources"
                ]
        })


receptor_map = pd.DataFrame(
    rec_rows
)


receptor_map[
    "gene_present"
] = (
    receptor_map[
        "receptor_gene"
    ].isin(
        adata_receiver.var_names
    )
)


print(
    "Expanded receptor rows:",
    len(receptor_map)
)

print(
    "Unique receptor genes:",
    receptor_map[
        "receptor_gene"
    ].nunique()
)

print(
    "All receptor genes present:",
    receptor_map[
        "gene_present"
    ].all()
)

display(
    receptor_map.head(20)
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 99
# Patient-specific receptor-gene state contrasts
# =============================================================================

receptor_genes = sorted(
    receptor_map.loc[
        receptor_map[
            "gene_present"
        ],
        "receptor_gene"
    ].unique()
)


receiver_axis = (
    adata_receiver[
        adata_receiver.obs[
            "sample"
        ].isin(
            [
                "HCC09T",
                "HCC10T"
            ]
        )
    ]
    .copy()
)


sample_to_patient = {
    "HCC09T": "HCC09",
    "HCC10T": "HCC10"
}


rec_contrast_rows = []


for sample, patient in (
    sample_to_patient.items()
):

    sub = (
        receiver_axis[
            receiver_axis.obs[
                "sample"
            ] == sample
        ]
    )

    states = (
        sub.obs[
            "final_malignant_state_id"
        ]
        .astype(str)
    )


    for gene in receptor_genes:

        x = sub[:, gene].X

        if sp.issparse(x):
            values = (
                x.toarray()
                .ravel()
                .astype(float)
            )
        else:
            values = (
                np.asarray(x)
                .ravel()
                .astype(float)
            )

        sd = float(
            np.std(
                values,
                ddof=0
            )
        )

        if (
            not np.isfinite(sd)
            or
            sd == 0
        ):
            continue

        z = (
            values
            -
            float(
                np.mean(values)
            )
        ) / sd


        tmp = pd.DataFrame({
            "state":
                states.values,

            "z":
                z
        })


        state_means = (
            tmp
            .groupby("state")["z"]
            .mean()
        )


        if not all(
            s in state_means.index
            for s in AXIS_STATES
        ):
            continue


        for target_state in AXIS_STATES:

            others = [
                s
                for s in AXIS_STATES
                if s != target_state
            ]

            contrast = (
                float(
                    state_means[
                        target_state
                    ]
                )
                -
                float(
                    np.mean([
                        state_means[s]
                        for s in others
                    ])
                )
            )

            rec_contrast_rows.append({
                "patient":
                    patient,

                "target_state":
                    target_state,

                "receptor_gene":
                    gene,

                "contrast":
                    contrast
            })


rec_contrast_long = pd.DataFrame(
    rec_contrast_rows
)

print(
    "Receptor contrast rows:",
    len(rec_contrast_long)
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 100
# Cross-patient receptor input scores
# =============================================================================

candidate_receptor_nodes = (
    receptor_map[
        [
            "target_state",
            "receptor_gene"
        ]
    ]
    .drop_duplicates()
)


rec_expected = (
    candidate_receptor_nodes
    .merge(
        rec_contrast_long,
        on=[
            "target_state",
            "receptor_gene"
        ],
        how="left"
    )
)


rec_wide = (
    rec_expected
    .pivot(
        index=[
            "target_state",
            "receptor_gene"
        ],
        columns="patient",
        values="contrast"
    )
    .reset_index()
)


rec_wide[
    "positive_both"
] = (
    (rec_wide["HCC09"] > 0)
    &
    (rec_wide["HCC10"] > 0)
)


rec_wide[
    "mean_contrast"
] = (
    rec_wide[
        ["HCC09", "HCC10"]
    ]
    .mean(axis=1)
)


rec_wide[
    "min_contrast"
] = (
    rec_wide[
        ["HCC09", "HCC10"]
    ]
    .min(axis=1)
)


print(
    "Candidate receptor nodes:",
    len(rec_wide)
)

print(
    "Positive in both patients:",
    int(
        rec_wide[
            "positive_both"
        ].sum()
    )
)

display(
    rec_wide
    .sort_values(
        [
            "target_state",
            "mean_contrast"
        ],
        ascending=[
            True,
            False
        ]
    )
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 101
# Normalize receptor input scores
# =============================================================================

receptor_input_scores = (
    rec_wide.loc[
        rec_wide[
            "positive_both"
        ]
    ]
    .copy()
)


receptor_input_scores[
    "input_score"
] = np.nan


for state in AXIS_STATES:

    mask = (
        receptor_input_scores[
            "target_state"
        ] == state
    )

    if mask.sum() == 0:
        continue

    denom = float(
        receptor_input_scores.loc[
            mask,
            "mean_contrast"
        ]
        .abs()
        .max()
    )

    if denom > 0:

        receptor_input_scores.loc[
            mask,
            "input_score"
        ] = (
            receptor_input_scores.loc[
                mask,
                "mean_contrast"
            ]
            /
            denom
        )


display(
    receptor_input_scores
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 102
# Attach upstream LR evidence to receptor nodes
# =============================================================================

receptor_parent_summary = (
    receptor_map
    .groupby(
        [
            "target_state",
            "receptor_gene"
        ]
    )
    .agg(
        n_parent_lr=(
            "ligand_complex",
            "size"
        ),

        n_unique_ligands=(
            "ligand_complex",
            "nunique"
        ),

        max_resource_support=(
            "n_supporting_resources",
            "max"
        ),

        parent_lr_pairs=(
            "ligand_complex",
            lambda x:
                " | ".join(
                    sorted(
                        set(
                            x.astype(str)
                        )
                    )
                )
        ),

        evidence_tiers=(
            "evidence_tier",
            lambda x:
                " | ".join(
                    sorted(
                        set(
                            x.astype(str)
                        )
                    )
                )
        )
    )
    .reset_index()
)


receptor_input_scores = (
    receptor_input_scores
    .merge(
        receptor_parent_summary,
        on=[
            "target_state",
            "receptor_gene"
        ],
        how="left"
    )
)


display(
    receptor_input_scores
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 103
# Save causal-network input/output scores
# =============================================================================

TF_SCORE_PATH = (
    TABLE_DIR /
    "Step30F_TF_OutputScores.csv"
)

RECEPTOR_SCORE_PATH = (
    TABLE_DIR /
    "Step30F_Receptor_InputScores.csv"
)


tf_output_scores.to_csv(
    TF_SCORE_PATH,
    index=False
)

receptor_input_scores.to_csv(
    RECEPTOR_SCORE_PATH,
    index=False
)


print(
    "Saved:"
)

print(
    TF_SCORE_PATH
)

print(
    RECEPTOR_SCORE_PATH
)

In [ ]:
# =============================================================================
# Step30F.1 Cell 104
# REPORT — causal-network score lock
# =============================================================================

print("=" * 100)

print(
    "STEP30F.1 CAUSAL-NETWORK INPUT / OUTPUT SCORE REPORT"
)

print("=" * 100)


print(
    "\nFrozen Step29 regulators:"
)

print(
    len(tf_wide)
)


print(
    "Regulators enriched in their expected state in BOTH patients:"
)

print(
    int(
        tf_wide[
            "positive_both"
        ].sum()
    )
)


print(
    "\nBy target state:"
)

display(
    pd.crosstab(
        tf_wide[
            "target_state"
        ],
        tf_wide[
            "positive_both"
        ]
    )
)


print(
    "\nTF output scores:"
)

display(
    tf_output_scores[
        [
            "target_state",
            "TF",
            "HCC09",
            "HCC10",
            "mean_contrast",
            "output_score"
        ]
    ]
)


print(
    "\nReceptor nodes before cross-patient direction filter:",
    len(rec_wide)
)

print(
    "Receptor nodes enriched in expected state in BOTH patients:",
    len(receptor_input_scores)
)


print(
    "\nReceptor inputs by state:"
)

print(
    receptor_input_scores[
        "target_state"
    ].value_counts()
)


print(
    "\nTop receptor inputs:"
)

display(
    receptor_input_scores[
        [
            "target_state",
            "receptor_gene",
            "HCC09",
            "HCC10",
            "mean_contrast",
            "input_score",
            "parent_lr_pairs",
            "evidence_tiers",
            "max_resource_support"
        ]
    ]
    .sort_values(
        [
            "target_state",
            "input_score"
        ],
        ascending=[
            True,
            False
        ]
    )
)


print(
    "\nSTEP30F.1 SCORE LOCK COMPLETE"
)

print(
    "No causal network has yet been fitted."
)

## Step30F.2 — Signed intracellular prior and reachability audit

### Objective

Determine whether the frozen state-enriched receptor inputs can reach the
frozen Step29 regulator outputs through an independent signed intracellular
prior-knowledge network.

### Primary prior

Literature-supported OmniPath interactions.

Only interactions with:

- supported direction
- unambiguous activating or inhibitory sign

will be retained in the primary signed network.

### Interpretation

Network reachability and causal-network compatibility indicate that a
known signaling route can connect the observed receptor and regulator
states. They do not establish receptor activation or causal signaling
in these tumors.

In [ ]:
# =============================================================================
# Step30F.2 Cell 106
# Retrieve literature-supported OmniPath interactions
# =============================================================================

import omnipath as op

omni_raw = (
    op.interactions.OmniPath.get()
)

print(
    "OmniPath raw interactions:",
    omni_raw.shape
)

print(
    "\nColumns:"
)

for c in omni_raw.columns:
    print(c)

In [ ]:
# =============================================================================
# Step30F.2 Cell 106
# Retrieve literature-supported OmniPath interactions WITH gene symbols
# =============================================================================

import omnipath as op

omni_raw = (
    op.interactions.OmniPath.get(
        genesymbols=True
    )
)

print(
    "OmniPath raw interactions:",
    omni_raw.shape
)

print(
    "\nColumns:"
)

for c in omni_raw.columns:
    print(c)

In [ ]:
# =============================================================================
# Step30F.2 Cell 107
# Audit OmniPath signed-interaction fields
# =============================================================================

required_logic = [
    "is_directed",
    "is_stimulation",
    "is_inhibition",
    "consensus_direction",
    "consensus_stimulation",
    "consensus_inhibition"
]

for c in required_logic:
    print(
        c,
        "->",
        c in omni_raw.columns
    )

source_candidates = [
    c
    for c in omni_raw.columns
    if c.lower() in [
        "source_genesymbol",
        "genesymbol_source"
    ]
]

target_candidates = [
    c
    for c in omni_raw.columns
    if c.lower() in [
        "target_genesymbol",
        "genesymbol_target"
    ]
]

print(
    "\nSource symbol candidates:",
    source_candidates
)

print(
    "Target symbol candidates:",
    target_candidates
)

assert len(source_candidates) >= 1
assert len(target_candidates) >= 1

OMNI_SOURCE = source_candidates[0]
OMNI_TARGET = target_candidates[0]

print(
    "\nUsing:"
)

print(
    OMNI_SOURCE,
    "->",
    OMNI_TARGET
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 108
# Build conservative signed directed OmniPath PPI
# =============================================================================

signed = (
    omni_raw.loc[
        omni_raw[
            "consensus_direction"
        ].fillna(False)
    ]
    .copy()
)

stim = (
    signed[
        "consensus_stimulation"
    ].fillna(False)
    .astype(bool)
)

inhib = (
    signed[
        "consensus_inhibition"
    ].fillna(False)
    .astype(bool)
)

signed = (
    signed.loc[
        stim ^ inhib
    ]
    .copy()
)

signed[
    "mor"
] = np.where(
    signed[
        "consensus_stimulation"
    ].astype(bool),
    1,
    -1
)

signed_ppi = (
    signed[
        [
            OMNI_SOURCE,
            "mor",
            OMNI_TARGET
        ]
    ]
    .rename(
        columns={
            OMNI_SOURCE:
                "source",
            OMNI_TARGET:
                "target"
        }
    )
    .dropna()
    .drop_duplicates()
)

signed_ppi[
    "source"
] = signed_ppi[
    "source"
].astype(str)

signed_ppi[
    "target"
] = signed_ppi[
    "target"
].astype(str)

signed_ppi = (
    signed_ppi.loc[
        (signed_ppi["source"] != "")
        &
        (signed_ppi["target"] != "")
        &
        (signed_ppi["source"] != signed_ppi["target"])
    ]
    .reset_index(drop=True)
)

print(
    "Primary signed PPI edges:",
    len(signed_ppi)
)

print(
    "\nSign distribution:"
)

print(
    signed_ppi[
        "mor"
    ].value_counts()
)

print(
    "\nUnique source nodes:",
    signed_ppi["source"].nunique()
)

print(
    "Unique target nodes:",
    signed_ppi["target"].nunique()
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 109
# Signed-edge conflict audit
# =============================================================================

edge_sign_n = (
    signed_ppi
    .groupby(
        [
            "source",
            "target"
        ]
    )[
        "mor"
    ]
    .nunique()
)

conflict_pairs = (
    edge_sign_n[
        edge_sign_n > 1
    ]
)

print(
    "Source-target pairs with conflicting signs:",
    len(conflict_pairs)
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 110
# Save primary signed PPI
# =============================================================================

SIGNED_PPI_PATH = (
    RESOURCE_DIR /
    "Step30F_OmniPath_Primary_SignedPPI.csv"
)

signed_ppi.to_csv(
    SIGNED_PPI_PATH,
    index=False
)

print(
    "Saved:",
    SIGNED_PPI_PATH
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 111
# Directed reachability graph
# =============================================================================

import networkx as nx

G = nx.from_pandas_edgelist(
    signed_ppi,
    source="source",
    target="target",
    edge_attr="mor",
    create_using=nx.DiGraph
)

print(
    "Graph nodes:",
    G.number_of_nodes()
)

print(
    "Graph edges:",
    G.number_of_edges()
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 112
# Input/output coverage in signed prior
# =============================================================================

coverage_rows = []

for state in AXIS_STATES:

    inputs = (
        receptor_input_scores.loc[
            receptor_input_scores[
                "target_state"
            ] == state,
            "receptor_gene"
        ]
        .tolist()
    )

    outputs = (
        tf_output_scores.loc[
            tf_output_scores[
                "target_state"
            ] == state,
            "TF"
        ]
        .tolist()
    )

    input_present = [
        x
        for x in inputs
        if x in G
    ]

    output_present = [
        x
        for x in outputs
        if x in G
    ]

    coverage_rows.append({
        "state":
            state,

        "input_total":
            len(inputs),

        "input_in_prior":
            len(input_present),

        "output_total":
            len(outputs),

        "output_in_prior":
            len(output_present),

        "inputs_in_prior":
            " | ".join(input_present),

        "outputs_in_prior":
            " | ".join(output_present)
    })


prior_coverage = pd.DataFrame(
    coverage_rows
)

display(
    prior_coverage
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 113
# Receptor-to-regulator directed reachability
# =============================================================================

path_rows = []

for state in AXIS_STATES:

    inputs = (
        receptor_input_scores.loc[
            receptor_input_scores[
                "target_state"
            ] == state,
            "receptor_gene"
        ]
        .tolist()
    )

    outputs = (
        tf_output_scores.loc[
            tf_output_scores[
                "target_state"
            ] == state,
            "TF"
        ]
        .tolist()
    )

    for receptor in inputs:

        for tf in outputs:

            if (
                receptor not in G
                or
                tf not in G
            ):

                reachable = False
                shortest_n_edges = np.nan

            else:

                reachable = nx.has_path(
                    G,
                    receptor,
                    tf
                )

                if reachable:

                    shortest_n_edges = (
                        nx.shortest_path_length(
                            G,
                            receptor,
                            tf
                        )
                    )

                else:

                    shortest_n_edges = np.nan

            path_rows.append({
                "state":
                    state,

                "receptor":
                    receptor,

                "TF":
                    tf,

                "reachable":
                    reachable,

                "shortest_n_edges":
                    shortest_n_edges
            })


reachability = pd.DataFrame(
    path_rows
)

print(
    "Receptor-TF pairs:",
    len(reachability)
)

print(
    "Reachable:",
    int(
        reachability[
            "reachable"
        ].sum()
    )
)

display(
    pd.crosstab(
        reachability[
            "state"
        ],
        reachability[
            "reachable"
        ]
    )
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 114
# State-level reachability summary
# =============================================================================

state_reachability = (
    reachability
    .groupby("state")
    .agg(
        n_pairs=(
            "reachable",
            "size"
        ),

        n_reachable=(
            "reachable",
            "sum"
        ),

        median_shortest_path=(
            "shortest_n_edges",
            "median"
        )
    )
    .reset_index()
)

state_reachability[
    "reachable_fraction"
] = (
    state_reachability[
        "n_reachable"
    ]
    /
    state_reachability[
        "n_pairs"
    ]
)

display(
    state_reachability
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 115
# Focused MS3 IFNAR1-to-regulator reachability
# =============================================================================

ms3_reachability = (
    reachability.loc[
        reachability[
            "state"
        ] == "MS3"
    ]
    .sort_values(
        [
            "reachable",
            "shortest_n_edges"
        ],
        ascending=[
            False,
            True
        ]
    )
)

display(
    ms3_reachability
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 116
# Save signed-prior reachability audit
# =============================================================================

prior_coverage.to_csv(
    TABLE_DIR /
    "Step30F2_SignedPrior_NodeCoverage.csv",
    index=False
)

reachability.to_csv(
    TABLE_DIR /
    "Step30F2_Receptor_TF_Reachability.csv",
    index=False
)

state_reachability.to_csv(
    TABLE_DIR /
    "Step30F2_State_Reachability_Summary.csv",
    index=False
)

print(
    "STEP30F.2 AUDIT TABLES SAVED"
)

In [ ]:
# =============================================================================
# Step30F.2 Cell 117
# REPORT — signed prior reachability
# =============================================================================

print("=" * 100)

print(
    "STEP30F.2 SIGNED INTRACELLULAR PRIOR / REACHABILITY REPORT"
)

print("=" * 100)


print(
    "\nPrimary signed PPI edges:",
    len(signed_ppi)
)

print(
    "Graph nodes:",
    G.number_of_nodes()
)

print(
    "Graph edges:",
    G.number_of_edges()
)


print(
    "\nInput/output prior coverage:"
)

display(
    prior_coverage
)


print(
    "\nState-level receptor-to-TF reachability:"
)

display(
    state_reachability
)


print(
    "\nMS3 IFNAR1 reachability:"
)

display(
    ms3_reachability
)


print(
    "\nSTEP30F.2 COMPLETE"
)

print(
    "No causal-network optimization has yet been performed."
)

In [ ]:
# =============================================================================
# Step30F.2a Cell 118
# Reachability using only prior-covered input/output pairs
# =============================================================================

covered_rows = []

for state in AXIS_STATES:

    inputs = (
        receptor_input_scores.loc[
            receptor_input_scores["target_state"] == state,
            "receptor_gene"
        ]
        .tolist()
    )

    outputs = (
        tf_output_scores.loc[
            tf_output_scores["target_state"] == state,
            "TF"
        ]
        .tolist()
    )

    inputs_in = [
        x for x in inputs
        if x in G
    ]

    outputs_in = [
        x for x in outputs
        if x in G
    ]

    sub = reachability.loc[
        (reachability["state"] == state)
        &
        reachability["receptor"].isin(inputs_in)
        &
        reachability["TF"].isin(outputs_in)
    ]

    covered_rows.append({
        "state": state,
        "inputs_in_prior": len(inputs_in),
        "outputs_in_prior": len(outputs_in),
        "covered_pairs": len(sub),
        "reachable_pairs": int(sub["reachable"].sum()),
        "reachable_fraction_covered":
            float(sub["reachable"].mean())
            if len(sub) > 0
            else np.nan,
        "median_shortest_path":
            sub.loc[
                sub["reachable"],
                "shortest_n_edges"
            ].median()
    })


covered_reachability = pd.DataFrame(
    covered_rows
)

display(
    covered_reachability
)

In [ ]:
# =============================================================================
# Step30F.2a Cell 119
# Signed directed reachability
# =============================================================================

from collections import deque


def signed_shortest_paths(
    graph,
    source
):
    """
    BFS on expanded state:
    (node, cumulative_sign)

    cumulative_sign:
      +1 = net activating
      -1 = net inhibitory
    """

    dist = {
        (source, 1): 0
    }

    q = deque([
        (source, 1)
    ])

    while q:

        node, sign = q.popleft()

        d = dist[
            (node, sign)
        ]

        for nxt in graph.successors(
            node
        ):

            edge_sign = int(
                graph[
                    node
                ][
                    nxt
                ]["mor"]
            )

            new_sign = (
                sign * edge_sign
            )

            key = (
                nxt,
                new_sign
            )

            if key not in dist:

                dist[key] = d + 1

                q.append(
                    key
                )

    return dist

In [ ]:
# =============================================================================
# Step30F.2a Cell 120
# Signed receptor-to-TF compatibility
# =============================================================================

signed_path_rows = []

all_receptors = (
    receptor_input_scores[
        "receptor_gene"
    ]
    .unique()
)

signed_cache = {}

for receptor in all_receptors:

    if receptor in G:

        signed_cache[
            receptor
        ] = signed_shortest_paths(
            G,
            receptor
        )


for _, row in (
    reachability
    .iterrows()
):

    receptor = row["receptor"]
    tf = row["TF"]
    state = row["state"]

    pos_len = np.nan
    neg_len = np.nan

    if receptor in signed_cache:

        dist = signed_cache[
            receptor
        ]

        if (
            tf,
            1
        ) in dist:

            pos_len = dist[
                (tf, 1)
            ]

        if (
            tf,
            -1
        ) in dist:

            neg_len = dist[
                (tf, -1)
            ]

    signed_path_rows.append({
        "state": state,
        "receptor": receptor,
        "TF": tf,

        "positive_path":
            np.isfinite(
                pos_len
            ),

        "negative_path":
            np.isfinite(
                neg_len
            ),

        "positive_shortest":
            pos_len,

        "negative_shortest":
            neg_len
    })


signed_reachability = pd.DataFrame(
    signed_path_rows
)

display(
    pd.crosstab(
        signed_reachability[
            "state"
        ],
        signed_reachability[
            "positive_path"
        ]
    )
)

In [ ]:
# =============================================================================
# Step30F.2a Cell 121
# Positive-sign reachability among prior-covered pairs
# =============================================================================

signed_summary_rows = []

for state in AXIS_STATES:

    inputs_in = (
        receptor_input_scores.loc[
            (
                receptor_input_scores[
                    "target_state"
                ] == state
            )
            &
            receptor_input_scores[
                "receptor_gene"
            ].isin(G.nodes),
            "receptor_gene"
        ]
        .tolist()
    )

    outputs_in = (
        tf_output_scores.loc[
            (
                tf_output_scores[
                    "target_state"
                ] == state
            )
            &
            tf_output_scores[
                "TF"
            ].isin(G.nodes),
            "TF"
        ]
        .tolist()
    )

    temp = (
        signed_reachability.loc[
            (
                signed_reachability[
                    "state"
                ] == state
            )
            &
            signed_reachability[
                "receptor"
            ].isin(
                inputs_in
            )
            &
            signed_reachability[
                "TF"
            ].isin(
                outputs_in
            )
        ]
    )

    signed_summary_rows.append({
        "state":
            state,

        "covered_pairs":
            len(temp),

        "positive_signed_pairs":
            int(
                temp[
                    "positive_path"
                ].sum()
            ),

        "positive_fraction":
            float(
                temp[
                    "positive_path"
                ].mean()
            )
            if len(temp)
            else np.nan,

        "median_positive_length":
            temp.loc[
                temp[
                    "positive_path"
                ],
                "positive_shortest"
            ].median()
    })


signed_summary = pd.DataFrame(
    signed_summary_rows
)

display(
    signed_summary
)

In [ ]:
# =============================================================================
# Step30F.2a Cell 122
# Focused signed IFNAR1 -> MS3 regulator audit
# =============================================================================

ms3_signed = (
    signed_reachability.loc[
        signed_reachability[
            "state"
        ] == "MS3"
    ]
    .sort_values(
        [
            "positive_path",
            "positive_shortest"
        ],
        ascending=[
            False,
            True
        ]
    )
)

display(
    ms3_signed
)

In [ ]:
# =============================================================================
# Step30G Cell 123
# State-specific causal-network score dictionaries
# =============================================================================

STATE_INPUTS = {}
STATE_OUTPUTS = {}

for state in AXIS_STATES:

    STATE_INPUTS[
        state
    ] = dict(
        zip(
            receptor_input_scores.loc[
                receptor_input_scores[
                    "target_state"
                ] == state,
                "receptor_gene"
            ],

            receptor_input_scores.loc[
                receptor_input_scores[
                    "target_state"
                ] == state,
                "input_score"
            ]
        )
    )

    STATE_OUTPUTS[
        state
    ] = dict(
        zip(
            tf_output_scores.loc[
                tf_output_scores[
                    "target_state"
                ] == state,
                "TF"
            ],

            tf_output_scores.loc[
                tf_output_scores[
                    "target_state"
                ] == state,
                "output_score"
            ]
        )
    )


for state in AXIS_STATES:

    print(
        "\n",
        state
    )

    print(
        "Inputs:",
        STATE_INPUTS[state]
    )

    print(
        "Outputs:",
        STATE_OUTPUTS[state]
    )

In [ ]:
# =============================================================================
# Step30G Compatibility Audit
# LIANA namespace compatibility
# =============================================================================

import importlib.metadata as md
import inspect

print(
    "LIANA version:",
    md.version("liana")
)

print(
    "li.mt.build_prior_network:",
    hasattr(
        li.mt,
        "build_prior_network"
    )
)

print(
    "li.rs.build_prior_network:",
    hasattr(
        li.rs,
        "build_prior_network"
    )
)

print(
    "li.mt.find_causalnet:",
    hasattr(
        li.mt,
        "find_causalnet"
    )
)

if hasattr(
    li.mt,
    "build_prior_network"
):
    BUILD_PRIOR = (
        li.mt.build_prior_network
    )

elif hasattr(
    li.rs,
    "build_prior_network"
):
    BUILD_PRIOR = (
        li.rs.build_prior_network
    )

else:
    raise AttributeError(
        "build_prior_network not found "
        "in li.mt or li.rs"
    )

print(
    "\nSelected builder:"
)

print(
    BUILD_PRIOR
)

print(
    "\nSignature:"
)

print(
    inspect.signature(
        BUILD_PRIOR
    )
)

print(
    "\nLIANA PRIOR BUILDER: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 124
# Build state-specific pruned prior graphs
# LIANA-version compatible
# =============================================================================

state_priors = {}

for state in AXIS_STATES:

    print(
        "\n" + "=" * 80
    )

    print(
        "Building prior:",
        state
    )

    print(
        "Inputs:",
        len(
            STATE_INPUTS[
                state
            ]
        )
    )

    print(
        "Outputs:",
        len(
            STATE_OUTPUTS[
                state
            ]
        )
    )

    state_priors[
        state
    ] = BUILD_PRIOR(
        ppis=signed_ppi,

        input_nodes=
            STATE_INPUTS[
                state
            ],

        output_nodes=
            STATE_OUTPUTS[
                state
            ],

        verbose=True
    )

    print(
        "Built:",
        state
    )

print(
    "\nSTEP30G STATE-SPECIFIC PRIORS: PASS"
)

In [ ]:
# =============================================================================
# Step30G CORNETO compatibility audit
# =============================================================================

import corneto as cn
import importlib.metadata as md
import inspect

print(
    "LIANA:",
    md.version("liana")
)

print(
    "CORNETO:",
    md.version("corneto")
)

print(
    "\nfind_causalnet signature:"
)

print(
    inspect.signature(
        li.mt.find_causalnet
    )
)

print(
    "\nCORNETO import: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 125
# Lock score dictionaries to nodes retained in each pruned prior
# LIANA 1.7.3 compatibility
# =============================================================================

STATE_INPUTS_PRIOR = {}
STATE_OUTPUTS_PRIOR = {}

prior_score_audit = []

for state in AXIS_STATES:

    prior_nodes = set(
        map(
            str,
            state_priors[state].V
        )
    )

    inputs_original = STATE_INPUTS[state]
    outputs_original = STATE_OUTPUTS[state]

    inputs_kept = {
        k: float(v)
        for k, v in inputs_original.items()
        if k in prior_nodes
    }

    outputs_kept = {
        k: float(v)
        for k, v in outputs_original.items()
        if k in prior_nodes
    }

    STATE_INPUTS_PRIOR[state] = inputs_kept
    STATE_OUTPUTS_PRIOR[state] = outputs_kept

    dropped_inputs = sorted(
        set(inputs_original)
        -
        set(inputs_kept)
    )

    dropped_outputs = sorted(
        set(outputs_original)
        -
        set(outputs_kept)
    )

    prior_score_audit.append({
        "state": state,

        "prior_nodes":
            len(prior_nodes),

        "input_original":
            len(inputs_original),

        "input_retained":
            len(inputs_kept),

        "output_original":
            len(outputs_original),

        "output_retained":
            len(outputs_kept),

        "dropped_inputs":
            " | ".join(
                dropped_inputs
            ),

        "dropped_outputs":
            " | ".join(
                dropped_outputs
            )
    })


prior_score_audit = pd.DataFrame(
    prior_score_audit
)

display(
    prior_score_audit
)

for state in AXIS_STATES:

    assert (
        len(
            STATE_INPUTS_PRIOR[state]
        ) > 0
    )

    assert (
        len(
            STATE_OUTPUTS_PRIOR[state]
        ) > 0
    )

print(
    "\nSTEP30G PRIOR-SCORE COMPATIBILITY: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 126
# Formal causal-network solver parameter lock
# =============================================================================

CAUSAL_PARAMS = {
    "node_weights": None,

    "missing_penalty": 10.0,

    "edge_penalty": 0.01,

    "solver": "SCIPY",

    "seed": 42,

    "max_runs": 1,

    "stable_runs": 5,

    "verbose": True
}


print(
    "LIANA:",
    md.version("liana")
)

print(
    "CORNETO:",
    md.version("corneto")
)

print(
    "\nFormal causal-network parameters:"
)

for k, v in CAUSAL_PARAMS.items():
    print(
        k,
        "=",
        v
    )

print(
    "\nSTEP30G SOLVER PARAMETER LOCK: PASS"
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step30G RECOVERY CELL
# Restore saved Step30F inputs after kernel restart
# =============================================================================

from pathlib import Path
import importlib.metadata as md

import numpy as np
import pandas as pd
import liana as li
import corneto as cn


ROOT = Path(_hcc_path(''))

STEP30_DIR = (
    ROOT /
    "LIHC_scRNA_Step30_Communication"
)

TABLE_DIR = STEP30_DIR / "Tables"
RESOURCE_DIR = STEP30_DIR / "Resources"


print("LIANA:", md.version("liana"))
print("CORNETO:", md.version("corneto"))

assert md.version("liana") == "1.7.3"
assert md.version("corneto") == "0.9.1a6"

assert hasattr(
    cn.methods.carnival,
    "_extended_carnival_problem"
)


# -------------------------------------------------------------------------
# Restore frozen Step30F score tables
# -------------------------------------------------------------------------

receptor_input_scores = pd.read_csv(
    TABLE_DIR /
    "Step30F_Receptor_InputScores.csv"
)

tf_output_scores = pd.read_csv(
    TABLE_DIR /
    "Step30F_TF_OutputScores.csv"
)

signed_ppi = pd.read_csv(
    RESOURCE_DIR /
    "Step30F_OmniPath_Primary_SignedPPI.csv"
)


AXIS_STATES = [
    "MS6",
    "MS5",
    "MS3"
]


STATE_INPUTS = {}
STATE_OUTPUTS = {}


for state in AXIS_STATES:

    sub_in = receptor_input_scores.loc[
        receptor_input_scores[
            "target_state"
        ] == state
    ]

    STATE_INPUTS[state] = dict(
        zip(
            sub_in["receptor_gene"],
            sub_in["input_score"].astype(float)
        )
    )

    sub_out = tf_output_scores.loc[
        tf_output_scores[
            "target_state"
        ] == state
    ]

    STATE_OUTPUTS[state] = dict(
        zip(
            sub_out["TF"],
            sub_out["output_score"].astype(float)
        )
    )


print(
    "\nSigned PPI:",
    signed_ppi.shape
)

for state in AXIS_STATES:

    print(
        state,
        "| inputs:",
        len(STATE_INPUTS[state]),
        "| outputs:",
        len(STATE_OUTPUTS[state])
    )


print(
    "\nSTEP30G RECOVERY: PASS"
)

In [ ]:
# =============================================================================
# Step30G RECOVERY
# LIANA 1.7.3 prior-builder compatibility
# =============================================================================

if hasattr(
    li.mt,
    "build_prior_network"
):

    BUILD_PRIOR = li.mt.build_prior_network

elif hasattr(
    li.rs,
    "build_prior_network"
):

    BUILD_PRIOR = li.rs.build_prior_network

else:

    raise AttributeError(
        "build_prior_network not found."
    )


print(
    "Prior builder:",
    BUILD_PRIOR
)

print(
    "\nPRIOR BUILDER: PASS"
)

In [ ]:
# =============================================================================
# Step30G RECOVERY
# Rebuild state-specific pruned priors
# =============================================================================

state_priors = {}


for state in AXIS_STATES:

    print(
        "\n" + "=" * 80
    )

    print(
        "Building prior:",
        state
    )

    state_priors[state] = BUILD_PRIOR(
        ppis=signed_ppi,

        input_nodes=
            STATE_INPUTS[state],

        output_nodes=
            STATE_OUTPUTS[state],

        verbose=True
    )


print(
    "\nSTATE PRIORS REBUILT: PASS"
)

In [ ]:
# =============================================================================
# Step30G RECOVERY
# Retain only input/output nodes preserved by prior pruning
# CORNETO 0.9.1a6 compatible
# =============================================================================

STATE_INPUTS_PRIOR = {}
STATE_OUTPUTS_PRIOR = {}


def get_prior_nodes(graph):
    """
    Compatibility helper for different CORNETO Graph versions.
    """

    if hasattr(graph, "vertices"):
        return set(
            map(
                str,
                graph.vertices
            )
        )

    elif hasattr(graph, "V"):
        return set(
            map(
                str,
                graph.V
            )
        )

    elif hasattr(graph, "nodes"):
        nodes = graph.nodes

        if callable(nodes):
            nodes = nodes()

        return set(
            map(
                str,
                nodes
            )
        )

    else:
        raise AttributeError(
            "Cannot identify graph node interface."
        )


for state in AXIS_STATES:

    prior_nodes = get_prior_nodes(
        state_priors[state]
    )

    STATE_INPUTS_PRIOR[state] = {
        node: float(score)

        for node, score
        in STATE_INPUTS[state].items()

        if node in prior_nodes
    }

    STATE_OUTPUTS_PRIOR[state] = {
        node: float(score)

        for node, score
        in STATE_OUTPUTS[state].items()

        if node in prior_nodes
    }


    dropped_inputs = sorted(
        set(
            STATE_INPUTS[state]
        )
        -
        set(
            STATE_INPUTS_PRIOR[state]
        )
    )

    dropped_outputs = sorted(
        set(
            STATE_OUTPUTS[state]
        )
        -
        set(
            STATE_OUTPUTS_PRIOR[state]
        )
    )


    print(
        "\n",
        state
    )

    print(
        "Prior nodes:",
        len(prior_nodes)
    )

    print(
        "Inputs:",
        len(
            STATE_INPUTS_PRIOR[state]
        ),
        "/",
        len(
            STATE_INPUTS[state]
        )
    )

    print(
        "Outputs:",
        len(
            STATE_OUTPUTS_PRIOR[state]
        ),
        "/",
        len(
            STATE_OUTPUTS[state]
        )
    )

    print(
        "Dropped inputs:",
        dropped_inputs
    )

    print(
        "Dropped outputs:",
        dropped_outputs
    )


print(
    "\nPRIOR SCORE FILTER: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 127A
# Audit MS3 receptor-to-TF path-corridor sizes
# =============================================================================

import networkx as nx
import numpy as np
import pandas as pd


G_full = nx.from_pandas_edgelist(
    signed_ppi,
    source="source",
    target="target",
    edge_attr="mor",
    create_using=nx.DiGraph
)

MS3_INPUTS = list(
    STATE_INPUTS_PRIOR["MS3"].keys()
)

MS3_OUTPUTS = list(
    STATE_OUTPUTS_PRIOR["MS3"].keys()
)

print("Inputs:", MS3_INPUTS)
print("Outputs:", MS3_OUTPUTS)


def build_path_corridor(
    graph,
    inputs,
    outputs,
    max_length
):
    # Minimum distance from any input
    d_from_input = {}

    for inp in inputs:
        if inp not in graph:
            continue

        dist = nx.single_source_shortest_path_length(
            graph,
            inp,
            cutoff=max_length
        )

        for node, d in dist.items():
            d_from_input[node] = min(
                d,
                d_from_input.get(node, np.inf)
            )

    # Minimum distance from node to any output
    reverse_graph = graph.reverse(
        copy=False
    )

    d_to_output = {}

    for out in outputs:
        if out not in graph:
            continue

        dist = nx.single_source_shortest_path_length(
            reverse_graph,
            out,
            cutoff=max_length
        )

        for node, d in dist.items():
            d_to_output[node] = min(
                d,
                d_to_output.get(node, np.inf)
            )

    # Keep only nodes that can lie on an input -> output path
    # with total path length <= max_length
    corridor_nodes = {
        node
        for node in graph.nodes
        if (
            node in d_from_input
            and
            node in d_to_output
            and
            d_from_input[node]
            +
            d_to_output[node]
            <= max_length
        )
    }

    # Keep only edges that can participate in such a path
    corridor_edges = []

    for u, v, data in graph.edges(
        data=True
    ):

        if (
            u in corridor_nodes
            and
            v in corridor_nodes
            and
            u in d_from_input
            and
            v in d_to_output
            and
            d_from_input[u]
            + 1
            + d_to_output[v]
            <= max_length
        ):

            corridor_edges.append(
                (
                    u,
                    v,
                    data.get("mor")
                )
            )

    return (
        corridor_nodes,
        corridor_edges
    )


audit_rows = []

for L in [
    4,
    5,
    6,
    7,
    8
]:

    nodes_L, edges_L = (
        build_path_corridor(
            G_full,
            MS3_INPUTS,
            MS3_OUTPUTS,
            max_length=L
        )
    )

    outputs_retained = [
        x
        for x in MS3_OUTPUTS
        if x in nodes_L
    ]

    audit_rows.append({
        "max_path_length":
            L,

        "n_nodes":
            len(nodes_L),

        "n_edges":
            len(edges_L),

        "outputs_retained":
            len(outputs_retained),

        "output_names":
            " | ".join(
                outputs_retained
            )
    })


corridor_audit = pd.DataFrame(
    audit_rows
)

display(
    corridor_audit
)

In [ ]:
# =============================================================================
# Step30G Cell 127B
# Freeze PRIMARY MS3 path corridor
# Primary cutoff: <= 5 directed edges
# =============================================================================

MS3_MAX_PATH_PRIMARY = 5

ms3_corridor_nodes, ms3_corridor_edges = (
    build_path_corridor(
        G_full,
        MS3_INPUTS,
        MS3_OUTPUTS,
        max_length=MS3_MAX_PATH_PRIMARY
    )
)

ms3_corridor_ppi = pd.DataFrame(
    ms3_corridor_edges,
    columns=[
        "source",
        "target",
        "mor"
    ]
)

print(
    "Primary max path length:",
    MS3_MAX_PATH_PRIMARY
)

print(
    "MS3 corridor nodes:",
    len(ms3_corridor_nodes)
)

print(
    "MS3 corridor edges:",
    len(ms3_corridor_ppi)
)

inputs_retained = [
    x
    for x in MS3_INPUTS
    if x in ms3_corridor_nodes
]

outputs_retained = [
    x
    for x in MS3_OUTPUTS
    if x in ms3_corridor_nodes
]

print(
    "\nInputs retained:",
    inputs_retained
)

print(
    "Outputs retained:",
    outputs_retained
)

assert len(inputs_retained) == len(MS3_INPUTS)
assert len(outputs_retained) == len(MS3_OUTPUTS)

print(
    "\nMS3 PRIMARY PATH CORRIDOR: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 127C
# Save path-corridor audit and primary MS3 PPI
# =============================================================================

corridor_audit.to_csv(
    TABLE_DIR /
    "Step30G_MS3_PathCorridor_CutoffAudit.csv",
    index=False
)

ms3_corridor_ppi.to_csv(
    TABLE_DIR /
    "Step30G_MS3_PathCorridor_L5_Primary.csv",
    index=False
)

print(
    "Saved cutoff audit and primary L5 corridor."
)

In [ ]:
# =============================================================================
# Step30G Cell 127D
# Build reduced MS3 prior from L5 corridor
# =============================================================================

ms3_prior_reduced = BUILD_PRIOR(
    ppis=
        ms3_corridor_ppi,

    input_nodes=
        STATE_INPUTS_PRIOR["MS3"],

    output_nodes=
        STATE_OUTPUTS_PRIOR["MS3"],

    verbose=True
)

ms3_reduced_nodes = set(
    map(
        str,
        ms3_prior_reduced.vertices
    )
)

print(
    "\nReduced prior nodes:",
    len(ms3_reduced_nodes)
)

print(
    "Inputs retained:",
    [
        x
        for x in MS3_INPUTS
        if x in ms3_reduced_nodes
    ]
)

print(
    "Outputs retained:",
    [
        x
        for x in MS3_OUTPUTS
        if x in ms3_reduced_nodes
    ]
)

assert all(
    x in ms3_reduced_nodes
    for x in MS3_INPUTS
)

assert all(
    x in ms3_reduced_nodes
    for x in MS3_OUTPUTS
)

print(
    "\nREDUCED MS3 PRIOR: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 127E
# MS3 causal-network solver on reduced L5 prior
# =============================================================================

ms3_network, ms3_problem = (
    li.mt.find_causalnet(

        prior_graph=
            ms3_prior_reduced,

        input_node_scores=
            STATE_INPUTS_PRIOR["MS3"],

        output_node_scores=
            STATE_OUTPUTS_PRIOR["MS3"],

        node_weights=None,

        missing_penalty=10.0,
        edge_penalty=0.01,

        solver="scipy",

        seed=42,
        max_runs=1,
        stable_runs=5,

        verbose=True,

        scipy_options={
            "disp": True,
            "presolve": True,
            "time_limit": 900,
            "mip_rel_gap": 0.01
        }
    )
)

print(
    "\nMS3 causal network shape:",
    ms3_network.shape
)

print(
    "\nColumns:"
)

print(
    ms3_network.columns.tolist()
)

display(
    ms3_network.head(50)
)

assert len(ms3_network) > 0

print(
    "\nSTEP30G MS3 L5 CAUSAL SOLVER: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 128
# Clean MS3 causal network and audit selected endpoints
# =============================================================================

TECH_NODES = {
    "_s",
    "_pert_c0",
    "_meas_c0",
    ""
}

ms3_network_clean = (
    ms3_network.loc[
        ~ms3_network["source"].astype(str).isin(TECH_NODES)
        &
        ~ms3_network["target"].astype(str).isin(TECH_NODES)
    ]
    .copy()
)

ms3_nodes_clean = set(
    ms3_network_clean["source"].astype(str)
).union(
    set(
        ms3_network_clean["target"].astype(str)
    )
)

ms3_selected_inputs = [
    x
    for x in STATE_INPUTS_PRIOR["MS3"]
    if x in ms3_nodes_clean
]

ms3_selected_outputs = [
    x
    for x in STATE_OUTPUTS_PRIOR["MS3"]
    if x in ms3_nodes_clean
]

print(
    "Clean biological edges:",
    len(ms3_network_clean)
)

print(
    "Clean biological nodes:",
    len(ms3_nodes_clean)
)

print(
    "\nSelected inputs:",
    ms3_selected_inputs
)

print(
    "Selected outputs:",
    ms3_selected_outputs
)

display(
    ms3_network_clean
)

In [ ]:
# =============================================================================
# Step30G Cell 129
# MS3 primary network formal lock
# =============================================================================

ms3_prior_nodes = set(
    map(
        str,
        ms3_prior_reduced.vertices
    )
)

MS3_LOCK = {
    "path_corridor_max_edges":
        MS3_MAX_PATH_PRIMARY,

    "corridor_nodes":
        len(ms3_corridor_nodes),

    "corridor_edges":
        len(ms3_corridor_ppi),

    "reduced_prior_nodes":
        len(ms3_prior_nodes),

    "solver":
        "SCIPY/HiGHS",

    "seed":
        42,

    "edge_penalty":
        0.01,

    "missing_penalty":
        10.0,

    "mip_rel_gap":
        0.01,

    "selected_inputs":
        ms3_selected_inputs,

    "selected_outputs":
        ms3_selected_outputs,

    "clean_network_edges":
        len(ms3_network_clean),

    "clean_network_nodes":
        len(ms3_nodes_clean)
}

print(
    "MS3 PRIMARY CAUSAL NETWORK LOCK"
)

for k, v in MS3_LOCK.items():
    print(
        k,
        ":",
        v
    )

ms3_network_clean.to_csv(
    TABLE_DIR /
    "Step30G_MS3_CausalNetwork_Primary_Clean.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step30G Cell 130
# Generic state-specific path-corridor audit
# =============================================================================

def audit_state_corridors(
    state,
    graph,
    state_inputs_prior,
    state_outputs_prior,
    lengths=(3, 4, 5, 6, 7, 8)
):

    inputs = list(
        state_inputs_prior[
            state
        ].keys()
    )

    outputs = list(
        state_outputs_prior[
            state
        ].keys()
    )

    rows = []

    for L in lengths:

        nodes_L, edges_L = (
            build_path_corridor(
                graph,
                inputs,
                outputs,
                max_length=L
            )
        )

        inputs_retained = [
            x
            for x in inputs
            if x in nodes_L
        ]

        outputs_retained = [
            x
            for x in outputs
            if x in nodes_L
        ]

        rows.append({
            "state":
                state,

            "max_path_length":
                L,

            "n_nodes":
                len(nodes_L),

            "n_edges":
                len(edges_L),

            "inputs_total":
                len(inputs),

            "inputs_retained":
                len(inputs_retained),

            "outputs_total":
                len(outputs),

            "outputs_retained":
                len(outputs_retained),

            "input_names":
                " | ".join(
                    inputs_retained
                ),

            "output_names":
                " | ".join(
                    outputs_retained
                )
        })

    return pd.DataFrame(
        rows
    )

In [ ]:
# =============================================================================
# Step30G Cell 131
# MS5 path-corridor audit
# =============================================================================

print(
    "MS5 inputs:"
)

print(
    STATE_INPUTS_PRIOR[
        "MS5"
    ]
)

print(
    "\nMS5 outputs:"
)

print(
    STATE_OUTPUTS_PRIOR[
        "MS5"
    ]
)


ms5_corridor_audit = (
    audit_state_corridors(
        state="MS5",

        graph=G_full,

        state_inputs_prior=
            STATE_INPUTS_PRIOR,

        state_outputs_prior=
            STATE_OUTPUTS_PRIOR,

        lengths=[
            3,
            4,
            5,
            6,
            7,
            8
        ]
    )
)

display(
    ms5_corridor_audit
)

In [ ]:
# =============================================================================
# Step30G Cell 132
# MS6 path-corridor audit
# =============================================================================

print(
    "MS6 inputs:"
)

print(
    STATE_INPUTS_PRIOR[
        "MS6"
    ]
)

print(
    "\nMS6 outputs:"
)

print(
    STATE_OUTPUTS_PRIOR[
        "MS6"
    ]
)


ms6_corridor_audit = (
    audit_state_corridors(
        state="MS6",

        graph=G_full,

        state_inputs_prior=
            STATE_INPUTS_PRIOR,

        state_outputs_prior=
            STATE_OUTPUTS_PRIOR,

        lengths=[
            3,
            4,
            5,
            6,
            7,
            8
        ]
    )
)

display(
    ms6_corridor_audit
)

In [ ]:
# =============================================================================
# Step30G Cell 133
# Save state-specific corridor audits
# =============================================================================

corridor_audit_all = pd.concat(
    [
        corridor_audit.assign(
            state="MS3"
        ),
        ms5_corridor_audit,
        ms6_corridor_audit
    ],
    ignore_index=True,
    sort=False
)


corridor_audit_all.to_csv(
    TABLE_DIR /
    "Step30G_AllStates_PathCorridor_CutoffAudit.csv",
    index=False
)


print(
    "Saved:"
)

print(
    TABLE_DIR /
    "Step30G_AllStates_PathCorridor_CutoffAudit.csv"
)

print(
    "\nMS5:"
)

display(
    ms5_corridor_audit
)

print(
    "\nMS6:"
)

display(
    ms6_corridor_audit
)

In [ ]:
# =============================================================================
# Step30G Cell 134
# FINAL primary path-corridor lock
# =============================================================================

PRIMARY_PATH_CUTOFF = {
    "MS3": 5,
    "MS5": 4,
    "MS6": 5
}

print("PRIMARY PATH-CORRIDOR LOCK")

for state, cutoff in PRIMARY_PATH_CUTOFF.items():
    print(
        state,
        "->",
        cutoff,
        "directed edges"
    )

In [ ]:
# =============================================================================
# Step30G Cell 135
# Build MS5 primary path corridor
# =============================================================================

MS5_INPUTS = list(
    STATE_INPUTS_PRIOR["MS5"].keys()
)

MS5_OUTPUTS = list(
    STATE_OUTPUTS_PRIOR["MS5"].keys()
)

ms5_corridor_nodes, ms5_corridor_edges = (
    build_path_corridor(
        G_full,
        MS5_INPUTS,
        MS5_OUTPUTS,
        max_length=PRIMARY_PATH_CUTOFF["MS5"]
    )
)

ms5_corridor_ppi = pd.DataFrame(
    ms5_corridor_edges,
    columns=[
        "source",
        "target",
        "mor"
    ]
)

print(
    "MS5 corridor nodes:",
    len(ms5_corridor_nodes)
)

print(
    "MS5 corridor edges:",
    len(ms5_corridor_ppi)
)

print(
    "Inputs retained:",
    [
        x
        for x in MS5_INPUTS
        if x in ms5_corridor_nodes
    ]
)

print(
    "Outputs retained:",
    [
        x
        for x in MS5_OUTPUTS
        if x in ms5_corridor_nodes
    ]
)

assert all(
    x in ms5_corridor_nodes
    for x in MS5_INPUTS
)

assert all(
    x in ms5_corridor_nodes
    for x in MS5_OUTPUTS
)

print(
    "\nMS5 PRIMARY CORRIDOR: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 136
# Build MS6 primary path corridor
# =============================================================================

MS6_INPUTS = list(
    STATE_INPUTS_PRIOR["MS6"].keys()
)

MS6_OUTPUTS = list(
    STATE_OUTPUTS_PRIOR["MS6"].keys()
)

ms6_corridor_nodes, ms6_corridor_edges = (
    build_path_corridor(
        G_full,
        MS6_INPUTS,
        MS6_OUTPUTS,
        max_length=PRIMARY_PATH_CUTOFF["MS6"]
    )
)

ms6_corridor_ppi = pd.DataFrame(
    ms6_corridor_edges,
    columns=[
        "source",
        "target",
        "mor"
    ]
)

print(
    "MS6 corridor nodes:",
    len(ms6_corridor_nodes)
)

print(
    "MS6 corridor edges:",
    len(ms6_corridor_ppi)
)

print(
    "Inputs retained:",
    [
        x
        for x in MS6_INPUTS
        if x in ms6_corridor_nodes
    ]
)

print(
    "Outputs retained:",
    [
        x
        for x in MS6_OUTPUTS
        if x in ms6_corridor_nodes
    ]
)

assert all(
    x in ms6_corridor_nodes
    for x in MS6_INPUTS
)

assert all(
    x in ms6_corridor_nodes
    for x in MS6_OUTPUTS
)

print(
    "\nMS6 PRIMARY CORRIDOR: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 137
# Save primary state-specific corridors
# =============================================================================

ms5_corridor_ppi.to_csv(
    TABLE_DIR /
    "Step30G_MS5_PathCorridor_L4_Primary.csv",
    index=False
)

ms6_corridor_ppi.to_csv(
    TABLE_DIR /
    "Step30G_MS6_PathCorridor_L5_Primary.csv",
    index=False
)

print(
    "MS5/MS6 primary corridors saved."
)

In [ ]:
# =============================================================================
# Step30G Cell 138
# Build reduced MS5 prior
# =============================================================================

ms5_prior_reduced = BUILD_PRIOR(
    ppis=
        ms5_corridor_ppi,

    input_nodes=
        STATE_INPUTS_PRIOR["MS5"],

    output_nodes=
        STATE_OUTPUTS_PRIOR["MS5"],

    verbose=True
)

ms5_reduced_nodes = set(
    map(
        str,
        ms5_prior_reduced.vertices
    )
)

print(
    "\nMS5 reduced prior nodes:",
    len(ms5_reduced_nodes)
)

print(
    "Inputs retained:",
    [
        x
        for x in MS5_INPUTS
        if x in ms5_reduced_nodes
    ]
)

print(
    "Outputs retained:",
    [
        x
        for x in MS5_OUTPUTS
        if x in ms5_reduced_nodes
    ]
)

assert all(
    x in ms5_reduced_nodes
    for x in MS5_INPUTS
)

assert all(
    x in ms5_reduced_nodes
    for x in MS5_OUTPUTS
)

print(
    "\nMS5 REDUCED PRIOR: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 139
# Build reduced MS6 prior
# =============================================================================

ms6_prior_reduced = BUILD_PRIOR(
    ppis=
        ms6_corridor_ppi,

    input_nodes=
        STATE_INPUTS_PRIOR["MS6"],

    output_nodes=
        STATE_OUTPUTS_PRIOR["MS6"],

    verbose=True
)

ms6_reduced_nodes = set(
    map(
        str,
        ms6_prior_reduced.vertices
    )
)

print(
    "\nMS6 reduced prior nodes:",
    len(ms6_reduced_nodes)
)

print(
    "Inputs retained:",
    [
        x
        for x in MS6_INPUTS
        if x in ms6_reduced_nodes
    ]
)

print(
    "Outputs retained:",
    [
        x
        for x in MS6_OUTPUTS
        if x in ms6_reduced_nodes
    ]
)

assert all(
    x in ms6_reduced_nodes
    for x in MS6_INPUTS
)

assert all(
    x in ms6_reduced_nodes
    for x in MS6_OUTPUTS
)

print(
    "\nMS6 REDUCED PRIOR: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 140
# MS5 primary causal-network inference
# =============================================================================

ms5_network, ms5_problem = (
    li.mt.find_causalnet(

        prior_graph=
            ms5_prior_reduced,

        input_node_scores=
            STATE_INPUTS_PRIOR["MS5"],

        output_node_scores=
            STATE_OUTPUTS_PRIOR["MS5"],

        node_weights=None,

        missing_penalty=10.0,
        edge_penalty=0.01,

        solver="scipy",

        seed=42,
        max_runs=1,
        stable_runs=5,

        verbose=True,

        scipy_options={
            "disp": True,
            "presolve": True,
            "time_limit": 900,
            "mip_rel_gap": 0.01
        }
    )
)

print(
    "\nMS5 causal network shape:",
    ms5_network.shape
)

print(
    "\nColumns:"
)

print(
    ms5_network.columns.tolist()
)

display(
    ms5_network.head(60)
)

print(
    "\nSTEP30G MS5 CAUSAL SOLVER: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 141
# Clean MS5 biological causal network
# =============================================================================

TECH_NODES = {
    "_s",
    "_pert_c0",
    "_meas_c0",
    ""
}

ms5_network_clean = (
    ms5_network.loc[
        ~ms5_network[
            "source"
        ].astype(str).isin(
            TECH_NODES
        )
        &
        ~ms5_network[
            "target"
        ].astype(str).isin(
            TECH_NODES
        )
    ]
    .copy()
)

ms5_nodes_clean = set(
    ms5_network_clean[
        "source"
    ].astype(str)
).union(
    set(
        ms5_network_clean[
            "target"
        ].astype(str)
    )
)

ms5_selected_inputs = [
    x
    for x in STATE_INPUTS_PRIOR["MS5"]
    if x in ms5_nodes_clean
]

ms5_selected_outputs = [
    x
    for x in STATE_OUTPUTS_PRIOR["MS5"]
    if x in ms5_nodes_clean
]

print(
    "Clean biological edges:",
    len(ms5_network_clean)
)

print(
    "Clean biological nodes:",
    len(ms5_nodes_clean)
)

print(
    "\nSelected inputs:",
    ms5_selected_inputs
)

print(
    "Selected outputs:",
    ms5_selected_outputs
)

display(
    ms5_network_clean
)

In [ ]:
# =============================================================================
# Step30G Cell 142
# MS6 primary causal-network inference
# =============================================================================

ms6_network, ms6_problem = (
    li.mt.find_causalnet(

        prior_graph=
            ms6_prior_reduced,

        input_node_scores=
            STATE_INPUTS_PRIOR["MS6"],

        output_node_scores=
            STATE_OUTPUTS_PRIOR["MS6"],

        node_weights=None,

        missing_penalty=10.0,
        edge_penalty=0.01,

        solver="scipy",

        seed=42,
        max_runs=1,
        stable_runs=5,

        verbose=True,

        scipy_options={
            "disp": True,
            "presolve": True,
            "time_limit": 900,
            "mip_rel_gap": 0.01
        }
    )
)

print(
    "\nMS6 causal network shape:",
    ms6_network.shape
)

print(
    "\nColumns:"
)

print(
    ms6_network.columns.tolist()
)

display(
    ms6_network.head(80)
)

print(
    "\nSTEP30G MS6 CAUSAL SOLVER: PASS"
)

In [ ]:
# =============================================================================
# Step30G Cell 143
# Clean MS6 biological causal network
# =============================================================================

ms6_network_clean = (
    ms6_network.loc[
        ~ms6_network[
            "source"
        ].astype(str).isin(
            TECH_NODES
        )
        &
        ~ms6_network[
            "target"
        ].astype(str).isin(
            TECH_NODES
        )
    ]
    .copy()
)

ms6_nodes_clean = set(
    ms6_network_clean[
        "source"
    ].astype(str)
).union(
    set(
        ms6_network_clean[
            "target"
        ].astype(str)
    )
)

ms6_selected_inputs = [
    x
    for x in STATE_INPUTS_PRIOR["MS6"]
    if x in ms6_nodes_clean
]

ms6_selected_outputs = [
    x
    for x in STATE_OUTPUTS_PRIOR["MS6"]
    if x in ms6_nodes_clean
]

print(
    "Clean biological edges:",
    len(ms6_network_clean)
)

print(
    "Clean biological nodes:",
    len(ms6_nodes_clean)
)

print(
    "\nSelected inputs:",
    ms6_selected_inputs
)

print(
    "Selected outputs:",
    ms6_selected_outputs
)

display(
    ms6_network_clean
)

In [ ]:
# =============================================================================
# Step30G — Role-aware causal-network audit
# Run for MS6 first
# =============================================================================

import networkx as nx
import pandas as pd


def audit_causal_solution(
    network,
    state_inputs,
    state_outputs,
    tech_nodes=None
):
    if tech_nodes is None:
        tech_nodes = {
            "_s",
            "_pert_c0",
            "_meas_c0",
            ""
        }

    clean = (
        network.loc[
            ~network["source"].astype(str).isin(tech_nodes)
            &
            ~network["target"].astype(str).isin(tech_nodes)
        ]
        .copy()
    )

    # -------------------------------------------------------------
    # True CORNETO role-aware selected inputs
    # -------------------------------------------------------------
    true_inputs = set(
        clean.loc[
            clean["source_type"] == "input",
            "source"
        ].astype(str)
    ).union(
        set(
            clean.loc[
                clean["target_type"] == "input",
                "target"
            ].astype(str)
        )
    )

    true_inputs = sorted(
        true_inputs &
        set(state_inputs)
    )

    # -------------------------------------------------------------
    # True CORNETO role-aware selected outputs
    # -------------------------------------------------------------
    true_outputs = set(
        clean.loc[
            clean["source_type"] == "output",
            "source"
        ].astype(str)
    ).union(
        set(
            clean.loc[
                clean["target_type"] == "output",
                "target"
            ].astype(str)
        )
    )

    true_outputs = sorted(
        true_outputs &
        set(state_outputs)
    )

    # -------------------------------------------------------------
    # Directed biological graph
    # -------------------------------------------------------------
    H = nx.from_pandas_edgelist(
        clean,
        source="source",
        target="target",
        edge_attr=True,
        create_using=nx.DiGraph
    )

    # -------------------------------------------------------------
    # Which input-output pairs are actually connected?
    # -------------------------------------------------------------
    io_rows = []

    for inp in true_inputs:
        for out in true_outputs:

            connected = (
                inp in H
                and out in H
                and nx.has_path(
                    H,
                    inp,
                    out
                )
            )

            io_rows.append({
                "input": inp,
                "output": out,
                "connected": connected,
                "shortest_edges":
                    nx.shortest_path_length(
                        H,
                        inp,
                        out
                    )
                    if connected
                    else None
            })

    io_table = pd.DataFrame(
        io_rows
    )

    # -------------------------------------------------------------
    # Keep only edges belonging to a component/path that actually
    # connects a measured input to a measured output
    # -------------------------------------------------------------
    mechanistic_nodes = set()

    for inp in true_inputs:
        for out in true_outputs:

            if (
                inp in H
                and out in H
                and nx.has_path(
                    H,
                    inp,
                    out
                )
            ):
                for path in nx.all_simple_paths(
                    H,
                    inp,
                    out,
                    cutoff=10
                ):
                    mechanistic_nodes.update(
                        path
                    )

    mechanistic_network = (
        clean.loc[
            clean["source"].isin(
                mechanistic_nodes
            )
            &
            clean["target"].isin(
                mechanistic_nodes
            )
        ]
        .copy()
    )

    return (
        clean,
        true_inputs,
        true_outputs,
        io_table,
        mechanistic_network
    )


(
    ms6_network_clean2,
    ms6_true_inputs,
    ms6_true_outputs,
    ms6_io_connectivity,
    ms6_mechanistic_network
) = audit_causal_solution(

    ms6_network,

    state_inputs=
        STATE_INPUTS_PRIOR["MS6"],

    state_outputs=
        STATE_OUTPUTS_PRIOR["MS6"]
)


print(
    "True selected inputs:",
    ms6_true_inputs
)

print(
    "True selected outputs:",
    ms6_true_outputs
)

print(
    "\nInput -> output connectivity:"
)

display(
    ms6_io_connectivity
)

print(
    "\nMechanistically connected network:"
)

display(
    ms6_mechanistic_network
)

In [ ]:
# =============================================================================
# Step30G — MS6 primary mechanistic-network lock
# =============================================================================

MS6_PRIMARY_LOCK = {
    "state": "MS6",
    "path_corridor_max_edges": PRIMARY_PATH_CUTOFF["MS6"],

    "corridor_nodes": len(ms6_corridor_nodes),
    "corridor_edges": len(ms6_corridor_ppi),

    "reduced_prior_nodes": len(ms6_reduced_nodes),

    "true_selected_inputs": ms6_true_inputs,
    "true_selected_outputs": ms6_true_outputs,

    "connected_inputs": (
        ms6_io_connectivity.loc[
            ms6_io_connectivity["connected"],
            "input"
        ]
        .tolist()
    ),

    "mechanistic_nodes": sorted(
        set(
            ms6_mechanistic_network["source"].astype(str)
        ).union(
            set(
                ms6_mechanistic_network["target"].astype(str)
            )
        )
    ),

    "mechanistic_edges": len(
        ms6_mechanistic_network
    ),

    "primary_route": "RXRA -> NR1H4",

    "solver": "SCIPY/HiGHS",
    "seed": 42,
    "edge_penalty": 0.01,
    "missing_penalty": 10.0,
    "mip_rel_gap": 0.01
}


ms6_mechanistic_network.to_csv(
    TABLE_DIR /
    "Step30G_MS6_CausalNetwork_Primary_Mechanistic.csv",
    index=False
)


print("MS6 PRIMARY MECHANISTIC LOCK")

for k, v in MS6_PRIMARY_LOCK.items():
    print(k, ":", v)

In [ ]:
# =============================================================================
# Step30G — MS5 role-aware causal-network audit
# =============================================================================

(
    ms5_network_clean2,
    ms5_true_inputs,
    ms5_true_outputs,
    ms5_io_connectivity,
    ms5_mechanistic_network
) = audit_causal_solution(

    ms5_network,

    state_inputs=
        STATE_INPUTS_PRIOR["MS5"],

    state_outputs=
        STATE_OUTPUTS_PRIOR["MS5"]
)


print(
    "True selected inputs:",
    ms5_true_inputs
)

print(
    "True selected outputs:",
    ms5_true_outputs
)


print(
    "\nInput -> output connectivity:"
)

display(
    ms5_io_connectivity
)


print(
    "\nMechanistically connected network:"
)

display(
    ms5_mechanistic_network
)

In [ ]:
# =============================================================================
# Step30G — MS5 primary mechanistic-network lock
# =============================================================================

MS5_PRIMARY_LOCK = {
    "state": "MS5",

    "path_corridor_max_edges":
        PRIMARY_PATH_CUTOFF["MS5"],

    "corridor_nodes":
        len(ms5_corridor_nodes),

    "corridor_edges":
        len(ms5_corridor_ppi),

    "reduced_prior_nodes":
        len(ms5_reduced_nodes),

    "true_selected_inputs":
        ms5_true_inputs,

    "true_selected_outputs":
        ms5_true_outputs,

    "connected_inputs":
        ms5_io_connectivity.loc[
            ms5_io_connectivity[
                "connected"
            ],
            "input"
        ].tolist(),

    "mechanistic_nodes":
        sorted(
            set(
                ms5_mechanistic_network[
                    "source"
                ].astype(str)
            ).union(
                set(
                    ms5_mechanistic_network[
                        "target"
                    ].astype(str)
                )
            )
        ),

    "mechanistic_edges":
        len(
            ms5_mechanistic_network
        ),

    "primary_route":
        "ITGB1 -> SRC -> FOS",

    "communication_evidence_tier":
        "Secondary_Moderate_3to5pct",

    "solver":
        "SCIPY/HiGHS",

    "seed":
        42,

    "edge_penalty":
        0.01,

    "missing_penalty":
        10.0,

    "mip_rel_gap":
        0.01
}


ms5_mechanistic_network.to_csv(
    TABLE_DIR /
    "Step30G_MS5_CausalNetwork_Primary_Mechanistic.csv",
    index=False
)


print(
    "MS5 PRIMARY MECHANISTIC LOCK"
)

for k, v in MS5_PRIMARY_LOCK.items():
    print(
        k,
        ":",
        v
    )

In [ ]:
# =============================================================================
# Step30G — Primary mechanistic-route summary
# =============================================================================

primary_route_summary = pd.DataFrame([
    {
        "state": "MS3",
        "communication_level": "Primary robust, exploratory LR provenance",
        "entry_node": "IFNAR1",
        "intermediate_nodes": "PIK3CA_PIK3R1 | AKT1",
        "regulator_output": "BRCA1",
        "route": "IFNAR1 -> PIK3CA_PIK3R1 -> AKT1 -> BRCA1",
        "n_edges": 3
    },

    {
        "state": "MS5",
        "communication_level": "Secondary moderate",
        "entry_node": "ITGB1",
        "intermediate_nodes": "SRC",
        "regulator_output": "FOS",
        "route": "ITGB1 -> SRC -> FOS",
        "n_edges": 2
    },

    {
        "state": "MS6",
        "communication_level": "Primary robust communication landscape",
        "entry_node": "RXRA",
        "intermediate_nodes": "",
        "regulator_output": "NR1H4",
        "route": "RXRA -> NR1H4",
        "n_edges": 1
    }
])

display(
    primary_route_summary
)

primary_route_summary.to_csv(
    TABLE_DIR /
    "Step30G_Primary_MechanisticRoutes_Summary.csv",
    index=False
)

print(
    "\nSTEP30G PRIMARY ROUTE SUMMARY: SAVED"
)

## Step30G.1 — Causal-network sensitivity analysis

### Objective

Assess whether the primary receptor-to-regulator routes depend strongly on:

1. path-corridor width;
2. random seed;
3. edge sparsity penalty.

### Primary routes

- MS3: IFNAR1 → PIK3CA_PIK3R1 → AKT1 → BRCA1
- MS5: ITGB1 → SRC → FOS
- MS6: RXRA → NR1H4

### Interpretation

Sensitivity analysis evaluates robustness of the prior-constrained solution.
It does not establish biochemical causality.

A failed or timed-out optimization is recorded as a computational failure
and is not interpreted as biological absence.

In [ ]:
# =============================================================================
# Step30G.1 Cell 145
# Generic sensitivity-run helper
# =============================================================================

from cvxpy.error import SolverError


def run_sensitivity_solution(
    state,
    max_path_length,
    seed=42,
    edge_penalty=0.01,
    time_limit=900,
    mip_rel_gap=0.01,
    verbose=False
):

    inputs = STATE_INPUTS_PRIOR[state]
    outputs = STATE_OUTPUTS_PRIOR[state]

    # ---------------------------------------------------------
    # 1. Build path corridor
    # ---------------------------------------------------------

    corridor_nodes, corridor_edges = (
        build_path_corridor(
            G_full,
            list(inputs.keys()),
            list(outputs.keys()),
            max_length=max_path_length
        )
    )

    corridor_ppi = pd.DataFrame(
        corridor_edges,
        columns=[
            "source",
            "target",
            "mor"
        ]
    )

    # ---------------------------------------------------------
    # 2. Build reduced LIANA prior
    # ---------------------------------------------------------

    try:

        prior = BUILD_PRIOR(
            ppis=corridor_ppi,
            input_nodes=inputs,
            output_nodes=outputs,
            verbose=False
        )

    except Exception as e:

        return {
            "status": "prior_failed",
            "error": repr(e),
            "state": state,
            "max_path_length": max_path_length,
            "seed": seed,
            "edge_penalty": edge_penalty,
            "corridor_nodes": len(corridor_nodes),
            "corridor_edges": len(corridor_ppi)
        }

    prior_nodes = get_prior_nodes(
        prior
    )

    inputs_run = {
        k: v
        for k, v in inputs.items()
        if k in prior_nodes
    }

    outputs_run = {
        k: v
        for k, v in outputs.items()
        if k in prior_nodes
    }

    # ---------------------------------------------------------
    # 3. Solve
    # ---------------------------------------------------------

    try:

        network, problem = (
            li.mt.find_causalnet(

                prior_graph=prior,

                input_node_scores=
                    inputs_run,

                output_node_scores=
                    outputs_run,

                node_weights=None,

                missing_penalty=10.0,
                edge_penalty=edge_penalty,

                solver="scipy",

                seed=seed,

                max_runs=1,
                stable_runs=5,

                verbose=verbose,

                scipy_options={
                    "disp": False,
                    "presolve": True,
                    "time_limit": time_limit,
                    "mip_rel_gap": mip_rel_gap
                }
            )
        )

    except SolverError as e:

        return {
            "status": "solver_failed",
            "error": repr(e),
            "state": state,
            "max_path_length": max_path_length,
            "seed": seed,
            "edge_penalty": edge_penalty,
            "corridor_nodes": len(corridor_nodes),
            "corridor_edges": len(corridor_ppi),
            "reduced_prior_nodes": len(prior_nodes),
            "inputs_run": len(inputs_run),
            "outputs_run": len(outputs_run)
        }

    except Exception as e:

        return {
            "status": "other_failed",
            "error": repr(e),
            "state": state,
            "max_path_length": max_path_length,
            "seed": seed,
            "edge_penalty": edge_penalty,
            "corridor_nodes": len(corridor_nodes),
            "corridor_edges": len(corridor_ppi),
            "reduced_prior_nodes": len(prior_nodes)
        }

    # ---------------------------------------------------------
    # 4. Role-aware mechanistic audit
    # ---------------------------------------------------------

    (
        clean_network,
        true_inputs,
        true_outputs,
        io_table,
        mechanistic_network
    ) = audit_causal_solution(

        network,

        state_inputs=inputs_run,
        state_outputs=outputs_run
    )

    return {
        "status": "success",

        "state": state,
        "max_path_length": max_path_length,
        "seed": seed,
        "edge_penalty": edge_penalty,

        "corridor_nodes":
            len(corridor_nodes),

        "corridor_edges":
            len(corridor_ppi),

        "reduced_prior_nodes":
            len(prior_nodes),

        "inputs_run":
            len(inputs_run),

        "outputs_run":
            len(outputs_run),

        "raw_network":
            network,

        "clean_network":
            clean_network,

        "mechanistic_network":
            mechanistic_network,

        "true_inputs":
            true_inputs,

        "true_outputs":
            true_outputs,

        "io_connectivity":
            io_table
    }

In [ ]:
# =============================================================================
# Step30G.1 Cell 146
# Corridor sensitivity configurations
# =============================================================================

CORRIDOR_SENSITIVITY = {
    "MS3": [
        4,
        5,
        6
    ],

    "MS5": [
        4,
        5
    ],

    "MS6": [
        4,
        5,
        6
    ]
}


for state, lengths in (
    CORRIDOR_SENSITIVITY.items()
):

    print(
        state,
        "->",
        lengths
    )

In [ ]:
# =============================================================================
# Step30G.1 Cell 147
# Run corridor sensitivity
# =============================================================================

corridor_results = {}


for state, lengths in (
    CORRIDOR_SENSITIVITY.items()
):

    for L in lengths:

        key = (
            state,
            L
        )

        print(
            "\n" + "=" * 80
        )

        print(
            "Running:",
            state,
            "| L =",
            L
        )

        result = (
            run_sensitivity_solution(
                state=state,
                max_path_length=L,
                seed=42,
                edge_penalty=0.01,
                time_limit=900,
                mip_rel_gap=0.01,
                verbose=False
            )
        )

        corridor_results[
            key
        ] = result

        print(
            "Status:",
            result["status"]
        )

        if (
            result["status"]
            ==
            "success"
        ):

            print(
                "Mechanistic edges:",
                len(
                    result[
                        "mechanistic_network"
                    ]
                )
            )

            print(
                "Inputs:",
                result[
                    "true_inputs"
                ]
            )

            print(
                "Outputs:",
                result[
                    "true_outputs"
                ]
            )

In [ ]:
# =============================================================================
# Step30G.1 Cell 148
# Frozen primary route definitions
# =============================================================================

PRIMARY_CORE_EDGES = {

    "MS3": {
        (
            "IFNAR1",
            "PIK3CA_PIK3R1"
        ),

        (
            "PIK3CA_PIK3R1",
            "AKT1"
        ),

        (
            "AKT1",
            "BRCA1"
        )
    },


    "MS5": {
        (
            "ITGB1",
            "SRC"
        ),

        (
            "SRC",
            "FOS"
        )
    },


    "MS6": {
        (
            "RXRA",
            "NR1H4"
        )
    }
}


PRIMARY_CORE_NODES = {
    state:
        set(
            x
            for edge in edges
            for x in edge
        )

    for state, edges
    in PRIMARY_CORE_EDGES.items()
}


for state in AXIS_STATES:

    print(
        state
    )

    print(
        "Nodes:",
        PRIMARY_CORE_NODES[state]
    )

    print(
        "Edges:",
        PRIMARY_CORE_EDGES[state]
    )

In [ ]:
# =============================================================================
# Step30G.1 Cell 149
# Quantify corridor robustness
# =============================================================================

corridor_summary_rows = []


for (
    state,
    L
), result in corridor_results.items():

    row = {
        "state": state,
        "max_path_length": L,
        "status": result["status"]
    }

    if (
        result["status"]
        !=
        "success"
    ):

        row.update({
            "n_mechanistic_nodes":
                np.nan,

            "n_mechanistic_edges":
                np.nan,

            "core_node_fraction":
                np.nan,

            "core_edge_fraction":
                np.nan,

            "full_primary_route":
                False,

            "selected_inputs":
                "",

            "selected_outputs":
                ""
        })

        corridor_summary_rows.append(
            row
        )

        continue


    mech = (
        result[
            "mechanistic_network"
        ]
    )


    mech_edges = set(
        zip(
            mech[
                "source"
            ].astype(str),

            mech[
                "target"
            ].astype(str)
        )
    )


    mech_nodes = set(
        mech[
            "source"
        ].astype(str)
    ).union(
        set(
            mech[
                "target"
            ].astype(str)
        )
    )


    core_edges = (
        PRIMARY_CORE_EDGES[
            state
        ]
    )

    core_nodes = (
        PRIMARY_CORE_NODES[
            state
        ]
    )


    edge_fraction = (
        len(
            core_edges
            &
            mech_edges
        )
        /
        len(
            core_edges
        )
    )


    node_fraction = (
        len(
            core_nodes
            &
            mech_nodes
        )
        /
        len(
            core_nodes
        )
    )


    row.update({

        "n_mechanistic_nodes":
            len(
                mech_nodes
            ),

        "n_mechanistic_edges":
            len(
                mech_edges
            ),

        "core_node_fraction":
            node_fraction,

        "core_edge_fraction":
            edge_fraction,

        "full_primary_route":
            core_edges.issubset(
                mech_edges
            ),

        "selected_inputs":
            " | ".join(
                result[
                    "true_inputs"
                ]
            ),

        "selected_outputs":
            " | ".join(
                result[
                    "true_outputs"
                ]
            )
    })


    corridor_summary_rows.append(
        row
    )


corridor_sensitivity_summary = (
    pd.DataFrame(
        corridor_summary_rows
    )
)


display(
    corridor_sensitivity_summary
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 150
# Save corridor sensitivity
# =============================================================================

corridor_sensitivity_summary.to_csv(
    TABLE_DIR /
    "Step30G1_CorridorSensitivity_Summary.csv",
    index=False
)


for (
    state,
    L
), result in corridor_results.items():

    if (
        result["status"]
        ==
        "success"
    ):

        result[
            "mechanistic_network"
        ].to_csv(
            TABLE_DIR /
            f"Step30G1_{state}_L{L}_MechanisticNetwork.csv",
            index=False
        )


print(
    "CORRIDOR SENSITIVITY SAVED"
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 151
# One-factor-at-a-time solver sensitivity
# =============================================================================

SOLVER_SENSITIVITY = [

    {
        "label":
            "Primary",

        "seed":
            42,

        "edge_penalty":
            0.01
    },

    {
        "label":
            "Seed1337",

        "seed":
            1337,

        "edge_penalty":
            0.01
    },

    {
        "label":
            "Seed2026",

        "seed":
            2026,

        "edge_penalty":
            0.01
    },

    {
        "label":
            "Penalty005",

        "seed":
            42,

        "edge_penalty":
            0.005
    },

    {
        "label":
            "Penalty020",

        "seed":
            42,

        "edge_penalty":
            0.02
    }
]


pd.DataFrame(
    SOLVER_SENSITIVITY
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 152
# Run solver sensitivity on primary corridors
# =============================================================================

solver_results = {}


for state in AXIS_STATES:

    primary_L = (
        PRIMARY_PATH_CUTOFF[
            state
        ]
    )

    for config in (
        SOLVER_SENSITIVITY
    ):

        label = config[
            "label"
        ]

        print(
            "\n" + "=" * 80
        )

        print(
            state,
            "|",
            label
        )

        result = (
            run_sensitivity_solution(

                state=state,

                max_path_length=
                    primary_L,

                seed=
                    config["seed"],

                edge_penalty=
                    config[
                        "edge_penalty"
                    ],

                time_limit=900,

                mip_rel_gap=0.01,

                verbose=False
            )
        )

        solver_results[
            (
                state,
                label
            )
        ] = result


        print(
            "Status:",
            result["status"]
        )

        if (
            result["status"]
            ==
            "success"
        ):

            print(
                "Inputs:",
                result[
                    "true_inputs"
                ]
            )

            print(
                "Outputs:",
                result[
                    "true_outputs"
                ]
            )

            print(
                "Mechanistic edges:",
                len(
                    result[
                        "mechanistic_network"
                    ]
                )
            )

In [ ]:
# =============================================================================
# Step30G.1 Cell 153
# Quantify solver robustness
# =============================================================================

solver_summary_rows = []


for (
    state,
    label
), result in solver_results.items():

    row = {
        "state":
            state,

        "setting":
            label,

        "status":
            result["status"]
    }


    if (
        result["status"]
        !=
        "success"
    ):

        row.update({
            "core_node_fraction":
                np.nan,

            "core_edge_fraction":
                np.nan,

            "full_primary_route":
                False,

            "selected_inputs":
                "",

            "selected_outputs":
                ""
        })

        solver_summary_rows.append(
            row
        )

        continue


    mech = (
        result[
            "mechanistic_network"
        ]
    )


    mech_edges = set(
        zip(
            mech[
                "source"
            ].astype(str),

            mech[
                "target"
            ].astype(str)
        )
    )


    mech_nodes = set(
        mech[
            "source"
        ].astype(str)
    ).union(
        set(
            mech[
                "target"
            ].astype(str)
        )
    )


    core_edges = (
        PRIMARY_CORE_EDGES[
            state
        ]
    )

    core_nodes = (
        PRIMARY_CORE_NODES[
            state
        ]
    )


    row.update({

        "core_node_fraction":
            len(
                core_nodes
                &
                mech_nodes
            )
            /
            len(
                core_nodes
            ),

        "core_edge_fraction":
            len(
                core_edges
                &
                mech_edges
            )
            /
            len(
                core_edges
            ),

        "full_primary_route":
            core_edges.issubset(
                mech_edges
            ),

        "selected_inputs":
            " | ".join(
                result[
                    "true_inputs"
                ]
            ),

        "selected_outputs":
            " | ".join(
                result[
                    "true_outputs"
                ]
            )
    })


    solver_summary_rows.append(
        row
    )


solver_sensitivity_summary = (
    pd.DataFrame(
        solver_summary_rows
    )
)

display(
    solver_sensitivity_summary
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 154
# Save solver sensitivity results
# =============================================================================

solver_sensitivity_summary.to_csv(
    TABLE_DIR /
    "Step30G1_SolverSensitivity_Summary.csv",
    index=False
)


for (
    state,
    label
), result in solver_results.items():

    if (
        result["status"]
        ==
        "success"
    ):

        result[
            "mechanistic_network"
        ].to_csv(
            TABLE_DIR /
            f"Step30G1_{state}_{label}_MechanisticNetwork.csv",
            index=False
        )


print(
    "SOLVER SENSITIVITY SAVED"
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 155
# Overall route robustness summary
# =============================================================================

robustness_rows = []


for state in AXIS_STATES:

    corridor_sub = (
        corridor_sensitivity_summary.loc[
            (
                corridor_sensitivity_summary[
                    "state"
                ] == state
            )
            &
            (
                corridor_sensitivity_summary[
                    "status"
                ] == "success"
            )
        ]
    )

    solver_sub = (
        solver_sensitivity_summary.loc[
            (
                solver_sensitivity_summary[
                    "state"
                ] == state
            )
            &
            (
                solver_sensitivity_summary[
                    "status"
                ] == "success"
            )
        ]
    )


    robustness_rows.append({

        "state":
            state,

        "corridor_success_runs":
            len(
                corridor_sub
            ),

        "corridor_full_route_runs":
            int(
                corridor_sub[
                    "full_primary_route"
                ].sum()
            ),

        "corridor_full_route_fraction":
            (
                float(
                    corridor_sub[
                        "full_primary_route"
                    ].mean()
                )
                if len(
                    corridor_sub
                )
                else np.nan
            ),

        "solver_success_runs":
            len(
                solver_sub
            ),

        "solver_full_route_runs":
            int(
                solver_sub[
                    "full_primary_route"
                ].sum()
            ),

        "solver_full_route_fraction":
            (
                float(
                    solver_sub[
                        "full_primary_route"
                    ].mean()
                )
                if len(
                    solver_sub
                )
                else np.nan
            )
    })


route_robustness_summary = (
    pd.DataFrame(
        robustness_rows
    )
)

display(
    route_robustness_summary
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 156
# REPORT — causal-network sensitivity
# =============================================================================

print("=" * 100)

print(
    "STEP30G.1 CAUSAL-NETWORK SENSITIVITY REPORT"
)

print("=" * 100)


print(
    "\nPrimary routes:"
)

display(
    primary_route_summary
)


print(
    "\nCorridor sensitivity:"
)

display(
    corridor_sensitivity_summary
)


print(
    "\nSolver sensitivity:"
)

display(
    solver_sensitivity_summary
)


print(
    "\nOverall robustness:"
)

display(
    route_robustness_summary
)


print(
    "\nInterpretation boundary:"
)

print(
    "Route recurrence across corridor and solver settings "
    "supports robustness of the prior-constrained mechanistic hypothesis."
)

print(
    "Failure to reproduce an exact route may indicate alternative "
    "near-optimal signaling solutions and does not prove biological absence."
)


print(
    "\nSTEP30G.1 SENSITIVITY COMPLETE"
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 157
# Finalize corridor sensitivity summary
# =============================================================================

PRIMARY_CORE_EDGES = {
    "MS3": {
        ("IFNAR1", "PIK3CA_PIK3R1"),
        ("PIK3CA_PIK3R1", "AKT1"),
        ("AKT1", "BRCA1")
    },

    "MS5": {
        ("ITGB1", "SRC"),
        ("SRC", "FOS")
    },

    "MS6": {
        ("RXRA", "NR1H4")
    }
}

PRIMARY_PATH_CUTOFF = {
    "MS3": 5,
    "MS5": 4,
    "MS6": 5
}


rows = []

for (state, L), result in corridor_results.items():

    row = {
        "state": state,
        "max_path_length": L,
        "status": result["status"]
    }

    if result["status"] == "success":

        mech = result["mechanistic_network"]

        mech_edges = set(
            zip(
                mech["source"].astype(str),
                mech["target"].astype(str)
            )
        )

        core = PRIMARY_CORE_EDGES[state]

        row.update({
            "n_mechanistic_edges":
                len(mech_edges),

            "core_edges_recovered":
                len(
                    mech_edges & core
                ),

            "core_edges_total":
                len(core),

            "core_edge_fraction":
                len(
                    mech_edges & core
                ) / len(core),

            "full_primary_route":
                core.issubset(
                    mech_edges
                ),

            "selected_inputs":
                " | ".join(
                    result["true_inputs"]
                ),

            "selected_outputs":
                " | ".join(
                    result["true_outputs"]
                )
        })

    else:

        row.update({
            "n_mechanistic_edges":
                np.nan,

            "core_edges_recovered":
                np.nan,

            "core_edges_total":
                len(
                    PRIMARY_CORE_EDGES[
                        state
                    ]
                ),

            "core_edge_fraction":
                np.nan,

            "full_primary_route":
                False,

            "selected_inputs":
                "",

            "selected_outputs":
                ""
        })

    rows.append(row)


corridor_sensitivity_summary = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "state",
            "max_path_length"
        ]
    )
    .reset_index(drop=True)
)

display(
    corridor_sensitivity_summary
)

corridor_sensitivity_summary.to_csv(
    TABLE_DIR /
    "Step30G1_CorridorSensitivity_Summary.csv",
    index=False
)

print(
    "\nCORRIDOR SENSITIVITY SUMMARY: SAVED"
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 158
# Reload frozen primary mechanistic networks
# =============================================================================

PRIMARY_NETWORK_FILES = {
    "MS3":
        TABLE_DIR /
        "Step30G_MS3_CausalNetwork_Primary_Clean.csv",

    "MS5":
        TABLE_DIR /
        "Step30G_MS5_CausalNetwork_Primary_Mechanistic.csv",

    "MS6":
        TABLE_DIR /
        "Step30G_MS6_CausalNetwork_Primary_Mechanistic.csv"
}


primary_networks = {}

for state, path in (
    PRIMARY_NETWORK_FILES.items()
):

    assert path.exists(), path

    primary_networks[state] = (
        pd.read_csv(path)
    )

    print(
        state,
        "->",
        primary_networks[state].shape
    )


print(
    "\nPRIMARY NETWORK RELOAD: PASS"
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 159
# Compact solver-sensitivity design
# =============================================================================

SOLVER_ALT_SETTINGS = [
    {
        "label": "Seed1337",
        "seed": 1337,
        "edge_penalty": 0.01
    },

    {
        "label": "Seed2026",
        "seed": 2026,
        "edge_penalty": 0.01
    },

    {
        "label": "Penalty005",
        "seed": 42,
        "edge_penalty": 0.005
    },

    {
        "label": "Penalty020",
        "seed": 42,
        "edge_penalty": 0.020
    }
]

display(
    pd.DataFrame(
        SOLVER_ALT_SETTINGS
    )
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 160
# Run compact solver sensitivity
# =============================================================================

solver_results = {}


for state in [
    "MS3",
    "MS5",
    "MS6"
]:

    for cfg in SOLVER_ALT_SETTINGS:

        label = cfg["label"]

        print(
            "\n" + "=" * 90
        )

        print(
            state,
            "|",
            label
        )

        result = (
            run_sensitivity_solution(

                state=state,

                max_path_length=
                    PRIMARY_PATH_CUTOFF[
                        state
                    ],

                seed=
                    cfg["seed"],

                edge_penalty=
                    cfg["edge_penalty"],

                time_limit=300,

                mip_rel_gap=0.01,

                verbose=False
            )
        )

        solver_results[
            (
                state,
                label
            )
        ] = result

        print(
            "Status:",
            result["status"]
        )

        if result["status"] == "success":

            print(
                "Inputs:",
                result["true_inputs"]
            )

            print(
                "Outputs:",
                result["true_outputs"]
            )

            print(
                "Mechanistic edges:",
                len(
                    result[
                        "mechanistic_network"
                    ]
                )
            )

In [ ]:
# =============================================================================
# Step30G.1 Cell 161
# Add primary solutions to solver-sensitivity set
# =============================================================================

for state in [
    "MS3",
    "MS5",
    "MS6"
]:

    solver_results[
        (
            state,
            "Primary"
        )
    ] = {
        "status":
            "success",

        "mechanistic_network":
            primary_networks[state],

        "true_inputs":
            sorted(
                set(
                    primary_networks[state]
                    .loc[
                        primary_networks[state][
                            "source_type"
                        ] == "input",
                        "source"
                    ]
                    .astype(str)
                )
            ),

        "true_outputs":
            sorted(
                set(
                    primary_networks[state]
                    .loc[
                        primary_networks[state][
                            "target_type"
                        ] == "output",
                        "target"
                    ]
                    .astype(str)
                )
            )
    }


print(
    "PRIMARY SOLUTIONS ADDED"
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 162
# Solver-sensitivity summary
# =============================================================================

rows = []

for (
    state,
    setting
), result in solver_results.items():

    row = {
        "state":
            state,

        "setting":
            setting,

        "status":
            result["status"]
    }

    if result["status"] == "success":

        mech = result[
            "mechanistic_network"
        ]

        mech_edges = set(
            zip(
                mech["source"].astype(str),
                mech["target"].astype(str)
            )
        )

        core = (
            PRIMARY_CORE_EDGES[
                state
            ]
        )

        row.update({
            "n_mechanistic_edges":
                len(mech_edges),

            "core_edges_recovered":
                len(
                    mech_edges & core
                ),

            "core_edges_total":
                len(core),

            "core_edge_fraction":
                len(
                    mech_edges & core
                ) / len(core),

            "full_primary_route":
                core.issubset(
                    mech_edges
                ),

            "selected_inputs":
                " | ".join(
                    result["true_inputs"]
                ),

            "selected_outputs":
                " | ".join(
                    result["true_outputs"]
                )
        })

    else:

        row.update({
            "n_mechanistic_edges":
                np.nan,

            "core_edges_recovered":
                np.nan,

            "core_edges_total":
                len(
                    PRIMARY_CORE_EDGES[
                        state
                    ]
                ),

            "core_edge_fraction":
                np.nan,

            "full_primary_route":
                False,

            "selected_inputs":
                "",

            "selected_outputs":
                ""
        })

    rows.append(row)


solver_sensitivity_summary = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "state",
            "setting"
        ]
    )
    .reset_index(drop=True)
)

display(
    solver_sensitivity_summary
)

solver_sensitivity_summary.to_csv(
    TABLE_DIR /
    "Step30G1_SolverSensitivity_Summary.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step30G.1 Cell 163
# Final route-robustness summary
# =============================================================================

robustness_rows = []


for state in [
    "MS3",
    "MS5",
    "MS6"
]:

    c = (
        corridor_sensitivity_summary.loc[
            (
                corridor_sensitivity_summary[
                    "state"
                ] == state
            )
            &
            (
                corridor_sensitivity_summary[
                    "status"
                ] == "success"
            )
        ]
    )

    s = (
        solver_sensitivity_summary.loc[
            (
                solver_sensitivity_summary[
                    "state"
                ] == state
            )
            &
            (
                solver_sensitivity_summary[
                    "status"
                ] == "success"
            )
        ]
    )


    robustness_rows.append({

        "state":
            state,

        "corridor_success_runs":
            len(c),

        "corridor_full_route_runs":
            int(
                c[
                    "full_primary_route"
                ].sum()
            ),

        "corridor_full_route_fraction":
            (
                float(
                    c[
                        "full_primary_route"
                    ].mean()
                )
                if len(c)
                else np.nan
            ),

        "solver_success_runs":
            len(s),

        "solver_full_route_runs":
            int(
                s[
                    "full_primary_route"
                ].sum()
            ),

        "solver_full_route_fraction":
            (
                float(
                    s[
                        "full_primary_route"
                    ].mean()
                )
                if len(s)
                else np.nan
            )
    })


route_robustness_summary = (
    pd.DataFrame(
        robustness_rows
    )
)

display(
    route_robustness_summary
)

route_robustness_summary.to_csv(
    TABLE_DIR /
    "Step30G1_RouteRobustness_Final.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step30H Cell 164
# Reload frozen full-TME master
# =============================================================================

import scanpy as sc
from statsmodels.stats.multitest import multipletests


DATA_DIR = (
    STEP30_DIR /
    "Data"
)

AUDIT_DIR = (
    STEP30_DIR /
    "Audit"
)

FIGURE_DIR = (
    STEP30_DIR /
    "Figures"
)

for d in [
    AUDIT_DIR,
    FIGURE_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )


MASTER_PATH = (
    DATA_DIR /
    "GSE149614_STEP30_FULL_TME_MASTER.h5ad"
)

assert MASTER_PATH.exists()

adata_full = sc.read_h5ad(
    MASTER_PATH
)

print(
    adata_full.shape
)

print(
    "\nHCC10L communication groups:"
)

display(
    adata_full.obs.loc[
        adata_full.obs[
            "sample"
        ] == "HCC10L",
        "communication_group"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30H Cell 164
# Reload frozen full-TME master
# =============================================================================

import scanpy as sc
from statsmodels.stats.multitest import multipletests


DATA_DIR = (
    STEP30_DIR /
    "Data"
)

AUDIT_DIR = (
    STEP30_DIR /
    "Audit"
)

FIGURE_DIR = (
    STEP30_DIR /
    "Figures"
)

for d in [
    AUDIT_DIR,
    FIGURE_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )


MASTER_PATH = (
    DATA_DIR /
    "GSE149614_STEP30_FULL_TME_MASTER.h5ad"
)

assert MASTER_PATH.exists()

adata_full = sc.read_h5ad(
    MASTER_PATH
)

print(
    adata_full.shape
)

print(
    "\nHCC10L communication groups:"
)

display(
    adata_full.obs.loc[
        adata_full.obs[
            "sample"
        ] == "HCC10L",
        "communication_group"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30H Cell 165
# HCC10L metastatic-site sensitivity object
# =============================================================================

SENDERS = [
    "T/NK",
    "Myeloid",
    "B",
    "Fibroblast"
]

TARGETS = [
    "Malignant_MS6",
    "Malignant_MS5",
    "Malignant_MS3"
]


groupby_pairs = pd.DataFrame(
    [
        (
            sender,
            target
        )
        for sender in SENDERS
        for target in TARGETS
    ],
    columns=[
        "source",
        "target"
    ]
)


hcc10l_adata = (
    adata_full[
        (
            adata_full.obs[
                "sample"
            ] == "HCC10L"
        )
        &
        (
            adata_full.obs[
                "communication_group"
            ].isin(
                SENDERS +
                TARGETS
            )
        )
    ]
    .copy()
)


print(
    "HCC10L analysis cells:",
    hcc10l_adata.n_obs
)

display(
    hcc10l_adata.obs[
        "communication_group"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step30H Cell 166
# HCC10L LIANA Rank_Aggregate sensitivity
# =============================================================================

hcc10l_res = (
    li.mt.rank_aggregate.by_sample(

        hcc10l_adata,

        sample_key=
            "sample",

        groupby=
            "communication_group",

        resource_name=
            "consensus",

        expr_prop=
            0.10,

        min_cells=
            20,

        groupby_pairs=
            groupby_pairs,

        use_raw=
            False,

        n_perms=
            1000,

        seed=
            42,

        n_jobs=
            4,

        return_all_lrs=
            False,

        inplace=
            False,

        verbose=
            True
    )
)


print(
    "HCC10L LIANA rows:",
    len(hcc10l_res)
)

display(
    hcc10l_res.head()
)

hcc10l_res.to_csv(
    TABLE_DIR /
    "Step30H_LIANA_HCC10L.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step30H Cell 167
# CellPhoneDB-like permutation FDR within source-target blocks
# =============================================================================

hcc10l_res[
    "cellphone_fdr"
] = np.nan


for (
    source,
    target
), idx in (
    hcc10l_res
    .groupby(
        [
            "source",
            "target"
        ]
    )
    .groups.items()
):

    p = pd.to_numeric(
        hcc10l_res.loc[
            idx,
            "cellphone_pvals"
        ],
        errors="coerce"
    )

    valid = p.notna()

    if valid.sum() == 0:
        continue

    q = multipletests(
        p.loc[
            valid
        ],
        method="fdr_bh"
    )[1]

    hcc10l_res.loc[
        p.loc[
            valid
        ].index,
        "cellphone_fdr"
    ] = q


print(
    "FDR < .05:",
    int(
        (
            hcc10l_res[
                "cellphone_fdr"
            ] < 0.05
        ).sum()
    )
)

In [ ]:
# =============================================================================
# Step30H Cell 168
# Reload frozen Step30E candidates
# =============================================================================

candidate39 = pd.read_csv(
    TABLE_DIR /
    "Step30E1_MultiResource_Provenance.csv"
)

assert len(candidate39) == 39

print(
    "Frozen candidates:",
    len(candidate39)
)

display(
    candidate39[
        [
            "source",
            "ligand_complex",
            "receptor_complex",
            "target",
            "evidence_tier"
        ]
    ].head()
)

In [ ]:
# =============================================================================
# Step30H Cell 169
# Determine HCC10L preferred malignant target for each LR
# =============================================================================

hcc10l_axis = (
    hcc10l_res.loc[
        hcc10l_res[
            "target"
        ].isin(
            TARGETS
        )
    ]
    .copy()
)


pref_rows = []

for keys, sub in (
    hcc10l_axis
    .groupby(
        [
            "source",
            "ligand_complex",
            "receptor_complex"
        ]
    )
):

    sub2 = (
        sub.dropna(
            subset=[
                "lr_means"
            ]
        )
    )

    if len(sub2) == 0:
        continue

    best_idx = (
        sub2[
            "lr_means"
        ].astype(float)
        .idxmax()
    )

    best = sub2.loc[
        best_idx
    ]

    pref_rows.append({
        "source":
            keys[0],

        "ligand_complex":
            keys[1],

        "receptor_complex":
            keys[2],

        "preferred_target_HCC10L":
            best["target"],

        "preferred_lr_means_HCC10L":
            float(
                best["lr_means"]
            )
    })


hcc10l_pref = pd.DataFrame(
    pref_rows
)

print(
    "LRs with preferred-state estimate:",
    len(hcc10l_pref)
)

In [ ]:
# =============================================================================
# Step30H Cell 170
# HCC10L support for frozen Step30E candidates
# =============================================================================

hcc10l_match = (
    candidate39.merge(

        hcc10l_res[
            [
                "source",
                "target",
                "ligand_complex",
                "receptor_complex",
                "lr_means",
                "magnitude_rank",
                "specificity_rank",
                "cellphone_pvals",
                "cellphone_fdr"
            ]
        ],

        on=[
            "source",
            "target",
            "ligand_complex",
            "receptor_complex"
        ],

        how="left"
    )
    .merge(

        hcc10l_pref,

        on=[
            "source",
            "ligand_complex",
            "receptor_complex"
        ],

        how="left"
    )
)


hcc10l_match[
    "present_HCC10L"
] = (
    hcc10l_match[
        "lr_means"
    ].notna()
)


hcc10l_match[
    "FDR05_HCC10L"
] = (
    hcc10l_match[
        "cellphone_fdr"
    ] < 0.05
)


hcc10l_match[
    "preferred_target_match_HCC10L"
] = (
    hcc10l_match[
        "target"
    ]
    ==
    hcc10l_match[
        "preferred_target_HCC10L"
    ]
)


hcc10l_match[
    "support_HCC10L"
] = (
    hcc10l_match[
        "present_HCC10L"
    ]
    &
    hcc10l_match[
        "FDR05_HCC10L"
    ]
    &
    hcc10l_match[
        "preferred_target_match_HCC10L"
    ]
)


display(
    hcc10l_match[
        [
            "source",
            "ligand_complex",
            "receptor_complex",
            "target",
            "evidence_tier",
            "present_HCC10L",
            "FDR05_HCC10L",
            "preferred_target_HCC10L",
            "preferred_target_match_HCC10L",
            "support_HCC10L"
        ]
    ]
)


hcc10l_match.to_csv(
    TABLE_DIR /
    "Step30H_HCC10L_FrozenCandidateSensitivity.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step30H Cell 171
# REPORT — metastatic-site sensitivity
# =============================================================================

print("=" * 100)

print(
    "STEP30H HCC10L METASTATIC-SITE SENSITIVITY REPORT"
)

print("=" * 100)


print(
    "\nFrozen candidates:",
    len(
        hcc10l_match
    )
)


print(
    "Present in HCC10L:",
    int(
        hcc10l_match[
            "present_HCC10L"
        ].sum()
    )
)


print(
    "FDR < .05:",
    int(
        hcc10l_match[
            "FDR05_HCC10L"
        ].sum()
    )
)


print(
    "FDR < .05 AND same preferred target:",
    int(
        hcc10l_match[
            "support_HCC10L"
        ].sum()
    )
)


print(
    "\nSupport by frozen target:"
)

display(
    pd.crosstab(
        hcc10l_match[
            "target"
        ],
        hcc10l_match[
            "support_HCC10L"
        ]
    )
)


print(
    "\nSupport by evidence tier:"
)

display(
    pd.crosstab(
        hcc10l_match[
            "evidence_tier"
        ],
        hcc10l_match[
            "support_HCC10L"
        ]
    )
)


print(
    "\nHCC10L is a metastatic-site sensitivity analysis "
    "from the same HCC10 patient and is NOT an independent biological replicate."
)

print(
    "\nSTEP30H COMPLETE"
)

In [ ]:
# =============================================================================
# Step30I Cell 172
# Figure — state-skew communication candidates
# =============================================================================

import matplotlib.pyplot as plt


candidate_count = (
    candidate39
    .groupby(
        [
            "target",
            "evidence_tier"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        [
            "Malignant_MS6",
            "Malignant_MS5",
            "Malignant_MS3"
        ]
    )
)


ax = candidate_count.plot(
    kind="bar",
    figsize=(
        6.5,
        4.5
    )
)

ax.set_xlabel(
    "Malignant state"
)

ax.set_ylabel(
    "Number of state-skewed LR candidates"
)

ax.set_title(
    "Cross-patient state-skewed communication candidates"
)

ax.legend(
    title="Evidence tier",
    frameon=False
)

plt.xticks(
    rotation=0
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR /
    "Step30_Fig_CommunicationCandidateCounts.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    FIGURE_DIR /
    "Step30_Fig_CommunicationCandidateCounts.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =============================================================================
# Step30I Cell 173
# Figure — primary mechanistic routes
# =============================================================================

import networkx as nx


route_edges = [
    (
        "IFNAR1",
        "PIK3CA_PIK3R1"
    ),
    (
        "PIK3CA_PIK3R1",
        "AKT1"
    ),
    (
        "AKT1",
        "BRCA1"
    ),

    (
        "ITGB1",
        "SRC"
    ),
    (
        "SRC",
        "FOS"
    ),

    (
        "RXRA",
        "NR1H4"
    )
]


route_graph = nx.DiGraph()

route_graph.add_edges_from(
    route_edges
)


pos = {
    "IFNAR1": (0, 2),
    "PIK3CA_PIK3R1": (1, 2),
    "AKT1": (2, 2),
    "BRCA1": (3, 2),

    "ITGB1": (0, 1),
    "SRC": (1.5, 1),
    "FOS": (3, 1),

    "RXRA": (0, 0),
    "NR1H4": (3, 0)
}


plt.figure(
    figsize=(
        10,
        4.5
    )
)


nx.draw_networkx(
    route_graph,
    pos=pos,
    with_labels=True,
    arrows=True,
    node_size=2200,
    font_size=9
)


plt.text(
    -0.7,
    2,
    "MS3",
    fontsize=11,
    va="center"
)

plt.text(
    -0.7,
    1,
    "MS5",
    fontsize=11,
    va="center"
)

plt.text(
    -0.7,
    0,
    "MS6",
    fontsize=11,
    va="center"
)


plt.title(
    "Prior-constrained receiver-to-regulator routes"
)

plt.axis(
    "off"
)

plt.tight_layout()


plt.savefig(
    FIGURE_DIR /
    "Step30_Fig_PrimaryMechanisticRoutes.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    FIGURE_DIR /
    "Step30_Fig_PrimaryMechanisticRoutes.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =============================================================================
# Step30I Cell 174
# Figure — primary-route robustness
# =============================================================================

rob_plot = (
    route_robustness_summary
    .set_index(
        "state"
    )[
        [
            "corridor_full_route_fraction",
            "solver_full_route_fraction"
        ]
    ]
)


fig, ax = plt.subplots(
    figsize=(
        5.5,
        3.5
    )
)


im = ax.imshow(
    rob_plot.values,
    aspect="auto",
    vmin=0,
    vmax=1
)


ax.set_xticks(
    range(
        rob_plot.shape[1]
    )
)

ax.set_xticklabels(
    [
        "Corridor",
        "Solver parameters"
    ]
)


ax.set_yticks(
    range(
        rob_plot.shape[0]
    )
)

ax.set_yticklabels(
    rob_plot.index
)


for i in range(
    rob_plot.shape[0]
):

    for j in range(
        rob_plot.shape[1]
    ):

        value = (
            rob_plot.iloc[
                i,
                j
            ]
        )

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center"
        )


ax.set_title(
    "Primary-route reproduction across sensitivity analyses"
)

fig.colorbar(
    im,
    ax=ax,
    label="Full-route reproduction fraction"
)

plt.tight_layout()


plt.savefig(
    FIGURE_DIR /
    "Step30_Fig_RouteRobustness.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    FIGURE_DIR /
    "Step30_Fig_RouteRobustness.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =============================================================================
# Step30 FINAL Cell 175
# Final primary mechanistic-route table
# =============================================================================

primary_route_summary = pd.DataFrame([
    {
        "state":
            "MS3",

        "communication_level":
            "Primary robust; exploratory LR provenance",

        "entry_node":
            "IFNAR1",

        "intermediate_nodes":
            "PIK3CA_PIK3R1 | AKT1",

        "regulator_output":
            "BRCA1",

        "route":
            "IFNAR1 -> PIK3CA_PIK3R1 -> AKT1 -> BRCA1",

        "n_edges":
            3,

        "claim_boundary":
            "Intracellular prior compatibility; upstream MMP9-IFNAR1 remains exploratory."
    },

    {
        "state":
            "MS5",

        "communication_level":
            "Secondary moderate",

        "entry_node":
            "ITGB1",

        "intermediate_nodes":
            "SRC",

        "regulator_output":
            "FOS",

        "route":
            "ITGB1 -> SRC -> FOS",

        "n_edges":
            2,

        "claim_boundary":
            "Moderate state-skew communication support; not a primary robust LR axis."
    },

    {
        "state":
            "MS6",

        "communication_level":
            "Primary robust communication landscape",

        "entry_node":
            "RXRA",

        "intermediate_nodes":
            "",

        "regulator_output":
            "NR1H4",

        "route":
            "RXRA -> NR1H4",

        "n_edges":
            1,

        "claim_boundary":
            "Receiver-side interaction/signaling entry node; not evidence of biochemical activation."
    }
])


display(
    primary_route_summary
)


primary_route_summary.to_csv(
    TABLE_DIR /
    "Step30_FINAL_PrimaryMechanisticRoutes.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step30 FINAL Cell 176
# Environment and provenance audit
# =============================================================================

import importlib.metadata as md
import hashlib
import platform


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b""
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


env_info = {
    "python":
        platform.python_version(),

    "liana":
        md.version(
            "liana"
        ),

    "corneto":
        md.version(
            "corneto"
        ),

    "scanpy":
        md.version(
            "scanpy"
        ),

    "anndata":
        md.version(
            "anndata"
        ),

    "scipy":
        md.version(
            "scipy"
        )
}


master_sha256 = (
    sha256_file(
        MASTER_PATH
    )
)


print(
    env_info
)

print(
    "\nMaster SHA256:",
    master_sha256
)

In [ ]:
# =============================================================================
# Step30 FINAL Cell 177
# Write machine-readable Final Lock
# =============================================================================

import json


candidate_counts = (
    candidate39[
        "evidence_tier"
    ]
    .value_counts()
    .to_dict()
)


hcc10l_final_support = int(
    hcc10l_match[
        "support_HCC10L"
    ].sum()
)


STEP30_FINAL_LOCK = {

    "step":
        "Step30",

    "analysis":
        "Cell-cell communication",

    "master_dataset": {
        "file":
            str(
                MASTER_PATH
            ),

        "shape":
            [
                int(
                    adata_full.n_obs
                ),
                int(
                    adata_full.n_vars
                )
            ],

        "sha256":
            master_sha256
    },

    "primary_design": {
        "samples":
            [
                "HCC09T",
                "HCC10T"
            ],

        "senders":
            SENDERS,

        "receivers":
            TARGETS,

        "expr_prop":
            0.10,

        "min_cells":
            20,

        "n_perms":
            1000,

        "seed":
            42,

        "resource":
            "consensus",

        "method":
            "LIANA Rank_Aggregate"
    },

    "candidate_lock": {
        "n_candidates_ge3pct":
            39,

        "primary_robust_ge5pct":
            28,

        "secondary_moderate_3to5pct":
            11,

        "by_state": {
            "MS6":
                34,

            "MS5":
                4,

            "MS3":
                1
        }
    },

    "primary_routes":
        primary_route_summary.to_dict(
            orient="records"
        ),

    "route_robustness":
        route_robustness_summary.to_dict(
            orient="records"
        ),

    "hcc10l_sensitivity": {
        "same_patient_metastatic_site":
            True,

        "not_independent_replication":
            True,

        "frozen_candidates_tested":
            int(
                len(
                    hcc10l_match
                )
            ),

        "supported_fdr05_and_same_preferred_target":
            hcc10l_final_support
    },

    "environment":
        env_info,

    "claim_boundaries": [
        "Cell-cell communication inference does not establish physical signaling.",
        "Receiver-side expression consistency is not independent validation.",
        "Causal-network reconstruction is prior-constrained mechanistic compatibility, not experimental causality.",
        "MS5 communication evidence is moderate rather than robust under the predefined 5% state-skew threshold.",
        "MMP9-IFNAR1 remains exploratory because LR provenance is comparatively narrow.",
        "HCC10L is a same-patient metastatic-site sensitivity analysis, not an independent biological replicate."
    ]
}


JSON_PATH = (
    AUDIT_DIR /
    "Step30_FINAL_Communication.json"
)


with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        STEP30_FINAL_LOCK,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


print(
    "Saved:",
    JSON_PATH
)

In [ ]:
# =============================================================================
# Step30 FINAL Cell 178
# Human-readable Final Lock
# =============================================================================

rob_text = (
    route_robustness_summary
    .to_string(
        index=False
    )
)


md_text = f"""
# Step30 Final Lock — Cell-cell Communication

## Frozen master

- Full TME master: {MASTER_PATH.name}
- Shape: {adata_full.n_obs:,} cells × {adata_full.n_vars:,} genes
- Frozen malignant-state universe preserved.
- No reclustering or malignant-state redefinition performed.

## Primary communication design

Primary discovery samples:

- HCC09T
- HCC10T

Primary TME senders:

- T/NK
- Myeloid
- B
- Fibroblast

Primary malignant receivers:

- Malignant_MS6
- Malignant_MS5
- Malignant_MS3

LIANA Rank_Aggregate:
- consensus resource
- expr_prop = 0.10
- min_cells = 20
- n_perms = 1000
- seed = 42

## Frozen state-skew candidate set

Total candidates with >=3% cross-patient state skew: 39

- MS6: 34
- MS5: 4
- MS3: 1

Evidence tiers:

- Primary robust >=5%: 28
- Secondary moderate 3–5%: 11

Receiver-side expression preference agreed with the corresponding malignant
state in both primary patients for all 39 candidates. This is interpreted
as receiver-side decomposition/internal consistency, not independent validation.

## Primary prior-constrained mechanistic routes

MS3:
IFNAR1 -> PIK3CA_PIK3R1 -> AKT1 -> BRCA1

Interpretation:
Exploratory intracellular bridge. Upstream MMP9-IFNAR1 remains database-dependent
and is not interpreted as experimentally established signaling.

MS5:
ITGB1 -> SRC -> FOS

Interpretation:
Secondary/moderate communication-supported transitional-plasticity route.

MS6:
RXRA -> NR1H4

Interpretation:
Parsimonious route linking MS6-enriched receiver-side interaction evidence
to the frozen hepatic-identity regulatory architecture.

## Sensitivity

{rob_text}

## HCC10L sensitivity

Frozen candidates tested: {len(hcc10l_match)}
Candidates supported by FDR < 0.05 and concordant preferred malignant target:
{hcc10l_final_support}

HCC10L represents a metastatic-site sensitivity analysis from the same HCC10
patient and is not treated as an independent biological replicate.

## Claim boundaries

1. Communication inference does not establish physical ligand-receptor signaling.
2. Receiver-side receptor enrichment is not independent validation.
3. Prior-constrained causal networks represent mechanistic compatibility, not causality.
4. MS5 has no >=5% robust state-skew LR candidate under the frozen threshold.
5. MMP9-IFNAR1 remains exploratory.
6. RXRA is described as a receiver-side interaction/signaling entry node rather than
   uniformly as a cell-surface receptor.
7. Independent single-cell validation remains Step31 (GSE326201).

## Final Step30 interpretation

Step30 reveals a strongly asymmetric communication landscape across the frozen
MS6-MS5-MS3 malignant axis. MS6 exhibits the richest reproducible state-skewed
receiver-side interaction structure, whereas MS5 shows only moderate state-skew
and MS3 contains a single robust but database-dependent LR candidate.

Prior-constrained intracellular reconstruction identifies compact routes linking
receiver-side interaction evidence with frozen Step29 regulatory programs:
RXRA-NR1H4 for MS6, ITGB1-SRC-FOS for MS5, and
IFNAR1-PI3K-AKT-BRCA1 for MS3.

These routes are mechanistic hypotheses compatible with curated signed signaling
priors and observed state-specific receptor/regulon activity; they are not evidence
of direct causal state conversion.
"""


MD_PATH = (
    AUDIT_DIR /
    "Step30_FINAL_LOCK.md"
)


with open(
    MD_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        md_text
    )


print(
    "Saved:",
    MD_PATH
)

In [ ]:
# =============================================================================
# Step30 FINAL Cell 179
# Output manifest
# =============================================================================

important_outputs = [

    TABLE_DIR /
    "Step30E1_MultiResource_Provenance.csv",

    TABLE_DIR /
    "Step30F_Receptor_InputScores.csv",

    TABLE_DIR /
    "Step30F_TF_OutputScores.csv",

    TABLE_DIR /
    "Step30G_MS3_CausalNetwork_Primary_Clean.csv",

    TABLE_DIR /
    "Step30G_MS5_CausalNetwork_Primary_Mechanistic.csv",

    TABLE_DIR /
    "Step30G_MS6_CausalNetwork_Primary_Mechanistic.csv",

    TABLE_DIR /
    "Step30G1_CorridorSensitivity_Summary.csv",

    TABLE_DIR /
    "Step30G1_SolverSensitivity_Summary.csv",

    TABLE_DIR /
    "Step30G1_RouteRobustness_Final.csv",

    TABLE_DIR /
    "Step30H_HCC10L_FrozenCandidateSensitivity.csv",

    TABLE_DIR /
    "Step30_FINAL_PrimaryMechanisticRoutes.csv",

    JSON_PATH,

    MD_PATH
]


manifest_rows = []


for path in important_outputs:

    manifest_rows.append({
        "file":
            str(path),

        "exists":
            path.exists(),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else None
            )
    })


final_manifest = pd.DataFrame(
    manifest_rows
)

display(
    final_manifest
)


final_manifest.to_csv(
    AUDIT_DIR /
    "Step30_FINAL_OutputManifest.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step30 FINAL Cell 180
# FINAL REPORT
# =============================================================================

print("=" * 110)

print(
    "STEP30 FINAL COMMUNICATION LOCK"
)

print("=" * 110)


print(
    "\nFull TME master:",
    adata_full.shape
)


print(
    "\nFrozen communication candidates:"
)

print(
    "Total >=3%:",
    39
)

print(
    "Robust >=5%:",
    28
)

print(
    "Moderate 3-5%:",
    11
)


print(
    "\nPrimary mechanistic routes:"
)

display(
    primary_route_summary
)


print(
    "\nRoute robustness:"
)

display(
    route_robustness_summary
)


print(
    "\nHCC10L sensitivity:"
)

print(
    "Frozen candidates:",
    len(
        hcc10l_match
    )
)

print(
    "FDR<.05 + same preferred target:",
    hcc10l_final_support
)


print(
    "\nFinal files:"
)

print(
    JSON_PATH
)

print(
    MD_PATH
)


print(
    "\nCLAIM BOUNDARY:"
)

print(
    "Step30 identifies sample-aware communication patterns and "
    "prior-constrained receptor-to-regulator compatibility."
)

print(
    "It does NOT establish physical signaling, biochemical activation, "
    "direct causal TF regulation, or malignant-state conversion."
)


print(
    "\nSTEP30 FINAL LOCK: PASS"
)

print(
    "\nNEXT: STEP31 — GSE326201 independent single-cell validation"
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# STEP30 FINAL REPORT CELL
# One-cell comprehensive report for final scientific review
#
# Purpose:
#   - DO NOT rerun any analysis
#   - Reload frozen Step30 outputs
#   - Summarize communication candidates
#   - Summarize primary mechanistic routes
#   - Summarize corridor / solver sensitivity
#   - Summarize HCC10L metastatic-site sensitivity
#   - Verify final output files
#   - Print claim boundaries and next step
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import importlib.metadata as md

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)


# -----------------------------------------------------------------------------
# 0. Paths
# -----------------------------------------------------------------------------

ROOT = Path(_hcc_path(''))

STEP30_DIR = (
    ROOT /
    "LIHC_scRNA_Step30_Communication"
)

TABLE_DIR = STEP30_DIR / "Tables"
AUDIT_DIR = STEP30_DIR / "Audit"
FIGURE_DIR = STEP30_DIR / "Figures"
DATA_DIR = STEP30_DIR / "Data"


FILES = {

    "candidate39":
        TABLE_DIR /
        "Step30E1_MultiResource_Provenance.csv",

    "receptor_scores":
        TABLE_DIR /
        "Step30F_Receptor_InputScores.csv",

    "tf_scores":
        TABLE_DIR /
        "Step30F_TF_OutputScores.csv",

    "primary_routes":
        TABLE_DIR /
        "Step30_FINAL_PrimaryMechanisticRoutes.csv",

    "ms3_network":
        TABLE_DIR /
        "Step30G_MS3_CausalNetwork_Primary_Clean.csv",

    "ms5_network":
        TABLE_DIR /
        "Step30G_MS5_CausalNetwork_Primary_Mechanistic.csv",

    "ms6_network":
        TABLE_DIR /
        "Step30G_MS6_CausalNetwork_Primary_Mechanistic.csv",

    "corridor_sensitivity":
        TABLE_DIR /
        "Step30G1_CorridorSensitivity_Summary.csv",

    "solver_sensitivity":
        TABLE_DIR /
        "Step30G1_SolverSensitivity_Summary.csv",

    "route_robustness":
        TABLE_DIR /
        "Step30G1_RouteRobustness_Final.csv",

    "hcc10l":
        TABLE_DIR /
        "Step30H_HCC10L_FrozenCandidateSensitivity.csv",

    "final_json":
        AUDIT_DIR /
        "Step30_FINAL_Communication.json",

    "final_md":
        AUDIT_DIR /
        "Step30_FINAL_LOCK.md",

    "manifest":
        AUDIT_DIR /
        "Step30_FINAL_OutputManifest.csv"
}


def load_csv(key):
    path = FILES[key]

    if path.exists():
        return pd.read_csv(path)

    print(
        f"[MISSING] {key}: {path}"
    )

    return None


def section(title):
    print(
        "\n" +
        "=" * 110
    )

    print(title)

    print(
        "=" * 110
    )


# =============================================================================
# 1. WHAT STEP30 DID
# =============================================================================

section(
    "1. STEP30 — WHAT WAS DONE"
)

print(
"""
Step30 investigated how the tumor microenvironment may interact with the
frozen malignant-state axis established in Steps 23–29.

Primary analysis design:

  Primary samples:
    HCC09T + HCC10T

  TME senders:
    T/NK
    Myeloid
    B
    Fibroblast

  Malignant receivers:
    Malignant_MS6
    Malignant_MS5
    Malignant_MS3

Workflow:

  Full TME master
      ↓
  Sample-aware LIANA Rank_Aggregate
      ↓
  Cross-patient LR conservation
      ↓
  State-skew effect-size filtering
      ↓
  Receiver-side expression consistency
      ↓
  Multi-resource LR provenance audit
      ↓
  Receiver / regulator cross-patient score construction
      ↓
  Signed OmniPath intracellular prior
      ↓
  State-specific receptor-to-regulator causal-network reconstruction
      ↓
  Corridor / solver sensitivity
      ↓
  HCC10L same-patient metastatic-site sensitivity
      ↓
  Step30 Final Lock

No malignant-state reclustering or redefinition was performed.
Step29 frozen regulators were not reselected.
"""
)


# =============================================================================
# 2. ENVIRONMENT
# =============================================================================

section(
    "2. SOFTWARE ENVIRONMENT"
)

for pkg in [
    "liana",
    "corneto",
    "scanpy",
    "anndata",
    "scipy"
]:
    try:
        print(
            f"{pkg}: {md.version(pkg)}"
        )

    except Exception:
        print(
            f"{pkg}: unavailable"
        )


# =============================================================================
# 3. COMMUNICATION CANDIDATE LOCK
# =============================================================================

section(
    "3. STEP30E — COMMUNICATION CANDIDATE LOCK"
)

candidate39 = load_csv(
    "candidate39"
)

if candidate39 is not None:

    print(
        "\nTotal frozen >=3% state-skew candidates:",
        len(candidate39)
    )

    if "target" in candidate39.columns:

        print(
            "\nCandidates by malignant state:"
        )

        print(
            candidate39[
                "target"
            ]
            .value_counts()
        )

    if "evidence_tier" in candidate39.columns:

        print(
            "\nEvidence tiers:"
        )

        print(
            candidate39[
                "evidence_tier"
            ]
            .value_counts()
        )

        if "target" in candidate39.columns:

            print(
                "\nEvidence tier × malignant state:"
            )

            display(
                pd.crosstab(
                    candidate39[
                        "target"
                    ],
                    candidate39[
                        "evidence_tier"
                    ]
                )
            )

    if (
        "receptor_match_both"
        in candidate39.columns
    ):

        print(
            "\nReceiver-state expression agreement:"
        )

        print(
            candidate39[
                "receptor_match_both"
            ]
            .value_counts(
                dropna=False
            )
        )

    if (
        "n_supporting_resources"
        in candidate39.columns
    ):

        print(
            "\nNumber of supporting LR resources:"
        )

        print(
            candidate39[
                "n_supporting_resources"
            ]
            .value_counts()
            .sort_index()
        )

    key_pairs = [
        ("MMP9", "IFNAR1"),
        ("PTDSS1", "SCARB1"),
        ("PTPN6", "EGFR"),
        ("ADM", "RAMP1"),
        ("APOE", "SCARB1"),
        ("PLTP", "ABCA1")
    ]

    if {
        "ligand_complex",
        "receptor_complex"
    }.issubset(
        candidate39.columns
    ):

        key_mask = candidate39.apply(
            lambda r:
                (
                    str(
                        r[
                            "ligand_complex"
                        ]
                    ),
                    str(
                        r[
                            "receptor_complex"
                        ]
                    )
                )
                in key_pairs,
            axis=1
        )

        cols = [
            c
            for c in [
                "source",
                "ligand_complex",
                "receptor_complex",
                "target",
                "evidence_tier",
                "n_supporting_resources",
                "supporting_resources"
            ]
            if c in candidate39.columns
        ]

        print(
            "\nKey LR provenance:"
        )

        display(
            candidate39.loc[
                key_mask,
                cols
            ]
        )


# =============================================================================
# 4. STEP30F — RECEIVER AND REGULATOR SCORE LOCK
# =============================================================================

section(
    "4. STEP30F — RECEIVER / REGULATOR SCORE LOCK"
)

receptor_scores = load_csv(
    "receptor_scores"
)

tf_scores = load_csv(
    "tf_scores"
)


if tf_scores is not None:

    print(
        "\nFrozen Step29 regulator outputs:"
    )

    print(
        "N =",
        len(tf_scores)
    )

    if "target_state" in tf_scores.columns:

        print(
            tf_scores[
                "target_state"
            ]
            .value_counts()
        )

    cols = [
        c
        for c in [
            "target_state",
            "TF",
            "HCC09",
            "HCC10",
            "mean_contrast",
            "output_score"
        ]
        if c in tf_scores.columns
    ]

    display(
        tf_scores[
            cols
        ]
    )


if receptor_scores is not None:

    print(
        "\nReceiver-side signaling / interaction entry nodes:"
    )

    print(
        "N =",
        len(receptor_scores)
    )

    if "target_state" in receptor_scores.columns:

        print(
            receptor_scores[
                "target_state"
            ]
            .value_counts()
        )

    cols = [
        c
        for c in [
            "target_state",
            "receptor_gene",
            "HCC09",
            "HCC10",
            "mean_contrast",
            "input_score",
            "parent_lr_pairs",
            "evidence_tiers",
            "max_resource_support"
        ]
        if c in receptor_scores.columns
    ]

    display(
        receptor_scores[
            cols
        ]
        .sort_values(
            [
                "target_state",
                "input_score"
            ],
            ascending=[
                True,
                False
            ]
        )
    )


# =============================================================================
# 5. PRIMARY MECHANISTIC ROUTES
# =============================================================================

section(
    "5. STEP30G — PRIMARY PRIOR-CONSTRAINED MECHANISTIC ROUTES"
)

primary_routes = load_csv(
    "primary_routes"
)

if primary_routes is not None:

    display(
        primary_routes
    )


for state, key in [
    (
        "MS3",
        "ms3_network"
    ),
    (
        "MS5",
        "ms5_network"
    ),
    (
        "MS6",
        "ms6_network"
    )
]:

    net = load_csv(
        key
    )

    print(
        f"\n{state} mechanistic network:"
    )

    if net is not None:

        print(
            "Edges:",
            len(net)
        )

        nodes = set(
            net[
                "source"
            ].astype(str)
        ).union(
            set(
                net[
                    "target"
                ].astype(str)
            )
        )

        print(
            "Nodes:",
            len(nodes)
        )

        display(
            net
        )


# =============================================================================
# 6. CORRIDOR SENSITIVITY
# =============================================================================

section(
    "6. STEP30G.1 — PATH-CORRIDOR SENSITIVITY"
)

corridor_sensitivity = load_csv(
    "corridor_sensitivity"
)

if corridor_sensitivity is not None:

    display(
        corridor_sensitivity
    )

    if {
        "state",
        "status",
        "full_primary_route"
    }.issubset(
        corridor_sensitivity.columns
    ):

        print(
            "\nSuccessful runs / full-route reproduction:"
        )

        for state in [
            "MS3",
            "MS5",
            "MS6"
        ]:

            sub = corridor_sensitivity.loc[
                corridor_sensitivity[
                    "state"
                ] == state
            ]

            successful = sub.loc[
                sub[
                    "status"
                ] == "success"
            ]

            print(
                state,
                "| successful:",
                len(successful),
                "| full route:",
                int(
                    successful[
                        "full_primary_route"
                    ].sum()
                )
                if len(successful)
                else 0
            )


# =============================================================================
# 7. SOLVER SENSITIVITY
# =============================================================================

section(
    "7. STEP30G.1 — SEED / EDGE-PENALTY SENSITIVITY"
)

solver_sensitivity = load_csv(
    "solver_sensitivity"
)

if solver_sensitivity is not None:

    display(
        solver_sensitivity
    )


# =============================================================================
# 8. FINAL ROUTE ROBUSTNESS
# =============================================================================

section(
    "8. STEP30G.1 — FINAL ROUTE ROBUSTNESS"
)

route_robustness = load_csv(
    "route_robustness"
)

if route_robustness is not None:

    display(
        route_robustness
    )


# =============================================================================
# 9. HCC10L METASTATIC-SITE SENSITIVITY
# =============================================================================

section(
    "9. STEP30H — HCC10L SAME-PATIENT METASTATIC-SITE SENSITIVITY"
)

hcc10l = load_csv(
    "hcc10l"
)

if hcc10l is not None:

    print(
        "\nFrozen candidates tested:",
        len(hcc10l)
    )

    for col in [
        "present_HCC10L",
        "FDR05_HCC10L",
        "preferred_target_match_HCC10L",
        "support_HCC10L"
    ]:

        if col in hcc10l.columns:

            print(
                col,
                ":",
                int(
                    hcc10l[
                        col
                    ].fillna(
                        False
                    ).astype(bool).sum()
                ),
                "/",
                len(hcc10l)
            )

    if {
        "target",
        "support_HCC10L"
    }.issubset(
        hcc10l.columns
    ):

        print(
            "\nHCC10L support by frozen malignant target:"
        )

        display(
            pd.crosstab(
                hcc10l[
                    "target"
                ],
                hcc10l[
                    "support_HCC10L"
                ]
            )
        )

    if {
        "evidence_tier",
        "support_HCC10L"
    }.issubset(
        hcc10l.columns
    ):

        print(
            "\nHCC10L support by evidence tier:"
        )

        display(
            pd.crosstab(
                hcc10l[
                    "evidence_tier"
                ],
                hcc10l[
                    "support_HCC10L"
                ]
            )
        )

    supported = hcc10l.loc[
        hcc10l[
            "support_HCC10L"
        ].fillna(
            False
        ).astype(bool)
    ]

    print(
        "\nHCC10L-supported frozen candidates:"
    )

    cols = [
        c
        for c in [
            "source",
            "ligand_complex",
            "receptor_complex",
            "target",
            "evidence_tier",
            "lr_means",
            "cellphone_fdr",
            "preferred_target_HCC10L"
        ]
        if c in supported.columns
    ]

    display(
        supported[
            cols
        ]
    )

    # Focus on the exploratory MS3 candidate
    if {
        "ligand_complex",
        "receptor_complex"
    }.issubset(
        hcc10l.columns
    ):

        ms3_key = hcc10l.loc[
            (
                hcc10l[
                    "ligand_complex"
                ].astype(str)
                == "MMP9"
            )
            &
            (
                hcc10l[
                    "receptor_complex"
                ].astype(str)
                == "IFNAR1"
            )
        ]

        print(
            "\nFocused HCC10L MMP9-IFNAR1 audit:"
        )

        display(
            ms3_key
        )


# =============================================================================
# 10. FINAL LOCK FILES
# =============================================================================

section(
    "10. STEP30 FINAL LOCK FILE AUDIT"
)

file_rows = []

for key, path in FILES.items():

    file_rows.append({
        "key":
            key,

        "exists":
            path.exists(),

        "path":
            str(path)
    })


file_audit = pd.DataFrame(
    file_rows
)

display(
    file_audit
)


# =============================================================================
# 11. FINAL JSON SUMMARY
# =============================================================================

section(
    "11. FINAL JSON / CLAIM BOUNDARIES"
)

if FILES[
    "final_json"
].exists():

    with open(
        FILES[
            "final_json"
        ],
        "r",
        encoding="utf-8"
    ) as f:

        final_lock = json.load(
            f
        )

    print(
        "\nFinal primary routes:"
    )

    display(
        pd.DataFrame(
            final_lock.get(
                "primary_routes",
                []
            )
        )
    )

    print(
        "\nClaim boundaries:"
    )

    for i, text in enumerate(
        final_lock.get(
            "claim_boundaries",
            []
        ),
        start=1
    ):

        print(
            f"{i}. {text}"
        )

else:

    print(
        "Final JSON not found."
    )


# =============================================================================
# 12. SCIENTIFIC SUMMARY FOR CHATGPT
# =============================================================================

section(
    "12. STEP30 SCIENTIFIC SUMMARY — COPY THIS OUTPUT TO CHATGPT"
)

print(
"""
WHAT STEP30 DID
---------------
Step30 performed sample-aware TME-to-malignant communication analysis on the
frozen GSE149614 malignant-state framework.

HOW IT WAS DONE
---------------
1. Built a frozen full-TME master without reclustering.
2. Used HCC09T and HCC10T as the primary complete-axis tumors.
3. Used T/NK, Myeloid, B and Fibroblast as primary TME senders.
4. Used frozen MS6, MS5 and MS3 malignant states as receivers.
5. Ran LIANA Rank_Aggregate separately by sample.
6. Required cross-patient support.
7. Applied predefined malignant-state skew thresholds:
      >=5%  = primary robust
      3–5%  = secondary moderate
8. Audited receiver-side expression consistency.
9. Audited LR provenance across multiple resources.
10. Constructed cross-patient receptor/input and Step29 regulon/output scores.
11. Connected these nodes using a signed OmniPath intracellular prior.
12. Used state-specific path corridors and CORNETO causal-network reconstruction.
13. Performed corridor and solver sensitivity.
14. Used HCC10L only as a same-patient metastatic-site sensitivity analysis.

FROZEN PRIMARY INTERPRETATION
-----------------------------
Communication structure is strongly asymmetric across the malignant-state axis.

MS6:
  richest reproducible state-skewed communication landscape
  primary route:
      RXRA -> NR1H4

MS5:
  no >=5% robust state-skew LR candidate
  only moderate communication skew
  secondary route:
      ITGB1 -> SRC -> FOS

MS3:
  only one robust state-skew LR candidate
  upstream MMP9-IFNAR1 provenance is comparatively narrow
  exploratory intracellular route:
      IFNAR1 -> PIK3CA_PIK3R1 -> AKT1 -> BRCA1

IMPORTANT CLAIM BOUNDARY
------------------------
These routes represent prior-constrained mechanistic compatibility.
They do NOT establish:
  - physical ligand-receptor signaling,
  - biochemical receptor activation,
  - direct TF causality,
  - irreversible malignant-state conversion.

HCC10L is NOT an independent biological replicate.

NEXT STEP
---------
Step31 = GSE326201 independent single-cell validation.

Primary goals:
  1. Test portability of the frozen MS3 Core50.
  2. Test MS6 hepatic-differentiation biology.
  3. Test the reciprocal MS3 proliferative/CIN vs MS6 hepatic axis.
  4. Assess whether Step29 regulator architecture transfers.
  5. Assess Step30 communication/mechanistic routes only where the independent
     dataset provides adequate cell types and expression support.

Do NOT redefine MS1–MS8.
Do NOT reselect Core50.
Do NOT reconstruct Original RiskScore in single cells.
"""
)


print(
    "\n" +
    "=" * 110
)

print(
    "END OF STEP30 FINAL REPORT"
)

print(
    "Please copy the COMPLETE output of this cell back to ChatGPT."
)

print(
    "=" * 110
)